# Functional label propagation

Assign functional labels to AVGs that the seven `annotate` databases cannot label, by transferring labels from a labeled reference set in two tiers. Tier 1 takes the majority label of the labeled proteins in the query's MMseqs2 cluster. Tier 2 takes the label of the nearest labeled neighbor in the CheckAMG-PST embedding, gated by a calibrated correctness probability, where tier 1 gives no label. The rule is calibrated and evaluated on held-out reference proteins, then applied to the soil and human-gut AVGs. Outputs are `protein_assignments.parquet`, `evaluation_summary.json`, `manifest.json`, the `fig_*` tables, and the clustering comparison tables in `tables/propagation/`.

## 0. Setup

Imports and thread configuration.

In [1]:
import os
os.environ["POLARS_MAX_THREADS"] = "48"
import sys
import json
import time
import gc
import hashlib
import subprocess
import datetime
from pathlib import Path
import numpy as np
import polars as pl
import faiss
import tables as tb
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)
faiss.omp_set_num_threads(48)

All random steps draw from one generator seeded with 20260725, so the held-out three-way split, the correctness models, and the probability thresholds are reproducible.

Paths, constants, and the schema version downstream notebooks assert against.

In [2]:
SCHEMA_VERSION = "propagation-1.2"
SEED = 20260725
rng = np.random.default_rng(SEED)
CAT3 = ["metabolic", "physiological", "regulatory"]
VIRAL_OK = ["medium", "high", "very high"]
DATASETS = ["soil", "gut"]
LEVELS = ["specific", "L1", "category"]
TARGETS = [0.90, 0.80, 0.70]
PRIMARY_TARGET = 0.90
PCTS = [30, 50, 70, 90]
STRATA = [("<50%", 0.0, 0.5), ("50-90%", 0.5, 0.9), (">=90%", 0.9, 1.01), ("all", 0.0, 1.01)]
TARGETS_R = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99]
N_BOOT = 500
METHODS = ["cluster_only", "embedding_only", "tiered"]
TRUE = {"specific": "true_specific_n", "L1": "true_L1", "category": "true_category"}
EMB = {"specific": "raw_specific_n", "L1": "raw_L1", "category": "raw_category"}
REF = {"specific": "ref_specific_n", "L1": "ref_L1", "category": "ref_category"}

REPO = Path("/storage2/scratch/kosmopoulos/software/CheckAMG")
FILES = REPO / "CheckAMG" / "files"
MAIN = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript")
DENOVO_DB = REPO / "notebooks" / "CheckAMG_denovo_db_v1.1_20260714"
DB_STAMP = DENOVO_DB.name.replace("CheckAMG_denovo_db_", "")
CACHE = MAIN / "propagation" / DB_STAMP
CACHE.mkdir(parents=True, exist_ok=True)
OUT = REPO / "notebooks" / "tables" / "propagation"
OUT.mkdir(parents=True, exist_ok=True)
SEQCLUST = CACHE / "seqclust"
MMSEQS_BIN = Path(sys.prefix) / "bin" / "mmseqs"
CLU_EVAL = "clu_id{p}.tsv"
CLU_APPLIED = "clu_all_id{p}.tsv"
TRAPZ = getattr(np, "trapezoid", np.trapz)  # numpy 1.x spells it trapz

TRAIN_INDEX = DENOVO_DB / "checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss"
TRAIN_ANNO = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1")
TEST_OUT = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_outputs/checkamg_train_071426/checkAMG-PST_TL-P__large_5")
NOVEL = MAIN / "novel_avgs"

def den(ds):
    return MAIN / f"CheckAMG_denovo_v1.1_MetaVR{ds}"
def anno(ds):
    return MAIN / f"CheckAMG_annotate_v1.1_MetaVR{ds}" / "results"
def agg(ds):
    return MAIN / f"CheckAMG_aggregate_v1.1_MetaVR{ds}" / "aggregated_results_detailed.parquet"
print("cache:", CACHE)

cache: /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/propagation/v1.1_20260714


Every heavy stage is cached under a database-stamped directory, so a new de-novo database starts a fresh cache rather than silently reusing an old one.

In [3]:
def fresh(cache, *inputs):
    cache = Path(cache)
    if not cache.exists():
        return False
    ct = cache.stat().st_mtime
    return all(Path(i).exists() and ct >= Path(i).stat().st_mtime for i in inputs)

def sha256(p, cap=64 << 20):
    h = hashlib.sha256()
    n = 0
    with open(p, "rb") as f:
        while (b := f.read(1 << 20)):
            h.update(b)
            n += len(b)
            if n >= cap:
                break
    return h.hexdigest()

### 0.1 Cached inputs

The neighbor searches, query sets, sequence identities, and sequence clusterings that this notebook and novel_avgs.ipynb read. Each is built once and cached to save runtime, and a cached output is read instead of rebuilt.

The training protein behind each row of the CheckAMG-PST training index, recovered from the validation mask of the training graph and checked against the labels stored with the index.

In [ ]:
TRAIN_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data")
TRAIN_LABELS = DENOVO_DB / "checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5"
TRAIN_ANNOTATIONS = [TRAIN_ANNO / f"checkamg_annotate_{d}_dataset" / "results" / "final_results.parquet"
                     for d in ["genomad", "progenomes"]]
TRAIN_ORDER = NOVEL / "train_ref_index_order.parquet"

def build_train_order():
    NOVEL.mkdir(parents=True, exist_ok=True)
    with tb.open_file(str(TRAIN_DIR / "checkAMG_train_esm2_t30_150M.graphfmt.h5")) as fp:
        sizes = fp.root.sizes[:]
        validation_mask = fp.root.scaffold_val_mask[:]
    # The index holds the non-validation training proteins in graph order
    train = (pl.read_parquet(TRAIN_DIR / "train_ptns.parquet")
             .with_columns(pl.Series("validation", np.repeat(validation_mask, sizes)))
             .filter(~pl.col("validation")).with_row_index("index_row"))
    with tb.open_file(str(TRAIN_LABELS)) as fp:
        labels = fp.root.label[:]
    reconstructed = train.select((pl.col("AVG") & pl.col("viral")).cast(pl.Int64) * 3
                                 + (pl.col("AVG") & ~pl.col("viral")).cast(pl.Int64) * 2
                                 + (~pl.col("AVG") & pl.col("viral")).cast(pl.Int64)).to_series().to_numpy()
    assert np.array_equal(reconstructed, labels), "the reconstructed order does not reproduce the index labels"
    annotations = pl.concat([pl.read_parquet(p, columns=["Protein", "Protein Classification", "Function"])
                             for p in TRAIN_ANNOTATIONS]).unique("Protein")
    (train.join(annotations, on="Protein", how="left").sort("index_row").with_columns(pl.Series("label", labels))
          .select(["Protein", "Protein Classification", "Function", "label"]).write_parquet(TRAIN_ORDER))

if not TRAIN_ORDER.exists():
    build_train_order()
print(f"training index order: {pl.scan_parquet(TRAIN_ORDER).select(pl.len()).collect().item():,} proteins")

training index order: 13,293,418 proteins


The 50 nearest training-index neighbors of every held-out AVG-like test protein and, as a negative control, of every held-out non-auxiliary viral structural protein.

In [ ]:
HELDOUT_SEARCHES = ["test_avglike", "test_struct"]
STRUCT_PATTERN = (
    r"(?i)capsid|terminase|\bportal\b|baseplate|base plate|tail fib|tail spike|tail sheath|tail tube|tape measure|"
    r"tail assembly|\btail protein\b|major head|minor head|prohead|head-tail|head to tail|\bneck\b|\bcollar\b|"
    r"major coat|minor coat|virion structural|\bsheath\b|scaffold|\bspike protein\b|tail length")

def pull_embeddings(h5, rows, n_total):
    mask = np.zeros(n_total, bool)
    mask[rows] = True
    chunks = []
    with tb.open_file(str(h5)) as fp:
        embeddings = fp.root.ctx_ptn
        for start in range(0, embeddings.shape[0], 500_000):
            end = min(start + 500_000, embeddings.shape[0])
            if mask[start:end].any():
                chunks.append(embeddings[start:end][mask[start:end]])
    return np.ascontiguousarray(np.concatenate(chunks).astype(np.float32))

def build_heldout_searches():
    predictions = pl.read_csv(TEST_OUT / "predictions.tsv", separator="\t", columns=["Protein"]).with_row_index("row")
    annotations = pl.concat([pl.read_parquet(p, columns=["Protein", "Protein Classification", "Function"])
                             for p in TRAIN_ANNOTATIONS]).unique("Protein")
    query_sets = {
        "test_avglike": predictions.join(annotations, on="Protein", how="inner")
                                   .filter(pl.col("Protein Classification").is_in(CAT3)).sort("row"),
        "test_struct": predictions.join(annotations.filter(pl.col("Function").is_not_null()
                                                           & pl.col("Function").str.contains(STRUCT_PATTERN)
                                                           & ~pl.col("Protein Classification").is_in(CAT3)),
                                        on="Protein", how="inner").sort("row")}
    index = faiss.read_index(str(TRAIN_INDEX))
    index.nprobe = 64
    for name, queries in query_sets.items():
        embeddings = pull_embeddings(TEST_OUT / "combined_proteins.filtered.PST-EMBED.h5", queries["row"].to_numpy(),
                                     predictions.height)
        D, I = index.search(embeddings, 50)
        np.save(NOVEL / f"{name}_I.npy", I)
        np.save(NOVEL / f"{name}_D.npy", D.astype(np.float32))
        queries.select(["Protein", "Protein Classification", "Function"]).write_parquet(NOVEL / f"{name}_meta.parquet")

if not all((NOVEL / f"{name}_{part}").exists() for name in HELDOUT_SEARCHES for part in ["I.npy", "D.npy", "meta.parquet"]):
    build_heldout_searches()
for name in HELDOUT_SEARCHES:
    n_queries, k = np.load(NOVEL / f"{name}_I.npy", mmap_mode="r").shape
    print(f"{name}: {n_queries:,} proteins, {k} training neighbors each")

test_avglike: 415,180 proteins, 50 training neighbors each
test_struct: 172,379 proteins, 50 training neighbors each


The same search with the frozen ESM-2 (`esm2_t30_150M`) embeddings CheckAMG-PST was finetuned from, over each distinct held-out AVG-like protein and an IVF index of the same training proteins. The summary compares the category of the nearest AVG-like neighbor from both searches on the proteins where both reach one.

In [ ]:
ESM_QUERIES, ESM_NEIGHBORS, ESM_SUMMARY = NOVEL / "esm_query_names.parquet", NOVEL / "esm_query_I.npy", NOVEL / "step6b_nn.npz"
TRAINING = TRAIN_DIR.parent
TEST_SPLITS = ["equal_pos", "equal_source", "half_virus_host", "host_enriched", "input",
               "mge_enriched", "near_all_host", "near_all_virus", "provirus", "virus_enriched"]

def build_esm_searches():
    annotations = (pl.concat([pl.read_parquet(p, columns=["Protein", "Protein Classification", "Function"])
                              for p in TRAIN_ANNOTATIONS]).unique("Protein")
                     .filter(pl.col("Protein Classification").is_in(CAT3)))
    category = dict(zip(annotations["Protein"].to_list(), annotations["Protein Classification"].to_list()))
    function = dict(zip(annotations["Protein"].to_list(), annotations["Function"].fill_null("").to_list()))
    seen, vectors = {}, []
    for split in TEST_SPLITS:
        names = [line[1:].split()[0] for line in open(TRAINING / "test_data" / f"test_{split}_ptns.faa") if line[0] == ">"]
        h5 = TRAINING / "esm" / f"esm_test_data_test_{split}" / f"esm_test_data_test_{split}_esm2_t30_150M_results.h5"
        with tb.open_file(str(h5)) as fp:
            embeddings = fp.root.data[:]
        assert embeddings.shape[0] == len(names), f"{split}: {embeddings.shape[0]} embeddings for {len(names)} proteins"
        for i, name in enumerate(names):
            if name in category and name not in seen:
                seen[name] = (category[name], function[name])
                vectors.append(embeddings[i])
        del embeddings
    queries = pl.DataFrame({"Protein": list(seen), "true_cat": [v[0] for v in seen.values()],
                            "true_fn": [v[1] for v in seen.values()]})
    n_train = pl.scan_parquet(TRAIN_ORDER).select(pl.len()).collect().item()
    with tb.open_file(str(TRAIN_DIR / "checkAMG_train_esm2_t30_150M.graphfmt.h5")) as fp:
        keep = ~np.repeat(fp.root.scaffold_val_mask[:], fp.root.sizes[:])
    assert int(keep.sum()) == n_train
    train_esm = np.lib.format.open_memmap(NOVEL / "esm_train_nonval.f32.npy", mode="w+", dtype=np.float32, shape=(n_train, 640))
    with tb.open_file(str(TRAIN_DIR / "checkAMG_train_esm2_t30_150M.graphfmt.h5")) as fp:
        data, written = fp.root.data, 0
        for start in range(0, data.shape[0], 1_000_000):
            end = min(start + 1_000_000, data.shape[0])
            if keep[start:end].any():
                block = data[start:end][keep[start:end]]
                train_esm[written:written + block.shape[0]] = block
                written += block.shape[0]
    train_esm.flush()
    quantizer = faiss.IndexFlatL2(640)
    index = faiss.IndexIVFFlat(quantizer, 640, 2048, faiss.METRIC_L2)
    sample = np.sort(np.random.default_rng(0).choice(n_train, size=min(400_000, n_train), replace=False))
    index.train(np.ascontiguousarray(train_esm[sample].astype(np.float32)))
    for start in range(0, n_train, 2_000_000):
        index.add(np.ascontiguousarray(train_esm[start:start + 2_000_000].astype(np.float32)))
    index.nprobe = 32
    D, I = index.search(np.ascontiguousarray(np.stack(vectors).astype(np.float32)), 50)
    queries.write_parquet(ESM_QUERIES)
    np.save(ESM_NEIGHBORS, I)
    np.save(NOVEL / "esm_query_D.npy", D.astype(np.float32))

def build_esm_summary():
    train_classes = (pl.read_parquet(TRAIN_ORDER, columns=["Protein Classification"])["Protein Classification"]
                     .fill_null("unclassified").to_numpy().astype(str))
    def nearest_avglike_category(neighbors):
        classes = train_classes[neighbors]
        avglike = np.isin(classes, CAT3)
        found = avglike.any(1)
        first = np.where(found, avglike.argmax(1), 0)
        return np.where(found, classes[np.arange(neighbors.shape[0]), first], None), found
    pst_meta = pl.read_parquet(NOVEL / "test_avglike_meta.parquet").with_row_index("pst_row")
    matched = (pl.read_parquet(ESM_QUERIES).with_row_index("esm_row")
                 .join(pst_meta.group_by("Protein").agg(pl.col("pst_row").min()), on="Protein", how="inner"))
    truth = matched["true_cat"].to_numpy()
    esm_category, esm_found = nearest_avglike_category(np.load(ESM_NEIGHBORS)[matched["esm_row"].to_numpy()])
    pst_category, pst_found = nearest_avglike_category(np.load(NOVEL / "test_avglike_I.npy")[matched["pst_row"].to_numpy()])
    both = esm_found & pst_found
    np.savez(ESM_SUMMARY, pst_prec=float((pst_category[both] == truth[both]).mean()),
             esm_prec=float((esm_category[both] == truth[both]).mean()), n_matched=int(both.sum()))

if not (ESM_QUERIES.exists() and ESM_NEIGHBORS.exists()):
    build_esm_searches()
if not ESM_SUMMARY.exists():
    build_esm_summary()
n_queries, k = np.load(ESM_NEIGHBORS, mmap_mode="r").shape
print(f"frozen ESM-2 search: {n_queries:,} held-out AVG-like proteins, {k} training neighbors each")

frozen ESM-2 search: 139,856 held-out AVG-like proteins, 50 training neighbors each


The applied query set is every soil and human-gut AVG called by `annotate` or de novo, and the held-out query set is every distinct AVG-like test protein, each with its CheckAMG-PST embedding.

In [ ]:
APPLIED_KEYS, APPLIED_EMB = CACHE / "applied_keys.parquet", CACHE / "applied_emb.npy"
EVAL_KEYS, EVAL_EMB = CACHE / "eval_keys.parquet", CACHE / "eval_emb.npy"

def applied_avg_calls():
    thr = json.loads((FILES / "pst_thresholds.json").read_text())
    columns = ["Protein", "Genome", "Classification (annotate)", "Viral Confidence Level (annotate)", "Function (annotate)"]
    calls = pl.concat([
        pl.read_parquet(agg(ds), columns=columns)
          .join(pl.read_csv(den(ds) / "predictions.tsv", separator="\t",
                            columns=["Protein", "Viral prob", "Final AVG prob"]), on="Protein", how="left")
          .with_columns(pl.lit(ds).alias("dataset")) for ds in DATASETS])
    calls = calls.with_columns(
        (pl.col("Classification (annotate)").is_in(CAT3)
         & pl.col("Viral Confidence Level (annotate)").is_in(VIRAL_OK)).alias("annotate_avg"),
        ((pl.col("Final AVG prob") >= thr["AVG"]["medium"])
         & (pl.col("Viral prob") >= thr["Viral"]["medium"])).alias("denovo_avg"))
    calls = calls.with_columns(
        (pl.col("denovo_avg") & ~pl.col("annotate_avg")
         & (pl.col("Function (annotate)").is_null() | (pl.col("Function (annotate)") == ""))
         ).alias("homology_invisible"))
    return calls.filter(pl.col("annotate_avg") | pl.col("denovo_avg"))

def build_applied_queries():
    calls = applied_avg_calls()
    embeddings, keys = [], []
    for ds in DATASETS:
        pred = pl.read_csv(den(ds) / "predictions.tsv", separator="\t", columns=["Protein"]).with_row_index("row")
        want = pred.join(calls.filter(pl.col("dataset") == ds).select("Protein"), on="Protein", how="inner").sort("row")
        embeddings.append(pull_embeddings(den(ds) / "combined_proteins.filtered.PST-EMBED.h5", want["row"].to_numpy(), pred.height))
        keys.append(want.select("Protein"))
    keys = pl.concat(keys).join(
        calls.select(["Protein", "Genome", "dataset", "annotate_avg", "denovo_avg", "homology_invisible",
                      pl.col("Classification (annotate)").alias("annotate_classification"),
                      pl.col("Function (annotate)").alias("annotate_function")]), on="Protein", how="left")
    embeddings = np.concatenate(embeddings)
    assert keys.height == embeddings.shape[0]
    keys.write_parquet(APPLIED_KEYS)
    np.save(APPLIED_EMB, embeddings)

def build_eval_queries():
    pred = pl.read_csv(TEST_OUT / "predictions.tsv", separator="\t", columns=["Protein"]).with_row_index("row")
    annotations = pl.concat([pl.read_parquet(p, columns=["Protein", "Protein Classification", "Function"])
                             for p in TRAIN_ANNOTATIONS]).unique("Protein")
    heldout_avgs = (pred.join(annotations, on="Protein", how="inner")
                        .filter(pl.col("Protein Classification").is_in(CAT3))
                        .unique("Protein", keep="first").sort("row"))
    embeddings = pull_embeddings(TEST_OUT / "combined_proteins.filtered.PST-EMBED.h5", heldout_avgs["row"].to_numpy(), pred.height)
    heldout_avgs.select(["Protein", pl.col("Protein Classification").alias("true_category"),
                         pl.col("Function").alias("true_specific")]).write_parquet(EVAL_KEYS)
    np.save(EVAL_EMB, embeddings)

if not (APPLIED_KEYS.exists() and APPLIED_EMB.exists()):
    build_applied_queries()
if not (EVAL_KEYS.exists() and EVAL_EMB.exists()):
    build_eval_queries()
print(f"applied AVGs with a CheckAMG-PST embedding: {pl.scan_parquet(APPLIED_KEYS).select(pl.len()).collect().item():,}")
print(f"distinct held-out AVG-like proteins: {pl.scan_parquet(EVAL_KEYS).select(pl.len()).collect().item():,}")

applied AVGs with a CheckAMG-PST embedding: 541,007
distinct held-out AVG-like proteins: 139,856


The labeled reference is every AVG-like training protein with a specific function, with its embedding copied from the training index into an IVF index of its own. The L1 of a specific function is the one assigned by the most HMMs with that name, ties broken alphabetically.

In [ ]:
LABELED_REFERENCE, LABELED_INDEX = CACHE / "labeled_reference.parquet", CACHE / "labeled.index.faiss"
LABELED_VECTORS = CACHE / "labeled_vectors.npy"

def build_labeled_reference():
    order = pl.read_parquet(TRAIN_ORDER)
    labeled_rows = np.where((order["Protein Classification"].is_in(CAT3) & order["Function"].is_not_null()
                             & (order["Function"] != "")).to_numpy())[0].astype(np.int64)
    labeled = order[labeled_rows].select(["Protein", "Protein Classification", "Function"]).rename(
        {"Protein": "nn_ref_id", "Protein Classification": "ref_category", "Function": "ref_specific"})
    names = pl.concat([
        pl.read_csv(FILES / f"{t}_categorized.tsv", separator="\t", infer_schema_length=0)
          .select(["name", "category_L1"]) for t in ["AMGs", "APGs", "AReGs"]]).drop_nulls("name")
    canonical_l1 = (names.filter(pl.col("category_L1").is_not_null()).group_by(["name", "category_L1"]).len()
                         .sort(["name", "len", "category_L1"], descending=[False, True, False])
                         .group_by("name").first().select(["name", "category_L1"]))
    (labeled.join(canonical_l1.rename({"name": "ref_specific", "category_L1": "ref_L1"}), on="ref_specific", how="left")
            .with_row_index("lab_row").write_parquet(LABELED_REFERENCE))
    full_index = faiss.read_index(str(TRAIN_INDEX))
    full_index.make_direct_map()
    vectors = np.empty((labeled_rows.size, full_index.d), dtype=np.float32)
    for j, row in enumerate(labeled_rows):
        vectors[j] = full_index.reconstruct(int(row))
    del full_index
    gc.collect()
    quantizer = faiss.IndexFlatL2(vectors.shape[1])
    labeled_index = faiss.IndexIVFFlat(quantizer, vectors.shape[1], 4096, faiss.METRIC_L2)
    sample = np.sort(np.random.default_rng(0).choice(vectors.shape[0], size=min(400_000, vectors.shape[0]), replace=False))
    labeled_index.train(np.ascontiguousarray(vectors[sample]))
    labeled_index.add(vectors)
    faiss.write_index(labeled_index, str(LABELED_INDEX))
    np.save(LABELED_VECTORS, vectors)

if not all(p.exists() for p in [LABELED_REFERENCE, LABELED_INDEX, LABELED_VECTORS]):
    build_labeled_reference()
print(f"labeled reference proteins: {pl.scan_parquet(LABELED_REFERENCE).select(pl.len()).collect().item():,}")

labeled reference proteins: 959,449


The 50 nearest labeled references of every query protein and the neighbor features the correctness model uses. A reference's density is its mean distance to the labeled references with the same specific label among its 100 nearest, and it normalizes the nearest-neighbor distance.

In [ ]:
FEATURE_SETS = [("applied", APPLIED_KEYS, APPLIED_EMB), ("eval", EVAL_KEYS, EVAL_EMB)]
REF_DENSITY = CACHE / "ref_density.npy"

def build_features():
    labeled = pl.read_parquet(LABELED_REFERENCE)
    ref_specific = labeled["ref_specific"].to_numpy().astype(str)
    ref_category = labeled["ref_category"].to_numpy().astype(str)
    ref_l1 = labeled["ref_L1"].fill_null("").to_numpy().astype(str)
    ref_id = labeled["nn_ref_id"].to_numpy().astype(str)
    labeled_index = faiss.read_index(str(LABELED_INDEX))
    labeled_index.nprobe = 64
    neighbors = {}
    for name, _, emb_path in FEATURE_SETS:
        fi, fd = CACHE / f"{name}_I.npy", CACHE / f"{name}_D.npy"
        if not (fi.exists() and fd.exists()):
            D, I = labeled_index.search(np.ascontiguousarray(np.load(emb_path).astype(np.float32)), 50)
            np.save(fi, I)
            np.save(fd, D.astype(np.float32))
        neighbors[name] = (np.load(fi), np.load(fd).astype(np.float64))
    if not REF_DENSITY.exists():
        Dr, Ir = labeled_index.search(np.ascontiguousarray(np.load(LABELED_VECTORS)), 101)
        same = ref_specific[Ir[:, 1:]] == ref_specific[:, None]
        with np.errstate(invalid="ignore"):
            density = np.nanmean(np.where(same, Dr[:, 1:], np.nan), axis=1)
        median = float(np.nanmedian(density))
        np.save(REF_DENSITY, np.where(np.isfinite(density) & (density > 0), density, median))
    density = np.load(REF_DENSITY)
    hub_count = np.bincount(np.concatenate([neighbors["applied"][0][:, 0], neighbors["eval"][0][:, 0]]), minlength=ref_id.size)
    for name, keys_path, _ in FEATURE_SETS:
        out_path = CACHE / f"{name}_features.parquet"
        if out_path.exists():
            continue
        I, D = neighbors[name]
        nn = I[:, 0]
        d1 = D[:, 0]
        raw_specific = ref_specific[nn]
        neighbor_specific = ref_specific[I]
        d2 = np.where(neighbor_specific != raw_specific[:, None], D, np.inf).min(1)
        s = D.std(1, keepdims=True)
        s[s == 0] = 1.0
        w = np.exp(-D / s)
        same = (neighbor_specific == raw_specific[:, None]).astype(np.float64)
        features = {
            "protein_row": np.arange(I.shape[0]),
            "nn_ref_id": ref_id[nn],
            "d1": d1,
            "d2": np.where(np.isfinite(d2), d2, np.nan),
            "margin": np.where(np.isfinite(d2), d2 - d1, np.nan),
            "ratio": np.where(np.isfinite(d2) & (d2 > 0), d1 / np.where(d2 > 0, d2, np.nan), np.nan),
            "raw_specific": raw_specific,
            "raw_category": ref_category[nn],
            "raw_L1": np.where(ref_l1[nn] == "", None, ref_l1[nn]),
            "density_norm_d1": d1 / density[nn]}
        for k in (5, 20, 50):
            features[f"vote_purity_k{k}"] = (w[:, :k] * same[:, :k]).sum(1) / np.clip(w[:, :k].sum(1), 1e-12, None)
        features["hub_count"] = hub_count[nn]
        (pl.read_parquet(keys_path).with_row_index("protein_row").join(pl.DataFrame(features), on="protein_row", how="left")
           .drop("protein_row").write_parquet(out_path))

if not all((CACHE / f"{name}_{part}").exists() for name, _, _ in FEATURE_SETS
           for part in ["I.npy", "D.npy", "features.parquet"]) or not REF_DENSITY.exists():
    build_features()
for name, _, _ in FEATURE_SETS:
    print(f"{name} features: {pl.scan_parquet(CACHE / f'{name}_features.parquet').select(pl.len()).collect().item():,} proteins")

applied features: 541,007 proteins
eval features: 139,856 proteins


Maximum identity from each held-out protein to any training protein, over the all-against-all alignment.

In [ ]:
EVAL_IDENTITY = CACHE / "eval_identity.parquet"
ALL_AGAINST_ALL = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/"
                       "genomad_progenomes_combined.alignments.parquet")

def build_eval_identity():
    import duckdb
    spill = CACHE / "duckdb_tmp"
    spill.mkdir(exist_ok=True)
    con = duckdb.connect()
    con.execute("PRAGMA threads=48")
    con.execute("PRAGMA memory_limit='300GB'")
    con.execute(f"PRAGMA temp_directory='{spill}'")
    con.execute(f"""
    COPY (
      SELECT a.query AS Protein, CAST(MAX(a.fident) AS FLOAT) AS max_ident_to_train
      FROM read_parquet('{ALL_AGAINST_ALL}') a
      SEMI JOIN read_parquet('{EVAL_KEYS}') heldout ON a.query = heldout.Protein
      SEMI JOIN read_parquet('{TRAIN_ORDER}') train ON a.target = train.Protein
      WHERE a.query <> a.target
      GROUP BY a.query
      ORDER BY a.query
    ) TO '{EVAL_IDENTITY}' (FORMAT PARQUET)
    """)

if not EVAL_IDENTITY.exists():
    build_eval_identity()
print(f"held-out proteins with an alignment to a training protein: {pl.scan_parquet(EVAL_IDENTITY).select(pl.len()).collect().item():,}")

held-out proteins with an alignment to a training protein: 139,356


Sequences of the sequence-similarity invisible AVGs, extracted once and cached. The MMseqs2 search below and the DefenseFinder scan in Section 13 read them.

In [ ]:
DFDIR = CACHE / "defensefinder"
DFDIR.mkdir(exist_ok=True)
FAA = DFDIR / "homology_invisible.faa"
hi_ids = pl.read_parquet(APPLIED_KEYS).filter(pl.col("homology_invisible")).select(["Protein", "dataset"])
if not FAA.exists():
    want = {ds: set(hi_ids.filter(pl.col("dataset") == ds)["Protein"].to_list()) for ds in DATASETS}
    n = 0
    with open(FAA, "w") as o:
        for ds in DATASETS:
            keep, buf = False, []
            with open(den(ds) / "combined_proteins.filtered.faa") as f:
                for line in f:
                    if line[0] == ">":
                        if keep and buf:
                            o.write("".join(buf))
                        pid = line[1:].split()[0]
                        keep = pid in want[ds]
                        buf = [">" + pid + "\n"] if keep else []
                        if keep:
                            n += 1
                    elif keep:
                        buf.append(line)
            if keep and buf:
                o.write("".join(buf))
    print(f"wrote {FAA.name} ({n:,} sequences)")
else:
    print(f"{FAA.name} cached ({sum(1 for l in open(FAA) if l[0]=='>'):,} sequences)")

homology_invisible.faa cached (109,107 sequences)


MMseqs2 alignments of the sequence-similarity invisible AVGs to all training proteins, and each protein's maximum identity.

In [ ]:
HI_ALIGNMENTS, APPLIED_IDENTITY = CACHE / "hi_vs_train_alignments.tsv", CACHE / "applied_identity.parquet"
# The cached alignments were made with this MMseqs2 release (14.7e284)
MMSEQS_SEARCH_BIN = Path("/storage2/scratch/kosmopoulos/miniconda3/envs/mmseqs2/bin/mmseqs")

def build_applied_identity():
    work = CACHE / "mmseqs_applied"
    work.mkdir(exist_ok=True)
    def mmseqs(*args):
        subprocess.run([str(MMSEQS_SEARCH_BIN), *map(str, args), "-v", "1"], check=True, cwd=work)
    if not HI_ALIGNMENTS.exists():
        mmseqs("createdb", TRAIN_DIR / "train_ptns.faa", "targetDB", "--shuffle", "0")
        mmseqs("createdb", FAA, "queryDB", "--shuffle", "0")
        mmseqs("search", "queryDB", "targetDB", "resDB", "tmp_s", "--threads", "128", "-s", "5.7", "-e", "1e-3", "--max-seqs", "50")
        mmseqs("convertalis", "queryDB", "targetDB", "resDB", HI_ALIGNMENTS,
               "--format-output", "query,target,fident,evalue,bits,qcov,tcov", "--threads", "128")
    alignments = pl.read_csv(HI_ALIGNMENTS, separator="\t", has_header=False,
                             new_columns=["query", "target", "fident", "evalue", "bits", "qcov", "tcov"])
    (alignments.group_by("query").agg(pl.col("fident").max().cast(pl.Float32).alias("max_ident_to_train"))
               .rename({"query": "Protein"}).sort("Protein").write_parquet(APPLIED_IDENTITY))

if not (HI_ALIGNMENTS.exists() and APPLIED_IDENTITY.exists()):
    build_applied_identity()
print(f"sequence-similarity invisible AVGs with an alignment to a training protein: "
      f"{pl.scan_parquet(APPLIED_IDENTITY).select(pl.len()).collect().item():,}")

sequence-similarity invisible AVGs with an alignment to a training protein: 100,556


MMseqs2 clusterings at 30, 50, 70, and 90 percent identity and 80 percent coverage, over two universes: the labeled reference with the held-out proteins, for evaluation, and the labeled reference with every applied AVG. Two applied AVGs are absent from the biome-filtered FASTAs the applied sequences come from, and stay unclustered.

In [ ]:
import shutil
CLUSTER_OPTIONS = ["-c", "0.8", "--cov-mode", "0", "-s", "7.5"]
# Universe, output file pattern, and the thread count each clustering was run with
CLUSTER_RUNS = [("universe", CLU_EVAL, 128), ("universe_all", CLU_APPLIED, 50)]
TRAIN_FAA = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/"
                 "genomad_progenomes_combined.faa")

def extract_fasta(paths, want, out_path):
    written = set()
    with open(out_path, "w") as out:
        for path in paths:
            keep = False
            with open(path) as f:
                for line in f:
                    if line[0] == ">":
                        pid = line[1:].split(None, 1)[0]
                        keep = pid in want and pid not in written
                        if keep:
                            written.add(pid)
                            line = ">" + pid + "\n"
                    if keep:
                        out.write(line)
    return written

def build_clusterings():
    work = CACHE / "seqclust_work"
    (work / "db").mkdir(parents=True, exist_ok=True)
    SEQCLUST.mkdir(exist_ok=True)
    if not ((work / "universe.faa").exists() and (work / "universe_all.faa").exists()):
        reference_ids = set(pl.read_parquet(LABELED_REFERENCE, columns=["nn_ref_id"])["nn_ref_id"].to_list())
        heldout_ids = set(pl.read_parquet(EVAL_KEYS, columns=["Protein"])["Protein"].to_list())
        applied_ids = set(applied_avg_calls()["Protein"].to_list())
        found = extract_fasta([TRAIN_FAA], reference_ids | heldout_ids, work / "universe.faa")
        assert not (reference_ids | heldout_ids) - found
        found = extract_fasta([TRAIN_FAA] + [MAIN / f"IMGVR5_UViG.{ds}.filtered.faa" for ds in DATASETS],
                              reference_ids | applied_ids, work / "universe_all.faa")
        assert not reference_ids - found and len(applied_ids - found) <= 10
    for universe, pattern, threads in CLUSTER_RUNS:
        if not (work / "db" / f"{universe}.dbtype").exists():
            subprocess.run([str(MMSEQS_BIN), "createdb", f"{universe}.faa", f"db/{universe}", "--shuffle", "0", "-v", "1"],
                           check=True, cwd=work)
        for p in PCTS:
            out = SEQCLUST / pattern.format(p=p)
            if out.exists():
                continue
            name = out.stem
            subprocess.run([str(MMSEQS_BIN), "cluster", f"db/{universe}", f"db/{name}", f"tmp_{name}",
                            "--min-seq-id", f"0.{p}", *CLUSTER_OPTIONS, "--threads", str(threads), "-v", "1"],
                           check=True, cwd=work)
            subprocess.run([str(MMSEQS_BIN), "createtsv", f"db/{universe}", f"db/{universe}", f"db/{name}", str(out),
                            "--threads", "32", "-v", "1"], check=True, cwd=work)
            shutil.rmtree(work / f"tmp_{name}")

if not all((SEQCLUST / pattern.format(p=p)).exists() for _, pattern, _ in CLUSTER_RUNS for p in PCTS):
    build_clusterings()
print("clusterings present:", ", ".join(pattern.format(p=p) for _, pattern, _ in CLUSTER_RUNS for p in PCTS))

clusterings present: clu_id30.tsv, clu_id50.tsv, clu_id70.tsv, clu_id90.tsv, clu_all_id30.tsv, clu_all_id50.tsv, clu_all_id70.tsv, clu_all_id90.tsv


## 1. Split integrity

Held-out precision estimates performance on new proteins only if the split separates protein families as well as genomes. The criterion is that fewer than 5 percent of held-out proteins exceed 50 percent sequence identity to any training protein.

Identity comes from the existing all-against-all MMseqs2 alignment of the combined geNomad and proGenomes protein set (`genomad_progenomes_combined.alignments.parquet`), restricted to held-out query against training target.

Maximum sequence identity from each held-out protein to any training protein.

In [4]:
ev_id = pl.read_parquet(CACHE / "eval_identity.parquet")
ev_keys = pl.read_parquet(CACHE / "eval_keys.parquet")
ident = ev_keys.select("Protein").join(ev_id, on="Protein", how="left").with_columns(
    pl.col("max_ident_to_train").fill_null(0.0))
N = ident.height
max_identity = ident["max_ident_to_train"].to_numpy()
print(f"held-out AVG-like proteins: {N:,}")
print(f"with a detectable alignment to a training protein: {int((max_identity>0).sum()):,} ({100*(max_identity>0).mean():.1f}%)")
for t in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]:
    print(f"  max identity > {t:.0%}: {int((max_identity>t).sum()):>7,} ({100*(max_identity>t).mean():5.1f}%)")
print(f"median {np.median(max_identity):.3f}  mean {max_identity.mean():.3f}")
SPLIT_PASS = bool((max_identity > 0.5).mean() < 0.05)
print(f"\npre-registered criterion (<5% above 50% identity): {'PASS' if SPLIT_PASS else 'FAIL'}")

held-out AVG-like proteins: 139,856
with a detectable alignment to a training protein: 139,356 (99.6%)
  max identity > 90%:  15,177 ( 10.9%)
  max identity > 80%:  27,527 ( 19.7%)
  max identity > 70%:  51,761 ( 37.0%)
  max identity > 60%:  85,361 ( 61.0%)
  max identity > 50%: 116,789 ( 83.5%)
  max identity > 40%: 133,650 ( 95.6%)
  max identity > 30%: 138,317 ( 98.9%)
median 0.645  mean 0.657

pre-registered criterion (<5% above 50% identity): FAIL


The criterion is not met, so the split is not family-disjoint. `accessory_scripts/train_test_split.py` splits whole genome clusters, so no genome cluster spans train and test, but conserved protein families recur across unrelated genomes. A family-disjoint split would require retraining CheckAMG-PST, so every precision estimate below is also reported by identity to the training set.

Where the applied proteins sit on the same identity axis, which determines which stratum transfers to them.

In [5]:
ap_id = pl.read_parquet(CACHE / "applied_identity.parquet")
ap_keys = pl.read_parquet(CACHE / "applied_keys.parquet")
hi_keys = ap_keys.filter(pl.col("homology_invisible")).select("Protein")
applied_max_identity = hi_keys.join(ap_id, on="Protein", how="left").with_columns(
    pl.col("max_ident_to_train").fill_null(0.0))["max_ident_to_train"].to_numpy()
print(f"sequence-similarity invisible applied AVGs: {applied_max_identity.size:,}")
print(f"with a detectable alignment to a training protein: {int((applied_max_identity>0).sum()):,} ({100*(applied_max_identity>0).mean():.1f}%)")
for t in [0.9, 0.7, 0.5, 0.3]:
    print(f"  max identity > {t:.0%}: {int((applied_max_identity>t).sum()):>7,} ({100*(applied_max_identity>t).mean():5.1f}%)")
print(f"median {np.median(applied_max_identity):.3f}  (held-out median {np.median(max_identity):.3f})")
print(f"\napplied-vs-heldout median identity gap: {np.median(applied_max_identity) - np.median(max_identity):+.3f}")

sequence-similarity invisible applied AVGs: 109,107
with a detectable alignment to a training protein: 100,556 (92.2%)
  max identity > 90%:  24,433 ( 22.4%)
  max identity > 70%:  39,050 ( 35.8%)
  max identity > 50%:  71,671 ( 65.7%)
  max identity > 30%:  97,827 ( 89.7%)
median 0.583  (held-out median 0.645)

applied-vs-heldout median identity gap: -0.062


## 2. The labeled reference and the label hierarchy

Both tiers draw labels from the same labeled reference set, and both emit labels at the same three granularities, so the reference and the hierarchy are defined once here and used by everything below.

Quantify the multi-parent structure of the curated hierarchy, which is why L2 is excluded.

In [6]:
cz_all = pl.concat([
    pl.read_csv(FILES / f"{t}_categorized.tsv", separator="\t", infer_schema_length=0)
      .select(["name", "category_L1", "category_L2"]) for t in ["AMGs", "APGs", "AReGs"]]).drop_nulls("name")
tref_fn = pl.read_parquet(NOVEL / "train_ref_index_order.parquet",
                          columns=["Protein Classification", "Function"])
tf = tref_fn.filter(pl.col("Protein Classification").is_in(CAT3)
                    & pl.col("Function").is_not_null() & (pl.col("Function") != ""))
fn_n = tf.group_by("Function").len()
par = (cz_all.unique(["name", "category_L1", "category_L2"]).group_by("name")
       .agg(pl.col("category_L1").n_unique().alias("nL1"), pl.col("category_L2").n_unique().alias("nL2")))
amb = fn_n.join(par, left_on="Function", right_on="name", how="inner")
HIER = {"training_functions": fn_n.height, "labeled_training_proteins": int(fn_n["len"].sum()),
        "multi_L1_functions": amb.filter(pl.col("nL1") > 1).height,
        "multi_L1_proteins": int(amb.filter(pl.col("nL1") > 1)["len"].sum()),
        "multi_L2_functions": amb.filter(pl.col("nL2") > 1).height,
        "multi_L2_proteins": int(amb.filter(pl.col("nL2") > 1)["len"].sum())}
HIER["multi_L1_function_pct"] = 100 * HIER["multi_L1_functions"] / HIER["training_functions"]
HIER["multi_L2_function_pct"] = 100 * HIER["multi_L2_functions"] / HIER["training_functions"]
HIER["multi_L1_protein_pct"] = 100 * HIER["multi_L1_proteins"] / HIER["labeled_training_proteins"]
HIER["multi_L2_protein_pct"] = 100 * HIER["multi_L2_proteins"] / HIER["labeled_training_proteins"]
print(f"distinct training function names: {HIER['training_functions']:,}")
print(f"  >1 L1 parent: {HIER['multi_L1_functions']:,} ({HIER['multi_L1_function_pct']:.1f}%), "
      f"affecting {HIER['multi_L1_protein_pct']:.1f}% of labeled training proteins")
print(f"  >1 L2 parent: {HIER['multi_L2_functions']:,} ({HIER['multi_L2_function_pct']:.1f}%), "
      f"affecting {HIER['multi_L2_protein_pct']:.1f}% of labeled training proteins")

distinct training function names: 9,906
  >1 L1 parent: 1,848 (18.7%), affecting 23.5% of labeled training proteins
  >1 L2 parent: 2,115 (21.4%), affecting 29.1% of labeled training proteins


The label hierarchy, and the reference frequency of each specific label.

In [7]:
reference = pl.read_parquet(CACHE / "labeled_reference.parquet")
EX2L1 = dict(zip(reference["ref_specific"].to_list(), reference["ref_L1"].to_list()))
EX2CAT = (reference.group_by("ref_specific").agg(pl.col("ref_category").mode().sort().first())
            .to_dict(as_series=False))
EX2CAT = dict(zip(EX2CAT["ref_specific"], EX2CAT["ref_category"]))
freq = reference.group_by("ref_specific").len().rename({"len": "ref_n"})
print(f"labeled reference proteins {reference.height:,} over {freq.height:,} specific labels")
print(f"specific labels resolving to an L1: {sum(v is not None for v in EX2L1.values()):,}")
q1, q2 = freq["ref_n"].quantile(1/3), freq["ref_n"].quantile(2/3)
freq = freq.with_columns(pl.when(pl.col("ref_n") <= q1).then(pl.lit("rare"))
                          .when(pl.col("ref_n") <= q2).then(pl.lit("mid"))
                          .otherwise(pl.lit("common")).alias("freq_tercile"))
print(f"frequency tercile cuts: rare <= {q1:.0f} < mid <= {q2:.0f} < common")
print(freq.group_by("freq_tercile").agg(pl.len().alias("labels"), pl.col("ref_n").sum().alias("proteins")))

labeled reference proteins 959,449 over 9,906 specific labels
specific labels resolving to an L1: 9,906
frequency tercile cuts: rare <= 5 < mid <= 40 < common
shape: (3, 3)
┌──────────────┬────────┬──────────┐
│ freq_tercile ┆ labels ┆ proteins │
│ ---          ┆ ---    ┆ ---      │
│ str          ┆ u64    ┆ u64      │
╞══════════════╪════════╪══════════╡
│ mid          ┆ 3224   ┆ 55064    │
│ rare         ┆ 3407   ┆ 7995     │
│ common       ┆ 3275   ┆ 896390   │
└──────────────┴────────┴──────────┘


Normalize function strings so that formatting differences are not scored as errors. Both tiers and the ground truth pass through this same function.

In [8]:
import re
def normfn(a):
    out = []
    for s in a:
        s = re.sub(r"\s*\[EC:[^\]]*\]", "", s)
        s = re.sub(r"^[A-Za-z0-9_/]+(,\s*[A-Za-z0-9_/.]+)*;\s*", "", s)
        out.append(re.sub(r"\s+", " ", s).strip().lower())
    return np.array(out)

reference = reference.with_columns(pl.Series("ref_specific_n", normfn(reference["ref_specific"].fill_null("").to_numpy().astype(str))))
NORM2DISP = (reference.filter(pl.col("ref_specific_n") != "")
                .group_by("ref_specific_n").agg(pl.col("ref_specific").mode().sort().first())
                .to_dict(as_series=False))
NORM2DISP = dict(zip(NORM2DISP["ref_specific_n"], NORM2DISP["ref_specific"]))
print(f"distinct specific labels: {reference['ref_specific'].n_unique():,} raw, {len(NORM2DISP):,} after normalization")

distinct specific labels: 9,906 raw, 9,189 after normalization


### 2.4 The evaluation protocol

The held-out feature frame carries the deployed 1-NN label of each held-out protein, its embedding features, its ground-truth labels at all three granularities, and its maximum identity to any training protein. Held-out proteins are split three ways by protein: **fit** (40 percent), **calibration** (30 percent), and **reporting** (30 percent). Every threshold in this notebook, for both tiers, is chosen on the calibration slice. Every number reported as a result is computed on the reporting slice, which nothing touches until the evaluation sections.

Load the held-out features and attach ground truth at each granularity.

In [9]:
heldout = pl.read_parquet(CACHE / "eval_features.parquet")
heldout = (heldout.join(ev_id, on="Protein", how="left")
          .with_columns(pl.col("max_ident_to_train").fill_null(0.0)))
heldout = heldout.with_columns(
    pl.col("true_specific").replace_strict(EX2L1, default=None).alias("true_L1"),
    pl.col("raw_specific").replace_strict(EX2L1, default=None).alias("raw_L1_chk"))
heldout = heldout.with_columns(
    pl.Series("y_specific", normfn(heldout["true_specific"].fill_null("").to_numpy().astype(str))
                         == normfn(heldout["raw_specific"].fill_null("").to_numpy().astype(str))),
    (pl.col("true_L1") == pl.col("raw_L1")).fill_null(False).alias("y_L1"),
    (pl.col("true_category") == pl.col("raw_category")).fill_null(False).alias("y_category"))
print(f"held-out proteins with labeled-index features: {heldout.height:,}")
for lvl in LEVELS:
    print(f"  raw 1-NN {lvl:9s} precision: {heldout['y_'+lvl].mean():.4f}")

held-out proteins with labeled-index features: 139,856
  raw 1-NN specific  precision: 0.3698
  raw 1-NN L1        precision: 0.5735
  raw 1-NN category  precision: 0.8840


Attach the normalized specific-label columns that both tiers are scored against, then take the three-way split.

In [10]:
heldout = heldout.with_columns(
    pl.Series("true_specific_n", normfn(heldout["true_specific"].fill_null("").to_numpy().astype(str))),
    pl.Series("raw_specific_n", normfn(heldout["raw_specific"].fill_null("").to_numpy().astype(str))),
    pl.col("max_ident_to_train").cast(pl.Float64).round(3))
assert ((heldout["true_specific_n"] == heldout["raw_specific_n"]) == heldout["y_specific"]).all()
print(f"held-out proteins {heldout.height:,}, normalized specific truth non-empty "
      f"{int((heldout['true_specific_n'] != '').sum()):,}")

held-out proteins 139,856, normalized specific truth non-empty 139,856


Three-way protein-level split of the held-out set.

In [11]:
perm = rng.permutation(heldout.height)
n1, n2 = int(0.4 * heldout.height), int(0.7 * heldout.height)
split = np.empty(heldout.height, dtype=object)
split[perm[:n1]] = "fit"
split[perm[n1:n2]] = "calib"
split[perm[n2:]] = "report"
heldout = heldout.with_columns(pl.Series("split", split))
print(heldout.group_by("split").agg(pl.len().alias("n"),
      *[pl.col("y_" + l).mean().round(4).alias(l) for l in LEVELS]).sort("split"))

shape: (3, 5)
┌────────┬───────┬──────────┬────────┬──────────┐
│ split  ┆ n     ┆ specific ┆ L1     ┆ category │
│ ---    ┆ ---   ┆ ---      ┆ ---    ┆ ---      │
│ str    ┆ u64   ┆ f64      ┆ f64    ┆ f64      │
╞════════╪═══════╪══════════╪════════╪══════════╡
│ calib  ┆ 41957 ┆ 0.3658   ┆ 0.5693 ┆ 0.8821   │
│ fit    ┆ 55942 ┆ 0.3694   ┆ 0.573  ┆ 0.8836   │
│ report ┆ 41957 ┆ 0.3743   ┆ 0.5783 ┆ 0.8864   │
└────────┴───────┴──────────┴────────┴──────────┘


## 3. Tier 1: cluster-majority label transfer

Every labeled training protein and every query protein are clustered together in one MMseqs2 run, and a query takes the majority label of the labeled training proteins that land in its cluster. No embedding is involved. Clustering is run at four minimum sequence identities so that the reach and precision of the tier can be reported as a function of how strict the cluster definition is.

Two separate clusterings are used, because the query set differs. The held-out evaluation clusters the labeled training proteins together with the held-out reference proteins, so that a query's ballot is cast only by training proteins and the held-out labels are never used to label each other. The applied assignment clusters the labeled training proteins together with every applied AVG protein. Both are described where they are used.

The MMseqs2 version and clustering command, read at run time from the binary and the clustering logs and recorded in the manifest.

In [ ]:
import subprocess
MMSEQS_VERSION = subprocess.run([str(MMSEQS_BIN), "version"], capture_output=True,
                                text=True).stdout.strip()
assert MMSEQS_VERSION, f"could not read an MMseqs2 version from {MMSEQS_BIN}"
MMSEQS_CMD = " ".join(["mmseqs cluster <universe> <clusters> <tmp> --min-seq-id",
                       "|".join(f"0.{p}" for p in PCTS)] + CLUSTER_OPTIONS)
assert SEQCLUST.exists(), f"clustering cache not found: {SEQCLUST}"
assert reference["nn_ref_id"].n_unique() == reference.height, "training protein ids are not unique"
print("mmseqs version:", MMSEQS_VERSION)
print("clustering invocation:", MMSEQS_CMD)
for p in PCTS:
    for nm in (CLU_EVAL, CLU_APPLIED):
        f = SEQCLUST / nm.format(p=p)
        sz = f"{f.stat().st_size/1e6:8.1f} MB" if f.exists() else "  absent"
        print(f"  {f.name:18s} {sz}")

mmseqs version: 18.8cc5c
clustering invocation: mmseqs cluster <universe> <clusters> <tmp> --min-seq-id 0.30|0.50|0.70|0.90 -c 0.8 --cov-mode 0 -s 7.5
  clu_id30.tsv           84.8 MB
  clu_all_id30.tsv      141.3 MB
  clu_id50.tsv           86.7 MB
  clu_all_id50.tsv      143.4 MB
  clu_id70.tsv           87.7 MB
  clu_all_id70.tsv      144.1 MB
  clu_id90.tsv           87.8 MB
  clu_all_id90.tsv      143.9 MB


Statistical helpers used by every precision number below. `wilson` gives a binomial confidence interval on a precision, and `band` gives a bootstrap interval over proteins for statistics that are not simple proportions.

In [13]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (None, None)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def prec_stat(y):
    y = np.asarray(y, dtype=float)
    n = y.size
    if n == 0:
        return {"n": 0, "precision": None, "lo": None, "hi": None}
    k = float(y.sum())
    lo, hi = wilson(k, n)
    return {"n": int(n), "precision": k / n, "lo": lo, "hi": hi}

def band(y, n_boot=N_BOOT, seed=SEED):
    y = np.asarray(y, dtype=float)
    if y.size == 0:
        return (None, None)
    r = np.random.default_rng(seed)
    b = [y[r.integers(0, y.size, y.size)].mean() for _ in range(n_boot)]
    return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))

print("helpers defined; bootstrap replicates:", N_BOOT)

helpers defined; bootstrap replicates: 500


Read a clustering, and take the majority label of the labeled training proteins in each cluster. A cluster with no labeled member casts no vote. A cluster whose top label is tied with another label casts no vote either, which is a deliberate abstention rather than an arbitrary pick, and it is counted as tier 1 failing to reach that protein.

In [14]:
def load_clu(path):
    return pl.read_csv(path, separator="\t", has_header=False,
                       new_columns=["cluster", "member"], infer_schema_length=0)

def majority(clu, lvl):
    col = REF[lvl]
    j = (clu.join(reference.select(["nn_ref_id", col]), left_on="member", right_on="nn_ref_id", how="inner")
            .filter(pl.col(col).is_not_null() & (pl.col(col) != "")))
    c = j.group_by(["cluster", col]).len()
    top = c.group_by("cluster").agg(
        pl.col(col).sort_by(["len", col], descending=[True, False]).first().alias("vote_label"),
        pl.col("len").max().alias("n_top"),
        pl.col("len").sum().alias("n_labeled"),
        (pl.col("len") == pl.col("len").max()).sum().alias("n_tied"))
    return top.with_columns(
        (pl.col("n_tied") > 1).alias("vote_tie"),
        (pl.col("n_top") / pl.col("n_labeled")).alias("vote_frac")).with_columns(
        pl.when(pl.col("n_tied") > 1).then(None).otherwise(pl.col("vote_label")).alias("vote_label"))

def tier1(clu, queries, lvl):
    qc = (clu.join(queries.select("Protein"), left_on="member", right_on="Protein", how="inner")
             .select(["member", "cluster"]))
    return (qc.join(majority(clu, lvl), on="cluster", how="left")
              .rename({"member": "Protein", "cluster": "cluster_id"}))

print("tier 1 vote defined")

tier 1 vote defined


### 3.1 Cluster structure of the held-out evaluation universe

How the clustering behaves before any label is transferred: how many clusters there are, how many are singletons, and what fraction of held-out query proteins share a cluster with at least one labeled training protein. That last quantity is the ceiling on tier-1 reach.

In [15]:
CLU = {p: load_clu(SEQCLUST / CLU_EVAL.format(p=p)) for p in PCTS}
rows = []
for p in PCTS:
    clu = CLU[p]
    assert clu.height == reference.height + heldout.height, (
        f"id{p} universe {clu.height:,} != {reference.height:,} training + {heldout.height:,} held-out")
    sz = clu.group_by("cluster").len()
    qc = tier1(clu, heldout, "specific")
    reach = qc.filter(pl.col("n_labeled") > 0)
    rows.append({"min_seq_id": p / 100, "universe_members": clu.height,
                 "n_clusters": sz.height,
                 "singleton_frac": float((sz["len"] == 1).mean()),
                 "mean_size": float(sz["len"].mean()), "max_size": int(sz["len"].max()),
                 "queries": qc.height,
                 "with_labeled_member": reach.height / qc.height,
                 "vote_available": float(qc["vote_label"].is_not_null().mean()),
                 "tie_frac_of_reached": float(reach["vote_tie"].mean()) if reach.height else None})
cstat = pl.DataFrame(rows)
print(cstat)

shape: (4, 10)
┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
│ min_seq_id ┆ universe_ ┆ n_cluster ┆ singleton ┆ … ┆ queries ┆ with_labe ┆ vote_avai ┆ tie_frac_ │
│ ---        ┆ members   ┆ s         ┆ _frac     ┆   ┆ ---     ┆ led_membe ┆ lable     ┆ of_reache │
│ f64        ┆ ---       ┆ ---       ┆ ---       ┆   ┆ i64     ┆ r         ┆ ---       ┆ d         │
│            ┆ i64       ┆ i64       ┆ f64       ┆   ┆         ┆ ---       ┆ f64       ┆ ---       │
│            ┆           ┆           ┆           ┆   ┆         ┆ f64       ┆           ┆ f64       │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
│ 0.3        ┆ 1099305   ┆ 144517    ┆ 0.596428  ┆ … ┆ 139856  ┆ 0.893111  ┆ 0.882501  ┆ 0.011881  │
│ 0.5        ┆ 1099305   ┆ 468639    ┆ 0.75935   ┆ … ┆ 139856  ┆ 0.584544  ┆ 0.577916  ┆ 0.011339  │
│ 0.7        ┆ 1099305   ┆ 820717    ┆ 0.880112  ┆ … ┆ 139856  ┆ 0.228986  ┆

### 3.2 Tier-1 reach and precision on the reporting slice

Tier-1 coverage and precision at every identity threshold and every granularity, on the reporting slice only. Coverage is the fraction of reporting proteins that receive a vote. Precision is the fraction of those votes that match the ground truth at that granularity. Both are pooled over the whole slice with no conditioning on identity to the training set, because identity to the training set is not knowable at application time and so cannot be used to pick an operating point.

In [16]:
for p in PCTS:
    for lvl in LEVELS:
        t = tier1(CLU[p], heldout, lvl)
        cols = [pl.col("vote_label").alias(f"c{p}_{lvl}")]
        if lvl == "specific":
            cols += [pl.col("cluster_id").alias(f"c{p}_cluster"),
                     pl.col("vote_frac").alias(f"c{p}_frac"),
                     pl.col("n_labeled").alias(f"c{p}_nlab")]
        heldout = heldout.join(t.select(["Protein"] + cols), on="Protein", how="left")
rows = []
for p in PCTS:
    for lvl in LEVELS:
        g = heldout.filter(pl.col("split") == "report")
        sel = g.filter(pl.col(f"c{p}_{lvl}").is_not_null())
        y = ((sel[f"c{p}_{lvl}"] == sel[TRUE[lvl]]).fill_null(False)
             .cast(pl.Float64).to_numpy()) if sel.height else np.array([])
        st = prec_stat(y)
        rows.append({"min_seq_id": p / 100, "level": lvl, "n_report": g.height,
                     "coverage": sel.height / g.height, "n_covered": sel.height,
                     "precision": st["precision"], "lo": st["lo"], "hi": st["hi"],
                     "recall": (st["precision"] * sel.height / g.height) if st["precision"] is not None else None})
sq = pl.DataFrame(rows)
print(sq)

shape: (12, 9)
┌────────────┬──────────┬──────────┬──────────┬───┬───────────┬──────────┬──────────┬──────────┐
│ min_seq_id ┆ level    ┆ n_report ┆ coverage ┆ … ┆ precision ┆ lo       ┆ hi       ┆ recall   │
│ ---        ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---      ┆ ---      ┆ ---      │
│ f64        ┆ str      ┆ i64      ┆ f64      ┆   ┆ f64       ┆ f64      ┆ f64      ┆ f64      │
╞════════════╪══════════╪══════════╪══════════╪═══╪═══════════╪══════════╪══════════╪══════════╡
│ 0.3        ┆ specific ┆ 41957    ┆ 0.882928 ┆ … ┆ 0.861925  ┆ 0.858374 ┆ 0.8654   ┆ 0.761017 │
│ 0.3        ┆ L1       ┆ 41957    ┆ 0.890292 ┆ … ┆ 0.945575  ┆ 0.943228 ┆ 0.94783  ┆ 0.841838 │
│ 0.3        ┆ category ┆ 41957    ┆ 0.89289  ┆ … ┆ 0.994234  ┆ 0.993415 ┆ 0.994952 ┆ 0.887742 │
│ 0.5        ┆ specific ┆ 41957    ┆ 0.579641 ┆ … ┆ 0.950164  ┆ 0.947358 ┆ 0.952829 ┆ 0.550754 │
│ 0.5        ┆ L1       ┆ 41957    ┆ 0.583478 ┆ … ┆ 0.981863  ┆ 0.980115 ┆ 0.983461 ┆ 0.572896 │
│ 0.5        ┆ 

## 4. Tier 2: calibrated embedding label transfer

Tier 2 is the CheckAMG-PST embedding rule. A query protein takes the functional label of its nearest labeled neighbor in the embedding, and that transfer is gated by a model that predicts, from the geometry of the neighborhood alone, whether the transferred label is correct. The probability threshold is chosen in section 5, on the calibration proteins tier 1 does not reach.

### 4.1 The raw embedding rule

Apply the deployed 1-NN rule to the held-out proteins using the full training index.

In [17]:
tref = pl.read_parquet(NOVEL / "train_ref_index_order.parquet")
train_class = tref["Protein Classification"].fill_null("unclassified").to_numpy().astype(str)
tfn = tref["Function"].fill_null("").to_numpy().astype(str)
meta = pl.read_parquet(NOVEL / "test_avglike_meta.parquet").with_row_index("row")
Ifull = np.load(NOVEL / "test_avglike_I.npy")
Dfull = np.load(NOVEL / "test_avglike_D.npy").astype(np.float64)
first_row = meta.unique("Protein", keep="first")
sel = first_row["row"].to_numpy()
nbrc = train_class[Ifull[sel]]
ok = np.isin(nbrc, CAT3)
has = ok.any(1)
rr = np.arange(sel.size)
fi = np.where(has, ok.argmax(1), 0)
dep = pl.DataFrame({
    "Protein": first_row["Protein"],
    "dep_has": has,
    "dep_category": np.where(has, nbrc[rr, fi], "").astype(str),
    "dep_specific": np.where(has, tfn[Ifull[sel][rr, fi]], "").astype(str),
    "dep_d1": np.where(has, Dfull[sel][rr, fi], np.inf)})
dep = dep.with_columns(
    pl.when(pl.col("dep_has")).then(pl.col("dep_category")).otherwise(None).alias("dep_category"))
print(f"held-out proteins: {dep.height:,}")
print(f"with an AVG-like neighbor in the top 50 (the deployed rule's reach): {int(has.sum()):,} ({100*has.mean():.1f}%)")

held-out proteins: 139,856
with an AVG-like neighbor in the top 50 (the deployed rule's reach): 133,744 (95.6%)


Build the held-out frame for the deployed rule and score it against the ground truth at each granularity.

In [18]:
ev = (ev_keys.join(dep, on="Protein", how="left")
             .join(ev_id, on="Protein", how="left")
             .with_columns(pl.col("max_ident_to_train").fill_null(0.0)))
ev = ev.with_columns(
    pl.col("true_specific").replace_strict(EX2L1, default=None).alias("true_L1"),
    pl.col("dep_specific").replace_strict(EX2L1, default=None).alias("dep_L1"),
    pl.col("true_specific").replace_strict(dict(zip(freq["ref_specific"], freq["freq_tercile"])),
                                        default="unseen").alias("freq_tercile"))
ev = ev.with_columns(
    pl.Series("specific_ok", normfn(ev["true_specific"].fill_null("").to_numpy().astype(str))
                          == normfn(ev["dep_specific"].fill_null("").to_numpy().astype(str))),
    (pl.col("true_L1") == pl.col("dep_L1")).alias("L1_ok"),
    (pl.col("true_category") == pl.col("dep_category")).alias("category_ok"))
print(ev.select(["Protein", "true_category", "dep_category", "true_L1", "dep_L1",
                 "specific_ok", "L1_ok", "category_ok", "max_ident_to_train"]).head(5))

shape: (5, 9)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────┬───────────┬───────────┐
│ Protein    ┆ true_categ ┆ dep_catego ┆ true_L1   ┆ … ┆ specific_ ┆ L1_ok ┆ category_ ┆ max_ident │
│ ---        ┆ ory        ┆ ry         ┆ ---       ┆   ┆ ok        ┆ ---   ┆ ok        ┆ _to_train │
│ str        ┆ ---        ┆ ---        ┆ str       ┆   ┆ ---       ┆ bool  ┆ ---       ┆ ---       │
│            ┆ str        ┆ str        ┆           ┆   ┆ bool      ┆       ┆ bool      ┆ f32       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════╪═══════════╪═══════════╡
│ 1001585.SA ┆ metabolic  ┆ metabolic  ┆ Cofactor  ┆ … ┆ false     ┆ false ┆ true      ┆ 0.748     │
│ MN02603190 ┆            ┆            ┆ & vitamin ┆   ┆           ┆       ┆           ┆           │
│ .CP002620~ ┆            ┆            ┆ metabolis ┆   ┆           ┆       ┆           ┆           │
│ 3298858-33 ┆            ┆            ┆ m         ┆   ┆           ┆       ┆ 

Micro-averaged and macro-averaged precision at each granularity, with the majority-class baseline.

In [19]:
g = ev.filter(pl.col("dep_has"))
majority_baseline = g.group_by("true_category").len()["len"].max() / g.height
rows = []
for lvl, tcol, pcol, okcol in [("category", "true_category", "dep_category", "category_ok"),
                               ("L1", "true_L1", "dep_L1", "L1_ok"),
                               ("specific", "true_specific", "dep_specific", "specific_ok")]:
    s = g.filter(pl.col(pcol).is_not_null() & (pl.col(pcol) != ""))
    micro = float(s[okcol].mean())
    per = s.group_by(pcol).agg(pl.col(okcol).mean().alias("p"), pl.len().alias("n"))
    macro = float(per["p"].mean())
    rows.append({"level": lvl, "n": s.height, "micro": round(micro, 4),
                 "macro": round(macro, 4), "n_labels": per.height})
ladder = pl.DataFrame(rows)
print(f"evaluated on {g.height:,} held-out proteins (majority-class baseline {majority_baseline:.4f})")
print(ladder)

evaluated on 133,744 held-out proteins (majority-class baseline 0.6688)
shape: (3, 5)
┌──────────┬────────┬────────┬────────┬──────────┐
│ level    ┆ n      ┆ micro  ┆ macro  ┆ n_labels │
│ ---      ┆ ---    ┆ ---    ┆ ---    ┆ ---      │
│ str      ┆ i64    ┆ f64    ┆ f64    ┆ i64      │
╞══════════╪════════╪════════╪════════╪══════════╡
│ category ┆ 133744 ┆ 0.8952 ┆ 0.8491 ┆ 3        │
│ L1       ┆ 133744 ┆ 0.5866 ┆ 0.5595 ┆ 28       │
│ specific ┆ 133744 ┆ 0.3813 ┆ 0.2255 ┆ 6181     │
└──────────┴────────┴────────┴────────┴──────────┘


Precision by label-frequency tercile, which governs whether rare-label claims are supportable.

In [20]:
rows = []
for lvl, pcol, okcol in [("category", "dep_category", "category_ok"),
                         ("L1", "dep_L1", "L1_ok"), ("specific", "dep_specific", "specific_ok")]:
    s = g.filter(pl.col(pcol).is_not_null() & (pl.col(pcol) != ""))
    for r in s.group_by("freq_tercile").agg(pl.col(okcol).mean().alias("precision"),
                                            pl.len().alias("n")).iter_rows(named=True):
        rows.append({"level": lvl, **r})
terciles = pl.DataFrame(rows).sort(["level", "freq_tercile"])
print(terciles)

shape: (12, 4)
┌──────────┬──────────────┬───────────┬────────┐
│ level    ┆ freq_tercile ┆ precision ┆ n      │
│ ---      ┆ ---          ┆ ---       ┆ ---    │
│ str      ┆ str          ┆ f64       ┆ i64    │
╞══════════╪══════════════╪═══════════╪════════╡
│ L1       ┆ common       ┆ 0.600404  ┆ 124776 │
│ L1       ┆ mid          ┆ 0.404361  ┆ 7659   │
│ L1       ┆ rare         ┆ 0.297535  ┆ 1136   │
│ L1       ┆ unseen       ┆ null      ┆ 173    │
│ category ┆ common       ┆ 0.897424  ┆ 124776 │
│ category ┆ mid          ┆ 0.867215  ┆ 7659   │
│ category ┆ rare         ┆ 0.857394  ┆ 1136   │
│ category ┆ unseen       ┆ 0.815029  ┆ 173    │
│ specific ┆ common       ┆ 0.398987  ┆ 124776 │
│ specific ┆ mid          ┆ 0.150281  ┆ 7659   │
│ specific ┆ rare         ┆ 0.053697  ┆ 1136   │
│ specific ┆ unseen       ┆ 0.011561  ┆ 173    │
└──────────┴──────────────┴───────────┴────────┘


Precision by maximum sequence identity to the training set, the stratification the failed split gate requires.

In [21]:
bands = [(0.0, 0.3), (0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
rows = []
for lo, hi in bands:
    s = g.filter((pl.col("max_ident_to_train") >= lo) & (pl.col("max_ident_to_train") < hi))
    if not s.height:
        continue
    rows.append({"band": f"{lo:.0%}-{hi:.0%}", "lo": lo, "n": s.height,
                 "category": round(float(s["category_ok"].mean()), 4),
                 "L1": round(float(s.filter(pl.col("dep_L1").is_not_null())["L1_ok"].mean()), 4),
                 "specific": round(float(s["specific_ok"].mean()), 4)})
strat = pl.DataFrame(rows)
print(strat)
lo50 = g.filter(pl.col("max_ident_to_train") < 0.5)
print(f"\nbelow 50% identity (n={lo50.height:,}, {100*lo50.height/g.height:.1f}%): "
      f"category {lo50['category_ok'].mean():.4f}, specific {lo50['specific_ok'].mean():.4f}")

shape: (8, 6)
┌──────────┬─────┬───────┬──────────┬────────┬──────────┐
│ band     ┆ lo  ┆ n     ┆ category ┆ L1     ┆ specific │
│ ---      ┆ --- ┆ ---   ┆ ---      ┆ ---    ┆ ---      │
│ str      ┆ f64 ┆ i64   ┆ f64      ┆ f64    ┆ f64      │
╞══════════╪═════╪═══════╪══════════╪════════╪══════════╡
│ 0%-30%   ┆ 0.0 ┆ 1170  ┆ 0.8043   ┆ 0.5377 ┆ 0.3419   │
│ 30%-40%  ┆ 0.3 ┆ 4158  ┆ 0.8492   ┆ 0.5529 ┆ 0.342    │
│ 40%-50%  ┆ 0.4 ┆ 15862 ┆ 0.8607   ┆ 0.5512 ┆ 0.3193   │
│ 50%-60%  ┆ 0.5 ┆ 30168 ┆ 0.8741   ┆ 0.5501 ┆ 0.3221   │
│ 60%-70%  ┆ 0.6 ┆ 32464 ┆ 0.8999   ┆ 0.5521 ┆ 0.3443   │
│ 70%-80%  ┆ 0.7 ┆ 23540 ┆ 0.9201   ┆ 0.585  ┆ 0.3894   │
│ 80%-90%  ┆ 0.8 ┆ 11901 ┆ 0.925    ┆ 0.6469 ┆ 0.4659   │
│ 90%-101% ┆ 0.9 ┆ 14481 ┆ 0.9226   ┆ 0.7452 ┆ 0.5873   │
└──────────┴─────┴───────┴──────────┴────────┴──────────┘

below 50% identity (n=21,190, 15.8%): category 0.8553, specific 0.3250


Write the ladder tables.

In [22]:
ladder.write_parquet(OUT / "fig_ladder_micro_macro.parquet")
terciles.write_parquet(OUT / "fig_ladder_by_frequency.parquet")
strat.write_parquet(OUT / "fig_ladder_by_identity.parquet")
print("wrote 3 ladder tables to", OUT)

wrote 3 ladder tables to /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/tables/propagation


### 4.2 Confidence scores

The deployed rule emits a label and a distance. This section asks whether any cheap geometric score predicts correctness better than `d1` alone, which the correctness model in section 4.3 uses.

All features come from a FAISS index built over the labeled training proteins only, so `d1` is the true distance to the nearest labeled protein rather than the nearest labeled protein that happened to fall in the top 50 of the full index. Identity to the training set is deliberately **not** a model feature: it is unavailable for most applied proteins, and using it would make the model unscoreable where it matters. It is used only to stratify.

AUROC of each candidate score against correctness, at the specific and L1 levels.

In [23]:
from sklearn.metrics import roc_auc_score
SCORES = ["d1", "d2", "margin", "ratio", "vote_purity_k5", "vote_purity_k20",
          "vote_purity_k50", "density_norm_d1", "hub_count"]
Xs = heldout.select(SCORES).to_numpy().astype(float)
keep = np.isfinite(Xs).all(1)
sub = heldout.filter(pl.Series(keep))
Xs = Xs[keep]
rows = []
for j, s in enumerate(SCORES):
    x = Xs[:, j]
    r = {"score": s}
    for lvl in ["specific", "L1"]:
        y = sub["y_" + lvl].cast(pl.Int8).to_numpy()
        a = roc_auc_score(y, x)
        r[f"auroc_{lvl}"] = round(max(a, 1 - a), 4)
        r[f"direction_{lvl}"] = "higher_better" if a >= 0.5 else "lower_better"
    rows.append(r)
auc = pl.DataFrame(rows).sort("auroc_specific", descending=True)
print(f"scored on {sub.height:,} held-out proteins")
print(auc.select(["score", "auroc_specific", "auroc_L1", "direction_specific"]))
base = float(auc.filter(pl.col("score") == "d1")["auroc_specific"][0])
best = float(auc["auroc_specific"].max())
print(f"\nbest single score {best:.4f} vs d1 baseline {base:.4f}  (gain {best-base:+.4f})")
SCORE_GATE = bool(best - base >= 0.03)
print(f"acceptance criterion (best beats d1 by >= 0.03): {'PASS' if SCORE_GATE else 'FAIL'}")

scored on 137,078 held-out proteins
shape: (9, 4)
┌─────────────────┬────────────────┬──────────┬────────────────────┐
│ score           ┆ auroc_specific ┆ auroc_L1 ┆ direction_specific │
│ ---             ┆ ---            ┆ ---      ┆ ---                │
│ str             ┆ f64            ┆ f64      ┆ str                │
╞═════════════════╪════════════════╪══════════╪════════════════════╡
│ vote_purity_k50 ┆ 0.8726         ┆ 0.7723   ┆ higher_better      │
│ vote_purity_k20 ┆ 0.871          ┆ 0.7692   ┆ higher_better      │
│ vote_purity_k5  ┆ 0.8555         ┆ 0.7544   ┆ higher_better      │
│ ratio           ┆ 0.7868         ┆ 0.7144   ┆ lower_better       │
│ margin          ┆ 0.6962         ┆ 0.6318   ┆ higher_better      │
│ density_norm_d1 ┆ 0.6086         ┆ 0.6046   ┆ lower_better       │
│ d1              ┆ 0.5813         ┆ 0.5847   ┆ lower_better       │
│ hub_count       ┆ 0.5387         ┆ 0.5318   ┆ higher_better      │
│ d2              ┆ 0.5381         ┆ 0.5543   ┆ lower

Hubness: how concentrated are the nearest-neighbor assignments, and are hub references worse?

In [24]:
hub = heldout.group_by("nn_ref_id").agg(pl.len().alias("n_queries"),
                                    pl.col("y_specific").mean().alias("err_free_rate"))
hub = hub.with_columns((1 - pl.col("err_free_rate")).alias("error_rate"))
print(f"distinct reference proteins used as 1-NN: {hub.height:,}")
print(f"  queries per reference: median {hub['n_queries'].median():.0f}, "
      f"p95 {hub['n_queries'].quantile(0.95):.0f}, max {hub['n_queries'].max():,}")
top = hub.sort("n_queries", descending=True).head(int(0.01 * hub.height))
print(f"  top 1% of references absorb {100*top['n_queries'].sum()/hub['n_queries'].sum():.1f}% of all queries")
big = hub.filter(pl.col("n_queries") >= 5).sort("nn_ref_id")
sp = np.corrcoef(np.argsort(np.argsort(big["n_queries"].to_numpy())),
                 np.argsort(np.argsort(big["error_rate"].to_numpy())))[0, 1]
print(f"  Spearman(hub count, specific error rate) over references with >=5 queries: {sp:+.3f}")

distinct reference proteins used as 1-NN: 111,822
  queries per reference: median 1, p95 2, max 257
  top 1% of references absorb 4.8% of all queries
  Spearman(hub count, specific error rate) over references with >=5 queries: -0.138


### 4.3 The calibrated correctness model

Fit a shallow gradient-boosted model per granularity predicting whether the raw 1-NN label is correct, then calibrate it with isotonic regression. The model is fit on the fit slice and calibrated on the calibration slice (section 2.4).

Fit and calibrate one model per granularity.

In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
FEATS = SCORES
Xall = heldout.select(FEATS).to_numpy().astype(np.float64)
Xall = np.nan_to_num(Xall, nan=0.0, posinf=0.0, neginf=0.0)
sp = heldout["split"].to_numpy()
models, isotonic = {}, {}
for lvl in LEVELS:
    y = heldout["y_" + lvl].cast(pl.Int8).to_numpy()
    m = HistGradientBoostingClassifier(max_depth=3, max_iter=250, learning_rate=0.06,
                                       l2_regularization=1.0, random_state=SEED)
    m.fit(Xall[sp == "fit"], y[sp == "fit"])
    pc = m.predict_proba(Xall[sp == "calib"])[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    iso.fit(pc, y[sp == "calib"])
    models[lvl], isotonic[lvl] = m, iso
    print(f"{lvl:9s} fitted on {(sp=='fit').sum():,}, calibrated on {(sp=='calib').sum():,}")

specific  fitted on 55,942, calibrated on 41,957


L1        fitted on 55,942, calibrated on 41,957


category  fitted on 55,942, calibrated on 41,957


Calibrated probabilities for every held-out protein.

In [26]:
for lvl in LEVELS:
    p = isotonic[lvl].predict(models[lvl].predict_proba(Xall)[:, 1])
    heldout = heldout.with_columns(pl.Series("p_" + lvl, p))
print(heldout.filter(pl.col("split") == "report").select(
    ["p_specific", "p_L1", "p_category"]).describe())

shape: (9, 4)
┌────────────┬────────────┬──────────┬────────────┐
│ statistic  ┆ p_specific ┆ p_L1     ┆ p_category │
│ ---        ┆ ---        ┆ ---      ┆ ---        │
│ str        ┆ f64        ┆ f64      ┆ f64        │
╞════════════╪════════════╪══════════╪════════════╡
│ count      ┆ 41957.0    ┆ 41957.0  ┆ 41957.0    │
│ null_count ┆ 0.0        ┆ 0.0      ┆ 0.0        │
│ mean       ┆ 0.370972   ┆ 0.573367 ┆ 0.882787   │
│ std        ┆ 0.336479   ┆ 0.259346 ┆ 0.081226   │
│ min        ┆ 0.0        ┆ 0.170213 ┆ 0.0        │
│ 25%        ┆ 0.070197   ┆ 0.350725 ┆ 0.827448   │
│ 50%        ┆ 0.220703   ┆ 0.506936 ┆ 0.887136   │
│ 75%        ┆ 0.631447   ┆ 0.816078 ┆ 0.942038   │
│ max        ┆ 1.0        ┆ 1.0      ┆ 1.0        │
└────────────┴────────────┴──────────┴────────────┘


Expected calibration error and the reliability curve on the reporting slice.

In [27]:
report = heldout.filter(pl.col("split") == "report")
bins = np.linspace(0, 1, 21)
rel_rows, ece_rows = [], []
for lvl in LEVELS:
    p = report["p_" + lvl].to_numpy()
    y = report["y_" + lvl].cast(pl.Float64).to_numpy()
    idx = np.clip(np.digitize(p, bins) - 1, 0, len(bins) - 2)
    ece = 0.0
    for b in range(len(bins) - 1):
        m = idx == b
        if not m.any():
            continue
        conf, acc, w = p[m].mean(), y[m].mean(), m.mean()
        ece += w * abs(acc - conf)
        rel_rows.append({"level": lvl, "bin_mid": float((bins[b] + bins[b+1]) / 2),
                         "confidence": float(conf), "accuracy": float(acc), "n": int(m.sum())})
    ece_rows.append({"level": lvl, "ece": round(float(ece), 4)})
rel = pl.DataFrame(rel_rows)
ece = pl.DataFrame(ece_rows)
print(ece)
ECE_GATE = bool(ece["ece"].max() < 0.05)
print(f"acceptance criterion (ECE < 0.05 at every level): {'PASS' if ECE_GATE else 'FAIL'}")
rel.write_parquet(OUT / "fig_reliability.parquet")

shape: (3, 2)
┌──────────┬────────┐
│ level    ┆ ece    │
│ ---      ┆ ---    │
│ str      ┆ f64    │
╞══════════╪════════╡
│ specific ┆ 0.0077 │
│ L1       ┆ 0.0055 │
│ category ┆ 0.0057 │
└──────────┴────────┘
acceptance criterion (ECE < 0.05 at every level): PASS


### 4.4 Risk-coverage

Precision as a function of coverage when proteins are ranked by calibrated probability, against the same curve ranked by `d1` alone.

Risk-coverage curves for both rankings, on the reporting slice.

In [28]:
def curve(score, y, higher_better=True):
    o = np.argsort(-score if higher_better else score)
    ys = y[o]
    cov = np.arange(1, ys.size + 1) / ys.size
    prec = np.cumsum(ys) / np.arange(1, ys.size + 1)
    return cov, prec

rc_rows = []
for lvl in LEVELS:
    y = report["y_" + lvl].cast(pl.Float64).to_numpy()
    for name, sc, hb in [("calibrated", report["p_" + lvl].to_numpy(), True),
                         ("d1_only", report["d1"].to_numpy(), False)]:
        cov, prec = curve(sc, y, hb)
        step = max(1, cov.size // 400)
        for i in range(0, cov.size, step):
            rc_rows.append({"level": lvl, "ranking": name,
                            "coverage": float(cov[i]), "precision": float(prec[i])})
rc = pl.DataFrame(rc_rows)
rc.write_parquet(OUT / "fig_risk_coverage.parquet")
print(f"risk-coverage rows: {rc.height:,}")

risk-coverage rows: 2,424


Area under the risk-coverage curve and coverage attainable at fixed precision.

In [29]:
rows = []
for lvl in LEVELS:
    y = report["y_" + lvl].cast(pl.Float64).to_numpy()
    for name, sc, hb in [("calibrated", report["p_" + lvl].to_numpy(), True),
                         ("d1_only", report["d1"].to_numpy(), False)]:
        cov, prec = curve(sc, y, hb)
        r = {"level": lvl, "ranking": name, "aurc": round(float(TRAPZ(prec, cov)), 4)}
        for tgt in TARGETS:
            okc = cov[prec >= tgt]
            r[f"cov@{tgt:.2f}"] = round(float(okc.max()), 4) if okc.size else 0.0
        rows.append(r)
risk_coverage_summary = pl.DataFrame(rows).sort(["level", "ranking"])
print(risk_coverage_summary)
risk_coverage_summary.write_parquet(OUT / "fig_risk_coverage_summary.parquet")

shape: (6, 6)
┌──────────┬────────────┬────────┬──────────┬──────────┬──────────┐
│ level    ┆ ranking    ┆ aurc   ┆ cov@0.90 ┆ cov@0.80 ┆ cov@0.70 │
│ ---      ┆ ---        ┆ ---    ┆ ---      ┆ ---      ┆ ---      │
│ str      ┆ str        ┆ f64    ┆ f64      ┆ f64      ┆ f64      │
╞══════════╪════════════╪════════╪══════════╪══════════╪══════════╡
│ L1       ┆ calibrated ┆ 0.8065 ┆ 0.3364   ┆ 0.5079   ┆ 0.6993   │
│ L1       ┆ d1_only    ┆ 0.6595 ┆ 0.0276   ┆ 0.0613   ┆ 0.1692   │
│ category ┆ calibrated ┆ 0.9501 ┆ 0.9182   ┆ 1.0      ┆ 1.0      │
│ category ┆ d1_only    ┆ 0.9179 ┆ 0.8162   ┆ 1.0      ┆ 1.0      │
│ specific ┆ calibrated ┆ 0.6801 ┆ 0.2339   ┆ 0.3424   ┆ 0.4462   │
│ specific ┆ d1_only    ┆ 0.461  ┆ 0.0001   ┆ 0.0232   ┆ 0.0414   │
└──────────┴────────────┴────────┴──────────┴──────────┴──────────┘


## 5. One operating point per granularity, for both tiers jointly

Tier 1's free parameter is the MMseqs2 minimum sequence identity. Tier 2's is the calibrated probability above which a label is emitted. Both are chosen on the calibration slice to meet the same precision target at the same granularity, in the order the rule applies them. Tier 1's identity threshold is the one that labels the most calibration proteins correctly while calibration precision stays at or above the target. Tier 2's threshold is then fit only on the calibration proteins tier 1 does not reach, because those are the only proteins tier 2 labels. Neither parameter depends on identity to the training set, which is unknown at application time.

The calibration and reporting slices, and tier 1's identity threshold per granularity per target.

In [30]:
calibration = heldout.filter(pl.col("split") == "calib")
CALT1 = {}
for lvl in LEVELS:
    for p in PCTS:
        s = calibration.filter(pl.col(f"c{p}_{lvl}").is_not_null())
        y = (s[f"c{p}_{lvl}"] == s[TRUE[lvl]]).fill_null(False) if s.height else None
        CALT1[(lvl, p)] = (s.height, float(y.mean()) if s.height else None,
                           float(y.sum()) if s.height else 0.0)

def pick_p(lvl, target):
    best = None
    for p in PCTS:
        n, pr, nc = CALT1[(lvl, p)]
        if n and pr >= target and (best is None or nc > best[1]):
            best = (p, nc)
    return best[0] if best else None

ALL_T = sorted(set(TARGETS) | set(TARGETS_R))
PSTAR = {t: {lvl: pick_p(lvl, t) for lvl in LEVELS} for t in ALL_T}
rows = []
for t in TARGETS:
    for lvl in LEVELS:
        for p in PCTS:
            n, pr, nc = CALT1[(lvl, p)]
            rows.append({"target": t, "level": lvl, "min_seq_id": p / 100,
                         "calib_coverage": n / calibration.height, "calib_precision": pr,
                         "calib_n_correct": int(nc), "meets_target": bool(n and pr >= t)})
t1sel = pl.DataFrame(rows)
print(t1sel.filter(pl.col("target") == PRIMARY_TARGET))
print()
for t in TARGETS:
    print(f"target {t:.2f}  tier-1 min_seq_id per level: "
          + ", ".join(f"{l}={'none' if PSTAR[t][l] is None else str(PSTAR[t][l]) + '%'}" for l in LEVELS))

shape: (12, 7)
┌────────┬──────────┬────────────┬────────────────┬────────────────┬────────────────┬──────────────┐
│ target ┆ level    ┆ min_seq_id ┆ calib_coverage ┆ calib_precisio ┆ calib_n_correc ┆ meets_target │
│ ---    ┆ ---      ┆ ---        ┆ ---            ┆ n              ┆ t              ┆ ---          │
│ f64    ┆ str      ┆ f64        ┆ f64            ┆ ---            ┆ ---            ┆ bool         │
│        ┆          ┆            ┆                ┆ f64            ┆ i64            ┆              │
╞════════╪══════════╪════════════╪════════════════╪════════════════╪════════════════╪══════════════╡
│ 0.9    ┆ specific ┆ 0.3        ┆ 0.88114        ┆ 0.863808       ┆ 31935          ┆ false        │
│ 0.9    ┆ specific ┆ 0.5        ┆ 0.577663       ┆ 0.950283       ┆ 23032          ┆ true         │
│ 0.9    ┆ specific ┆ 0.7        ┆ 0.226947       ┆ 0.979311       ┆ 9325           ┆ true         │
│ 0.9    ┆ specific ┆ 0.9        ┆ 0.091927       ┆ 0.993518       ┆ 3832   

Tier 2's probability threshold, inverted on the calibration proteins tier 1 does not reach. The threshold at a target is the calibrated probability at the boundary of the largest high-probability set whose observed precision still meets the target. Thresholding at the nominal target probability itself would give away coverage, because the calibration is conservative.

In [31]:
EMB_DISP = {"specific": "raw_specific", "L1": "raw_L1", "category": "raw_category"}

def t1_col(lvl, target):
    p = PSTAR[target][lvl]
    return None if p is None else f"c{p}_{lvl}"

def tier2_pool(df, lvl, target):
    c = t1_col(lvl, target)
    return df if c is None else df.filter(pl.col(c).is_null())

def _invert(sub, lvl, target):
    p = sub["p_" + lvl].to_numpy().astype(float)
    y = sub["y_" + lvl].cast(pl.Float64).to_numpy()
    keep = ~np.isnan(p)
    p, y = p[keep], y[keep]
    if not p.size:
        return 1.01
    o = np.argsort(-p)
    ys, ps = y[o], p[o]
    pr = np.cumsum(ys) / np.arange(1, ys.size + 1)
    ok = np.where(pr >= target)[0]
    return float(ps[ok[-1]]) if ok.size else 1.01

_thr_cache, _thrfull_cache = {}, {}

def threshold_for(lvl, target):
    key = (lvl, round(float(target), 4))
    if key not in _thr_cache:
        _thr_cache[key] = _invert(tier2_pool(calibration, lvl, target), lvl, target)
    return _thr_cache[key]

def threshold_full(lvl, target):
    key = (lvl, round(float(target), 4))
    if key not in _thrfull_cache:
        _thrfull_cache[key] = _invert(calibration, lvl, target)
    return _thrfull_cache[key]

THR = {t: {lvl: threshold_for(lvl, t) for lvl in LEVELS} for t in TARGETS}
THR_FULL = {t: {lvl: threshold_full(lvl, t) for lvl in LEVELS} for t in TARGETS}
for t in TARGETS:
    for lvl in LEVELS:
        n = tier2_pool(calibration, lvl, t).height
        print(f"target {t:.2f} {lvl:9s} tier-2 pool {n:>7,} of {calibration.height:,} calib "
              f"({100*n/calibration.height:5.1f}%)  threshold {THR[t][lvl]:.4f}  "
              f"(unmasked would be {THR_FULL[t][lvl]:.4f})")

target 0.90 specific  tier-2 pool  17,720 of 41,957 calib ( 42.2%)  threshold 0.7775  (unmasked would be 0.6895)
target 0.90 L1        tier-2 pool   4,687 of 41,957 calib ( 11.2%)  threshold 0.8513  (unmasked would be 0.7273)


target 0.90 category  tier-2 pool   4,558 of 41,957 calib ( 10.9%)  threshold 0.9129  (unmasked would be 0.7981)
target 0.80 specific  tier-2 pool   4,987 of 41,957 calib ( 11.9%)  threshold 0.7092  (unmasked would be 0.4572)
target 0.80 L1        tier-2 pool   4,687 of 41,957 calib ( 11.2%)  threshold 0.7051  (unmasked would be 0.5069)
target 0.80 category  tier-2 pool   4,558 of 41,957 calib ( 10.9%)  threshold 0.8093  (unmasked would be 0.0000)
target 0.70 specific  tier-2 pool   4,987 of 41,957 calib ( 11.9%)  threshold 0.5699  (unmasked would be 0.2983)
target 0.70 L1        tier-2 pool   4,687 of 41,957 calib ( 11.2%)  threshold 0.5138  (unmasked would be 0.3741)
target 0.70 category  tier-2 pool   4,558 of 41,957 calib ( 10.9%)  threshold 0.3889  (unmasked would be 0.0000)


The joint operating point table: what each tier contributes on the reporting slice at each target and granularity.

In [32]:
rows = []
for t in TARGETS:
    for lvl in LEVELS:
        c, thr = t1_col(lvl, t), THR[t][lvl]
        s1 = report.filter(pl.col(c).is_not_null()) if c else report.head(0)
        y1 = ((s1[c] == s1[TRUE[lvl]]).fill_null(False).cast(pl.Float64).to_numpy()
              if s1.height else np.array([]))
        pool = tier2_pool(report, lvl, t)
        s2 = pool.filter(pl.col("p_" + lvl).is_not_null() & (pl.col("p_" + lvl) >= thr))
        y2 = s2["y_" + lvl].cast(pl.Float64).to_numpy() if s2.height else np.array([])
        for nm, st, n_sel in [("tier1_cluster", prec_stat(y1), s1.height), ("tier2_embedding", prec_stat(y2), s2.height)]:
            rows.append({"target": t, "level": lvl, "tier": nm,
                         "min_seq_id": (PSTAR[t][lvl] / 100 if PSTAR[t][lvl] else None),
                         "p_threshold": thr if nm == "tier2_embedding" else None,
                         "n_report": report.height, "n_assigned": n_sel,
                         "coverage": n_sel / report.height, "precision": st["precision"],
                         "lo": st["lo"], "hi": st["hi"]})
opp = pl.DataFrame(rows)
print(opp.filter(pl.col("target") == PRIMARY_TARGET))
opp.write_parquet(OUT / "fig_operating_points.parquet")

shape: (6, 11)
┌────────┬──────────┬────────────────┬────────────┬───┬──────────┬───────────┬──────────┬──────────┐
│ target ┆ level    ┆ tier           ┆ min_seq_id ┆ … ┆ coverage ┆ precision ┆ lo       ┆ hi       │
│ ---    ┆ ---      ┆ ---            ┆ ---        ┆   ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ f64    ┆ str      ┆ str            ┆ f64        ┆   ┆ f64      ┆ f64       ┆ f64      ┆ f64      │
╞════════╪══════════╪════════════════╪════════════╪═══╪══════════╪═══════════╪══════════╪══════════╡
│ 0.9    ┆ specific ┆ tier1_cluster  ┆ 0.5        ┆ … ┆ 0.579641 ┆ 0.950164  ┆ 0.947358 ┆ 0.952829 │
│ 0.9    ┆ specific ┆ tier2_embeddin ┆ 0.5        ┆ … ┆ 0.055366 ┆ 0.894533  ┆ 0.881384 ┆ 0.906379 │
│        ┆          ┆ g              ┆            ┆   ┆          ┆           ┆          ┆          │
│ 0.9    ┆ L1       ┆ tier1_cluster  ┆ 0.3        ┆ … ┆ 0.890292 ┆ 0.945575  ┆ 0.943228 ┆ 0.94783  │
│ 0.9    ┆ L1       ┆ tier2_embeddin ┆ 0.3        ┆ … ┆ 0.011035 ┆ 0.915767 

## 6. The tiered assignment rule

The rule, stated once and applied unchanged to the held-out set here and to the applied set in section 12. For a protein, and for a precision target:

1. At the **specific** granularity, take tier 1's cluster majority label if its cluster has a non-tied majority among labeled training members. Otherwise take tier 2's nearest-neighbor specific label if its calibrated probability clears the specific threshold. If neither applies, descend.
2. At **L1**, apply the same two tests in the same order with the L1 threshold and the L1 clustering.
3. At **category**, apply them again.
4. If no test passes at any granularity, the protein is `unassigned`.

The two tiers are tried in that order within a granularity, and granularities are tried deepest first, so a protein always receives the most specific label that some tier can support at the requested precision. Every assignment carries the tier that produced it in `label_source`, which takes the values `cluster`, `embedding`, or `none`.

Build the rule, then verify that every tier-1 vote maps back to a display spelling before any label is emitted.

In [33]:
_unmapped = [v for v in reference["ref_specific_n"].unique().to_list() if v and v not in NORM2DISP]
assert not _unmapped, f"{len(_unmapped)} normalized labels without a display spelling"

def tiered_terms(target):
    terms = []
    for lvl in LEVELS:
        c = t1_col(lvl, target)
        if c is not None:
            lab = (pl.col(c).replace_strict(NORM2DISP, default=None) if lvl == "specific"
                   else pl.col(c))
            terms.append((pl.col(c).is_not_null(), lab, lvl, "cluster"))
        cond = pl.col("p_" + lvl).is_not_null() & (pl.col("p_" + lvl) >= threshold_for(lvl, target))
        if c is not None:
            cond = cond & pl.col(c).is_null()
        terms.append((cond, pl.col(EMB_DISP[lvl]), lvl, "embedding"))
    return terms

def _fold(terms):
    c0, l0, v0, s0 = terms[0]
    e_lab, e_lvl, e_src = pl.when(c0).then(l0), pl.when(c0).then(pl.lit(v0)), pl.when(c0).then(pl.lit(s0))
    for cond, lab, lvl, src in terms[1:]:
        e_lab = e_lab.when(cond).then(lab)
        e_lvl = e_lvl.when(cond).then(pl.lit(lvl))
        e_src = e_src.when(cond).then(pl.lit(src))
    return (e_lab.otherwise(None), e_lvl.otherwise(pl.lit("unassigned")),
            e_src.otherwise(pl.lit("none")))

def assign(df, target):
    return _fold(tiered_terms(target))

def score_assignment(df, lab, lvl):
    d = df.with_columns(pl.Series("_n", normfn(df[lab].fill_null("").to_numpy().astype(str))))
    return d.with_columns(
        (pl.when(pl.col(lvl) == "specific").then(pl.col("_n") == pl.col("true_specific_n"))
           .when(pl.col(lvl) == "L1").then(pl.col(lab) == pl.col("true_L1"))
           .when(pl.col(lvl) == "category").then(pl.col(lab) == pl.col("true_category"))
           .otherwise(None)).fill_null(False).alias("_ok")).drop("_n")

print("tiered rule defined for targets:", TARGETS)

tiered rule defined for targets: [0.9, 0.8, 0.7]


Apply the rule to every held-out protein at every target, and report the reporting slice.

In [34]:
for t in TARGETS:
    lab, lvl, src = assign(heldout, t)
    tag = f"{int(round(t*100))}"
    heldout = heldout.with_columns(lab.alias(f"label_{tag}"), lvl.alias(f"level_{tag}"),
                           src.alias(f"source_{tag}"))
PT = f"{int(PRIMARY_TARGET*100)}"
heldout = heldout.with_columns(pl.col(f"label_{PT}").alias("final_label"),
                       pl.col(f"level_{PT}").alias("final_level"),
                       pl.col(f"source_{PT}").alias("label_source"))
rpt = score_assignment(heldout.filter(pl.col("split") == "report"), "final_label", "final_level")
tot = rpt.height
print(f"reporting slice {tot:,} proteins at target {PRIMARY_TARGET:.2f}")
print(rpt.group_by(["final_level", "label_source"]).agg(
        pl.len().alias("n"), pl.col("_ok").mean().round(4).alias("precision"))
      .with_columns((100 * pl.col("n") / tot).round(2).alias("pct_of_slice"))
      .sort(["final_level", "label_source"]))
asn = rpt.filter(pl.col("final_level") != "unassigned")
_s = prec_stat(asn["_ok"].cast(pl.Float64).to_numpy())
print(f"\nassigned {asn.height:,} of {tot:,} ({100*asn.height/tot:.1f}%), "
      f"precision at assigned depth {_s['precision']:.4f} "
      f"[{_s['lo']:.4f}, {_s['hi']:.4f}]")

reporting slice 41,957 proteins at target 0.90
shape: (7, 5)
┌─────────────┬──────────────┬───────┬───────────┬──────────────┐
│ final_level ┆ label_source ┆ n     ┆ precision ┆ pct_of_slice │
│ ---         ┆ ---          ┆ ---   ┆ ---       ┆ ---          │
│ str         ┆ str          ┆ u64   ┆ f64       ┆ f64          │
╞═════════════╪══════════════╪═══════╪═══════════╪══════════════╡
│ L1          ┆ cluster      ┆ 11185 ┆ 0.9247    ┆ 26.66        │
│ L1          ┆ embedding    ┆ 63    ┆ 0.8889    ┆ 0.15         │
│ category    ┆ cluster      ┆ 68    ┆ 1.0       ┆ 0.16         │
│ category    ┆ embedding    ┆ 738   ┆ 0.836     ┆ 1.76         │
│ specific    ┆ cluster      ┆ 24320 ┆ 0.9502    ┆ 57.96        │
│ specific    ┆ embedding    ┆ 2323  ┆ 0.8945    ┆ 5.54         │
│ unassigned  ┆ none         ┆ 3260  ┆ 0.0       ┆ 7.77         │
└─────────────┴──────────────┴───────┴───────────┴──────────────┘

assigned 38,697 of 41,957 (92.2%), precision at assigned depth 0.9373 [0.9348, 0

The per-protein evaluation frame, with both tiers and the tiered outcome on every row.

In [35]:
_keep = ["Protein", "split", "raw_category", "raw_L1", "raw_specific",
         "true_category", "true_L1", "true_specific", "true_specific_n",
         "y_specific", "y_L1", "y_category", "p_specific", "p_L1", "p_category",
         "d1", "d2", "margin", "ratio", "vote_purity_k5", "vote_purity_k20",
         "vote_purity_k50", "density_norm_d1", "hub_count", "max_ident_to_train",
         "final_label", "final_level", "label_source"]
_keep += [f"{k}_{int(round(t*100))}" for t in TARGETS for k in ("label", "level", "source")]
_keep += [f"c{p}_{lvl}" for p in PCTS for lvl in LEVELS]
_keep += [f"c{p}_{s}" for p in PCTS for s in ("cluster", "frac", "nlab")]
_keep = [c for c in dict.fromkeys(_keep) if c in heldout.columns]
_ev = heldout.select(_keep)
_p = OUT / "evaluation_per_protein.parquet"
_ev.write_parquet(_p)
print(f"evaluation_per_protein: {_ev.height:,} x {_ev.width}  ({_p.stat().st_size/1e6:.1f} MB)")
print(_ev.group_by("split").agg(pl.len().alias("n")).sort("n", descending=True))
print("label_source on the reporting slice:")
print(_ev.filter(pl.col("split") == "report").group_by("label_source").agg(pl.len().alias("n"))
          .sort("n", descending=True))

evaluation_per_protein: 139,856 x 61  (15.7 MB)
shape: (3, 2)
┌────────┬───────┐
│ split  ┆ n     │
│ ---    ┆ ---   │
│ str    ┆ u64   │
╞════════╪═══════╡
│ fit    ┆ 55942 │
│ report ┆ 41957 │
│ calib  ┆ 41957 │
└────────┴───────┘
label_source on the reporting slice:
shape: (3, 2)
┌──────────────┬───────┐
│ label_source ┆ n     │
│ ---          ┆ ---   │
│ str          ┆ u64   │
╞══════════════╪═══════╡
│ cluster      ┆ 35573 │
│ none         ┆ 3260  │
│ embedding    ┆ 3124  │
└──────────────┴───────┘


## 7. Ablations: clustering alone, embedding alone, and the two tiered

Three rules are evaluated on the reporting slice at the same precision target. `cluster_only` uses tier 1 alone at the identity thresholds from section 5. `embedding_only` uses tier 2 alone, with probability thresholds fit on the whole calibration slice. `tiered` is the deployed rule, with tier 2's thresholds fit on what tier 1 leaves. All three assign the deepest granularity first and hold the same nominal precision, so they differ in coverage and depth.

The three rules, and their coverage and precision at each granularity on the reporting slice.

In [36]:
def variant_terms(target, method):
    terms = []
    for lvl in LEVELS:
        c = t1_col(lvl, target)
        if method in ("cluster_only", "tiered") and c is not None:
            lab = (pl.col(c).replace_strict(NORM2DISP, default=None) if lvl == "specific"
                   else pl.col(c))
            terms.append((pl.col(c).is_not_null(), lab, lvl, "cluster"))
        if method in ("embedding_only", "tier2_deployed", "tiered"):
            thr = threshold_full(lvl, target) if method == "embedding_only" else threshold_for(lvl, target)
            cond = pl.col("p_" + lvl).is_not_null() & (pl.col("p_" + lvl) >= thr)
            if method == "tiered" and c is not None:
                cond = cond & pl.col(c).is_null()
            terms.append((cond, pl.col(EMB_DISP[lvl]), lvl, "embedding"))
    return terms

def apply_variant(df, target, method):
    lab, lvl, src = _fold(variant_terms(target, method))
    return score_assignment(
        df.with_columns(lab.alias("_lab"), lvl.alias("_lvl"), src.alias("_src")), "_lab", "_lvl")

rows = []
for t in TARGETS:
    for m in METHODS:
        d = apply_variant(report, t, m)
        n = d.height
        for lvl in LEVELS:
            s = d.filter(pl.col("_lvl") == lvl)
            st = prec_stat(s["_ok"].cast(pl.Float64).to_numpy() if s.height else np.array([]))
            rows.append({"target": t, "method": m, "level": lvl, "n_report": n,
                         "n_assigned": s.height, "coverage": s.height / n,
                         "precision": st["precision"], "lo": st["lo"], "hi": st["hi"],
                         "recall": (st["precision"] * s.height / n) if st["precision"] is not None else 0.0})
        a = d.filter(pl.col("_lvl") != "unassigned")
        st = prec_stat(a["_ok"].cast(pl.Float64).to_numpy() if a.height else np.array([]))
        depth = {"specific": 3, "L1": 2, "category": 1, "unassigned": 0}
        rows.append({"target": t, "method": m, "level": "any", "n_report": n,
                     "n_assigned": a.height, "coverage": a.height / n,
                     "precision": st["precision"], "lo": st["lo"], "hi": st["hi"],
                     "recall": (st["precision"] * a.height / n) if st["precision"] is not None else 0.0,
                     "mean_depth": float(d["_lvl"].replace_strict(depth).mean())})
ablation = pl.DataFrame(rows)
print(ablation.filter(pl.col("target") == PRIMARY_TARGET)
          .select(["method", "level", "n_assigned", "coverage", "precision", "lo", "hi", "recall"]))
ablation.write_parquet(OUT / "fig_tier_coverage_precision.parquet")

shape: (12, 8)
┌────────────────┬──────────┬────────────┬──────────┬───────────┬──────────┬──────────┬──────────┐
│ method         ┆ level    ┆ n_assigned ┆ coverage ┆ precision ┆ lo       ┆ hi       ┆ recall   │
│ ---            ┆ ---      ┆ ---        ┆ ---      ┆ ---       ┆ ---      ┆ ---      ┆ ---      │
│ str            ┆ str      ┆ i64        ┆ f64      ┆ f64       ┆ f64      ┆ f64      ┆ f64      │
╞════════════════╪══════════╪════════════╪══════════╪═══════════╪══════════╪══════════╪══════════╡
│ cluster_only   ┆ specific ┆ 24320      ┆ 0.579641 ┆ 0.950164  ┆ 0.947358 ┆ 0.952829 ┆ 0.550754 │
│ cluster_only   ┆ L1       ┆ 13103      ┆ 0.312296 ┆ 0.932382  ┆ 0.927955 ┆ 0.936556 ┆ 0.291179 │
│ cluster_only   ┆ category ┆ 70         ┆ 0.001668 ┆ 0.985714  ┆ 0.923416 ┆ 0.997474 ┆ 0.001645 │
│ cluster_only   ┆ any      ┆ 37493      ┆ 0.893605 ┆ 0.944016  ┆ 0.941643 ┆ 0.946298 ┆ 0.843578 │
│ embedding_only ┆ specific ┆ 9714       ┆ 0.231523 ┆ 0.901894  ┆ 0.895819 ┆ 0.907652 ┆ 0.2088

### 7.1 What each tier contributes that the other cannot

The overlap between the two tiers on the reporting slice, at the primary target. A protein that both tiers can label is one where either would do, and the interesting counts are the proteins only one tier reaches.

In [37]:
rows = []
for lvl in LEVELS:
    c, thr = t1_col(lvl, PRIMARY_TARGET), THR[PRIMARY_TARGET][lvl]
    d = report.with_columns(
        (pl.col(c).is_not_null() if c else pl.lit(False)).alias("_t1"),
        (pl.col("p_" + lvl).is_not_null() & (pl.col("p_" + lvl) >= threshold_full(lvl, PRIMARY_TARGET))).alias("_t2"))
    for nm, f in [("both", pl.col("_t1") & pl.col("_t2")),
                  ("cluster_only", pl.col("_t1") & ~pl.col("_t2")),
                  ("embedding_only", ~pl.col("_t1") & pl.col("_t2")),
                  ("neither", ~pl.col("_t1") & ~pl.col("_t2"))]:
        s = d.filter(f)
        y1 = ((s[c] == s[TRUE[lvl]]).fill_null(False).cast(pl.Float64).to_numpy()
              if (c and s.height) else np.array([]))
        y2 = s["y_" + lvl].cast(pl.Float64).to_numpy() if s.height else np.array([])
        rows.append({"level": lvl, "reachable_by": nm, "n": s.height,
                     "pct_of_slice": 100 * s.height / d.height,
                     "tier1_precision": prec_stat(y1)["precision"],
                     "tier2_precision": prec_stat(y2)["precision"]})
ovl = pl.DataFrame(rows)
print(ovl)
ovl.write_parquet(OUT / "fig_tier_overlap.parquet")

shape: (12, 6)
┌──────────┬────────────────┬───────┬──────────────┬─────────────────┬─────────────────┐
│ level    ┆ reachable_by   ┆ n     ┆ pct_of_slice ┆ tier1_precision ┆ tier2_precision │
│ ---      ┆ ---            ┆ ---   ┆ ---          ┆ ---             ┆ ---             │
│ str      ┆ str            ┆ i64   ┆ f64          ┆ f64             ┆ f64             │
╞══════════╪════════════════╪═══════╪══════════════╪═════════════════╪═════════════════╡
│ specific ┆ both           ┆ 6888  ┆ 16.416808    ┆ 0.982433        ┆ 0.92349         │
│ specific ┆ cluster_only   ┆ 17432 ┆ 41.547298    ┆ 0.937414        ┆ 0.244034        │
│ specific ┆ embedding_only ┆ 2826  ┆ 6.735467     ┆ 0.0             ┆ 0.849257        │
│ specific ┆ neither        ┆ 14811 ┆ 35.300427    ┆ 0.0             ┆ 0.181487        │
│ L1       ┆ both           ┆ 12899 ┆ 30.74338     ┆ 0.973951        ┆ 0.910303        │
│ L1       ┆ cluster_only   ┆ 24455 ┆ 58.285864    ┆ 0.930607        ┆ 0.422817        │
│ L1  

## 8. Recall at matched precision

This section sweeps the precision target from 0.50 to 0.99 and reports, for each method and granularity, the recall attainable while holding that precision. Recall is the fraction of all reporting proteins labeled correctly at that granularity, so it penalizes both wrong labels and missing ones.

Every operating point in the sweep is chosen on the calibration slice and evaluated on the reporting slice.

In [38]:
rows = []
for t in TARGETS_R:
    for m in METHODS:
        d = apply_variant(report, t, m)
        n = d.height
        for lvl in LEVELS:
            s = d.filter(pl.col("_lvl") == lvl)
            k = float(s["_ok"].sum()) if s.height else 0.0
            pr = (k / s.height) if s.height else None
            rows.append({"target": t, "method": m, "level": lvl,
                         "coverage": s.height / n, "precision": pr, "recall": k / n,
                         "met": bool(pr is not None and pr >= t)})
        a = d.filter(pl.col("_lvl") != "unassigned")
        k = float(a["_ok"].sum()) if a.height else 0.0
        rows.append({"target": t, "method": m, "level": "any",
                     "coverage": a.height / n, "precision": (k / a.height) if a.height else None,
                     "recall": k / n, "met": bool(a.height and k / a.height >= t)})
rec = pl.DataFrame(rows)
print(rec.filter(pl.col("level") == "any").pivot(on="method", index="target", values="recall"))

shape: (9, 4)
┌────────┬──────────────┬────────────────┬──────────┐
│ target ┆ cluster_only ┆ embedding_only ┆ tiered   │
│ ---    ┆ ---          ┆ ---            ┆ ---      │
│ f64    ┆ f64          ┆ f64            ┆ f64      │
╞════════╪══════════════╪════════════════╪══════════╡
│ 0.5    ┆ 0.770408     ┆ 0.444074       ┆ 0.817861 │
│ 0.6    ┆ 0.770408     ┆ 0.526158       ┆ 0.835999 │
│ 0.7    ┆ 0.770408     ┆ 0.657745       ┆ 0.84465  │
│ 0.75   ┆ 0.770408     ┆ 0.738923       ┆ 0.848416 │
│ 0.8    ┆ 0.770408     ┆ 0.776676       ┆ 0.833353 │
│ 0.85   ┆ 0.770408     ┆ 0.820221       ┆ 0.814977 │
│ 0.9    ┆ 0.843578     ┆ 0.777868       ┆ 0.864456 │
│ 0.95   ┆ 0.862025     ┆ 0.475701       ┆ 0.873394 │
│ 0.99   ┆ 0.886932     ┆ 0.225421       ┆ 0.892056 │
└────────┴──────────────┴────────────────┴──────────┘


## 9. Tier performance by identity to the training set

A post hoc breakdown. Identity to the nearest training protein is unknown when a label is assigned, so no threshold depends on it.

In [39]:
rows = []
for nm, lo, hi in STRATA:
    g = report.filter((pl.col("max_ident_to_train") >= lo) & (pl.col("max_ident_to_train") < hi))
    if not g.height:
        continue
    for m in METHODS:
        d = apply_variant(g, PRIMARY_TARGET, m)
        a = d.filter(pl.col("_lvl") != "unassigned")
        st = prec_stat(a["_ok"].cast(pl.Float64).to_numpy() if a.height else np.array([]))
        rows.append({"stratum": nm, "ident_lo": lo, "ident_hi": min(hi, 1.0), "method": m,
                     "n_report": g.height, "n_assigned": a.height,
                     "coverage": a.height / g.height, "precision": st["precision"],
                     "lo": st["lo"], "hi": st["hi"],
                     "recall": float(a["_ok"].sum()) / g.height if a.height else 0.0})
tstrat = pl.DataFrame(rows)
print(tstrat.select(["stratum", "n_report", "method", "n_assigned", "coverage", "precision", "recall"]))
tstrat.write_parquet(OUT / "fig_tier_by_identity.parquet")

shape: (12, 7)
┌─────────┬──────────┬────────────────┬────────────┬──────────┬───────────┬──────────┐
│ stratum ┆ n_report ┆ method         ┆ n_assigned ┆ coverage ┆ precision ┆ recall   │
│ ---     ┆ ---      ┆ ---            ┆ ---        ┆ ---      ┆ ---       ┆ ---      │
│ str     ┆ i64      ┆ str            ┆ i64        ┆ f64      ┆ f64       ┆ f64      │
╞═════════╪══════════╪════════════════╪════════════╪══════════╪═══════════╪══════════╡
│ <50%    ┆ 6879     ┆ cluster_only   ┆ 4824       ┆ 0.701265 ┆ 0.94092   ┆ 0.659834 │
│ <50%    ┆ 6879     ┆ embedding_only ┆ 5878       ┆ 0.854485 ┆ 0.829874  ┆ 0.709115 │
│ <50%    ┆ 6879     ┆ tiered         ┆ 5402       ┆ 0.785289 ┆ 0.92003   ┆ 0.722489 │
│ 50-90%  ┆ 30434    ┆ cluster_only   ┆ 28213      ┆ 0.927022 ┆ 0.940559  ┆ 0.87192  │
│ 50-90%  ┆ 30434    ┆ embedding_only ┆ 27643      ┆ 0.908293 ┆ 0.869117  ┆ 0.789413 │
│ 50-90%  ┆ 30434    ┆ tiered         ┆ 28785      ┆ 0.945817 ┆ 0.935939  ┆ 0.885227 │
│ >=90%   ┆ 4644     ┆ clust

Tier-1 reach as a function of identity and clustering threshold, which is the same question asked of the clustering alone rather than of the assignment rule.

In [40]:
rows = []
for nm, lo, hi in STRATA:
    g = report.filter((pl.col("max_ident_to_train") >= lo) & (pl.col("max_ident_to_train") < hi))
    if not g.height:
        continue
    for p in PCTS:
        for lvl in LEVELS:
            s = g.filter(pl.col(f"c{p}_{lvl}").is_not_null())
            y = ((s[f"c{p}_{lvl}"] == s[TRUE[lvl]]).fill_null(False).cast(pl.Float64).to_numpy()
                 if s.height else np.array([]))
            rows.append({"stratum": nm, "min_seq_id": p / 100, "level": lvl,
                         "n_report": g.height, "n_covered": s.height,
                         "reach": s.height / g.height, "precision": prec_stat(y)["precision"]})
reach_id = pl.DataFrame(rows)
print(reach_id.filter(pl.col("level") == "specific")
              .pivot(on="stratum", index="min_seq_id", values="reach"))

shape: (4, 5)
┌────────────┬──────────┬──────────┬──────────┬──────────┐
│ min_seq_id ┆ <50%     ┆ 50-90%   ┆ >=90%    ┆ all      │
│ ---        ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ f64        ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞════════════╪══════════╪══════════╪══════════╪══════════╡
│ 0.3        ┆ 0.688472 ┆ 0.916212 ┆ 0.952842 ┆ 0.882928 │
│ 0.5        ┆ 0.018317 ┆ 0.658376 ┆ 0.895134 ┆ 0.579641 │
│ 0.7        ┆ 0.0      ┆ 0.189656 ┆ 0.823859 ┆ 0.228758 │
│ 0.9        ┆ 0.0      ┆ 0.01702  ┆ 0.745047 ┆ 0.094811 │
└────────────┴──────────┴──────────┴──────────┴──────────┘


## 10. Validation controls for the deployed rule

Label permutation, cross-domain, and specificity controls under the deployed 1-NN rule.

Label permutation: propagation accuracy must collapse to the base rate when training labels are shuffled.

In [41]:
perm_rng = np.random.default_rng(20260716)
tc_true = meta["Protein Classification"].to_numpy()
def prop_cat(cls_vec, Iv):
    nb = cls_vec[Iv]
    okv = np.isin(nb, CAT3)
    hv = okv.any(1)
    fv = np.where(hv, okv.argmax(1), 0)
    return np.where(hv, nb[np.arange(Iv.shape[0]), fv], None), hv
pc, hv = prop_cat(train_class, Ifull)
real = float((pc[hv] == tc_true[hv]).mean())
vals, cnts = np.unique(tc_true[hv], return_counts=True)
basec = float(cnts.max() / cnts.sum())
pos = np.where(np.isin(train_class, CAT3))[0]
perms = []
for _ in range(5):
    tp = train_class.copy()
    tp[pos] = train_class[pos][perm_rng.permutation(pos.size)]
    p2, h2 = prop_cat(tp, Ifull)
    perms.append(float((p2[h2] == tc_true[h2]).mean()))
perms = np.array(perms)
print(f"real category precision       : {real:.4f} on {int(hv.sum()):,} held-out rows")
print(f"label-permuted (5 shuffles)   : {perms.mean():.4f} +/- {perms.std():.4f}")
print(f"majority-class baseline       : {basec:.4f}")
print(f"real minus permuted           : {real - perms.mean():+.4f}")

real category precision       : 0.8952 on 397,069 held-out rows
label-permuted (5 shuffles)   : 0.5154 +/- 0.0011
majority-class baseline       : 0.6682
real minus permuted           : +0.3798


Cross-domain control: does precision depend on whether the nearest neighbor sits in viral or bacterial context?

In [42]:
gm = pl.read_parquet(TRAIN_ANNO / "checkamg_annotate_genomad_dataset" / "results" / "final_results.parquet",
                     columns=["Protein"]).with_columns(pl.lit("genomad").alias("src"))
pg = pl.read_parquet(TRAIN_ANNO / "checkamg_annotate_progenomes_dataset" / "results" / "final_results.parquet",
                     columns=["Protein"]).with_columns(pl.lit("progenomes").alias("src"))
t6 = (tref.with_row_index("idx").join(pl.concat([gm, pg]).unique("Protein"), on="Protein", how="left").sort("idx"))
src = t6["src"].fill_null("other").to_numpy()
nbc = train_class[Ifull]
okc = np.isin(nbc, CAT3)
hc = okc.any(1)
fc2 = np.where(hc, okc.argmax(1), 0)
rw = np.arange(Ifull.shape[0])
pcat = np.where(hc, nbc[rw, fc2], None)
nsrc = np.where(hc, src[Ifull[rw, fc2]], None)
m6 = hc & np.isin(tc_true, CAT3)
cor = (pcat[m6] == tc_true[m6])
sm = nsrc[m6]
cross_domain = {}
for s in ["genomad", "progenomes"]:
    k = sm == s
    if k.sum():
        cross_domain[s] = float(cor[k].mean())
        print(f"  neighbor in {s:11s} context n={int(k.sum()):>7,}: precision {cross_domain[s]:.4f}")
print(f"  overall {cor.mean():.4f}")

  neighbor in genomad     context n=332,432: precision 0.8904
  neighbor in progenomes  context n= 64,637: precision 0.9201
  overall 0.8952


Specificity control: non-auxiliary virion structural genes should rarely receive a confident AVG-like call.

In [43]:
Is = np.load(NOVEL / "test_struct_I.npy")
Ds = np.load(NOVEL / "test_struct_D.npy").astype(np.float64)
def near_avglike(Iv, Dv):
    nb = train_class[Iv]
    okv = np.isin(nb, CAT3)
    hv = okv.any(1)
    fv = np.where(hv, okv.argmax(1), 0)
    return hv, np.where(hv, Dv[np.arange(Iv.shape[0]), fv], np.inf)
nh, nd = near_avglike(Is, Ds)
ah, ad = near_avglike(Ifull, Dfull)
CONF = 11.77
spec = {"n_struct": int(Is.shape[0]), "struct_any": float(nh.mean()),
        "struct_conf": float((nd <= CONF).mean()), "struct_median": float(np.median(nd[np.isfinite(nd)])),
        "n_avg": int(Ifull.shape[0]), "avg_any": float(ah.mean()),
        "avg_conf": float((ad <= CONF).mean()), "avg_median": float(np.median(ad[np.isfinite(ad)]))}
print(f"structural (negative) n={spec['n_struct']:,}: any AVG-like {100*spec['struct_any']:.1f}%, "
      f"within {CONF} {100*spec['struct_conf']:.2f}%, median distance {spec['struct_median']:.1f}")
print(f"true AVG-like       n={spec['n_avg']:,}: any AVG-like {100*spec['avg_any']:.1f}%, "
      f"within {CONF} {100*spec['avg_conf']:.2f}%, median distance {spec['avg_median']:.1f}")

structural (negative) n=172,379: any AVG-like 0.5%, within 11.77 0.00%, median distance 148.1
true AVG-like       n=415,180: any AVG-like 95.6%, within 11.77 7.24%, median distance 57.1


Frozen ESM-2 baseline, from the search in Section 0.1.

In [44]:
esm = dict(np.load(NOVEL / "step6b_nn.npz")) if (NOVEL / "step6b_nn.npz").exists() else None
if esm is not None:
    print(f"matched n={int(esm['n_matched']):,}: CheckAMG-PST {float(esm['pst_prec']):.4f} vs "
          f"frozen ESM-2 {float(esm['esm_prec']):.4f}")

matched n=133,086: CheckAMG-PST 0.8956 vs frozen ESM-2 0.9882


Assemble the validation control table.

In [45]:
val = []
val += [{"panel": "Unseen held-out proteins", "condition": "True labels", "value": real, "metric": "category precision"},
        {"panel": "Unseen held-out proteins", "condition": "Shuffled labels", "value": float(perms.mean()), "metric": "category precision"},
        {"panel": "Unseen held-out proteins", "condition": "Majority class", "value": basec, "metric": "category precision"}]
for s, nm in [("genomad", "Viral context"), ("progenomes", "Bacterial context")]:
    if s in cross_domain:
        val.append({"panel": "Unseen genomic context", "condition": nm, "value": cross_domain[s], "metric": "category precision"})
val += [{"panel": "Specificity", "condition": "Auxiliary genes", "value": spec["avg_conf"], "metric": "confident-call rate"},
        {"panel": "Specificity", "condition": "Structural genes", "value": spec["struct_conf"], "metric": "confident-call rate"}]
if esm is not None:
    val += [{"panel": "Contribution of finetuning", "condition": "CheckAMG-PST", "value": float(esm["pst_prec"]), "metric": "category precision"},
            {"panel": "Contribution of finetuning", "condition": "ESM-2 backbone only", "value": float(esm["esm_prec"]), "metric": "category precision"}]
validation = pl.DataFrame(val)
validation.write_parquet(OUT / "fig_validation_controls.parquet")
print(validation)

shape: (9, 4)
┌────────────────────────────┬─────────────────────┬──────────┬─────────────────────┐
│ panel                      ┆ condition           ┆ value    ┆ metric              │
│ ---                        ┆ ---                 ┆ ---      ┆ ---                 │
│ str                        ┆ str                 ┆ f64      ┆ str                 │
╞════════════════════════════╪═════════════════════╪══════════╪═════════════════════╡
│ Unseen held-out proteins   ┆ True labels         ┆ 0.895215 ┆ category precision  │
│ Unseen held-out proteins   ┆ Shuffled labels     ┆ 0.515434 ┆ category precision  │
│ Unseen held-out proteins   ┆ Majority class      ┆ 0.668163 ┆ category precision  │
│ Unseen genomic context     ┆ Viral context       ┆ 0.890378 ┆ category precision  │
│ Unseen genomic context     ┆ Bacterial context   ┆ 0.920092 ┆ category precision  │
│ Specificity                ┆ Auxiliary genes     ┆ 0.072359 ┆ confident-call rate │
│ Specificity                ┆ Structura

## 11. Frequency recovery

On the reporting slice, predicted and true label counts are compared for every label with a true count of at least 20, under the unthresholded 1-NN rule (`raw_specific`) and under adaptive granularity. The acceptance criterion is a Spearman correlation of at least 0.8 and an overlap@12 of at least 8 of 12 in every category.

Score the reporting slice under the adaptive scheme.

In [46]:
report = heldout.filter(pl.col("split") == "report")
lab, lvl, _ = assign(report, PRIMARY_TARGET)
report = report.with_columns(lab.alias("adaptive_label"), lvl.alias("adaptive_level"))
print(report.group_by("adaptive_level").agg(pl.len().alias("n"))
        .with_columns((100 * pl.col("n") / report.height).round(2).alias("pct")).sort("n", descending=True))

shape: (4, 3)
┌────────────────┬───────┬───────┐
│ adaptive_level ┆ n     ┆ pct   │
│ ---            ┆ ---   ┆ ---   │
│ str            ┆ u64   ┆ f64   │
╞════════════════╪═══════╪═══════╡
│ specific       ┆ 26643 ┆ 63.5  │
│ L1             ┆ 11248 ┆ 26.81 │
│ unassigned     ┆ 3260  ┆ 7.77  │
│ category       ┆ 806   ┆ 1.92  │
└────────────────┴───────┴───────┘


Predicted versus true counts per specific label, under both methods.

In [47]:
truec = report.group_by("true_specific").len().rename({"len": "true_n"}).filter(pl.col("true_n") >= 20)
oldc = report.group_by("raw_specific").len().rename({"raw_specific": "true_specific", "len": "old_n"})
newc = (report.filter(pl.col("adaptive_level") == "specific").group_by("adaptive_label").len()
          .rename({"adaptive_label": "true_specific", "len": "new_n"}))
frequency_counts = (truec.join(oldc, on="true_specific", how="left").join(newc, on="true_specific", how="left")
           .with_columns(pl.col("old_n").fill_null(0), pl.col("new_n").fill_null(0))
           .with_columns(pl.col("true_specific").replace_strict(EX2CAT, default=None).alias("category"))
           .sort("true_specific"))
print(f"labels with true count >= 20: {frequency_counts.height:,}")

def spear(a, b):
    if len(a) < 3:
        return float("nan")
    return float(np.corrcoef(np.argsort(np.argsort(a)), np.argsort(np.argsort(b)))[0, 1])

rows = []
for name, col in [("old_raw_specific", "old_n"), ("adaptive", "new_n")]:
    rows.append({"method": name, "scope": "overall",
                 "spearman": round(spear(frequency_counts["true_n"].to_numpy(), frequency_counts[col].to_numpy()), 4),
                 "n_labels": frequency_counts.height})
    for c in CAT3:
        s = frequency_counts.filter(pl.col("category") == c)
        rows.append({"method": name, "scope": c,
                     "spearman": round(spear(s["true_n"].to_numpy(), s[col].to_numpy()), 4),
                     "n_labels": s.height})
freq_rec = pl.DataFrame(rows)
print(freq_rec)

labels with true count >= 20: 539
shape: (8, 4)
┌──────────────────┬───────────────┬──────────┬──────────┐
│ method           ┆ scope         ┆ spearman ┆ n_labels │
│ ---              ┆ ---           ┆ ---      ┆ ---      │
│ str              ┆ str           ┆ f64      ┆ i64      │
╞══════════════════╪═══════════════╪══════════╪══════════╡
│ old_raw_specific ┆ overall       ┆ 0.7109   ┆ 539      │
│ old_raw_specific ┆ metabolic     ┆ 0.6278   ┆ 377      │
│ old_raw_specific ┆ physiological ┆ 0.8411   ┆ 41       │
│ old_raw_specific ┆ regulatory    ┆ 0.8831   ┆ 121      │
│ adaptive         ┆ overall       ┆ 0.6969   ┆ 539      │
│ adaptive         ┆ metabolic     ┆ 0.7236   ┆ 377      │
│ adaptive         ┆ physiological ┆ 0.8348   ┆ 41       │
│ adaptive         ┆ regulatory    ┆ 0.6117   ┆ 121      │
└──────────────────┴───────────────┴──────────┴──────────┘


Overlap@12 between the predicted and true top-12 labels per category.

In [48]:
rows = []
for name, col in [("old_raw_specific", "old_n"), ("adaptive", "new_n")]:
    for c in CAT3:
        s = frequency_counts.filter(pl.col("category") == c)
        t12 = set(s.sort(["true_n", "true_specific"], descending=[True, False]).head(12)["true_specific"].to_list())
        p12 = set(s.sort([col, "true_specific"], descending=[True, False]).head(12)["true_specific"].to_list())
        rows.append({"method": name, "category": c, "overlap_at_12": len(t12 & p12),
                     "n_labels": s.height})
ov = pl.DataFrame(rows)
print(ov)
sl_rows = []
for name, col in [("old_raw_specific", "old_n"), ("adaptive", "new_n")]:
    m = frequency_counts.filter((pl.col("true_n") > 0) & (pl.col(col) > 0))
    x = np.log(m["true_n"].to_numpy().astype(float))
    y = np.log(m[col].to_numpy().astype(float))
    slope = float(np.polyfit(x, y, 1)[0]) if len(x) > 2 else float("nan")
    sl_rows.append({"method": name, "log_log_slope": round(slope, 4), "n_labels": m.height})
slopes = pl.DataFrame(sl_rows)
print(slopes)

shape: (6, 4)
┌──────────────────┬───────────────┬───────────────┬──────────┐
│ method           ┆ category      ┆ overlap_at_12 ┆ n_labels │
│ ---              ┆ ---           ┆ ---           ┆ ---      │
│ str              ┆ str           ┆ i64           ┆ i64      │
╞══════════════════╪═══════════════╪═══════════════╪══════════╡
│ old_raw_specific ┆ metabolic     ┆ 8             ┆ 377      │
│ old_raw_specific ┆ physiological ┆ 10            ┆ 41       │
│ old_raw_specific ┆ regulatory    ┆ 11            ┆ 121      │
│ adaptive         ┆ metabolic     ┆ 9             ┆ 377      │
│ adaptive         ┆ physiological ┆ 10            ┆ 41       │
│ adaptive         ┆ regulatory    ┆ 10            ┆ 121      │
└──────────────────┴───────────────┴───────────────┴──────────┘
shape: (2, 3)
┌──────────────────┬───────────────┬──────────┐
│ method           ┆ log_log_slope ┆ n_labels │
│ ---              ┆ ---           ┆ ---      │
│ str              ┆ f64           ┆ i64      │
╞═══════════

Evaluate the criterion.

In [49]:
sp_ok = {r["method"]: r["spearman"] for r in freq_rec.filter(pl.col("scope") == "overall").iter_rows(named=True)}
ov_min = {m: int(ov.filter(pl.col("method") == m)["overlap_at_12"].min()) for m in ["old_raw_specific", "adaptive"]}
FREQ_GATE = bool(sp_ok["adaptive"] >= 0.8 and ov_min["adaptive"] >= 8)
print(f"adaptive: Spearman {sp_ok['adaptive']:.3f} (need >= 0.8), "
      f"min overlap@12 {ov_min['adaptive']}/12 (need >= 8)")
print(f"old     : Spearman {sp_ok['old_raw_specific']:.3f}, min overlap@12 {ov_min['old_raw_specific']}/12")
print(f"acceptance criterion: {'PASS' if FREQ_GATE else 'FAIL'}")

adaptive: Spearman 0.697 (need >= 0.8), min overlap@12 9/12 (need >= 8)
old     : Spearman 0.711, min overlap@12 8/12
acceptance criterion: FAIL


## 12. The applied set

The rule fixed in sections 5 and 6 is applied to the soil and human-gut AVGs, which have no ground truth. The full union of applied AVGs is assembled first, so every protein in the output table is either labeled by a tier or recorded as `unassigned`.

Assemble the full union of applied AVGs, then attach the propagation features to it.

In [50]:
applied_features = pl.read_parquet(CACHE / "applied_features.parquet").join(ap_id, on="Protein", how="left")
ANNC = ["Protein", "Genome", "Classification (annotate)", "Viral Confidence Level (annotate)", "Function (annotate)"]
full = pl.concat([
    pl.read_parquet(agg(ds), columns=ANNC)
      .join(pl.read_csv(den(ds) / "predictions.tsv", separator="\t",
                        columns=["Protein", "Viral prob", "Final AVG prob"]), on="Protein", how="left")
      .with_columns(pl.lit(ds).alias("dataset")) for ds in DATASETS])
thr = json.loads((FILES / "pst_thresholds.json").read_text())
full = full.with_columns(
    (pl.col("Classification (annotate)").is_in(CAT3)
     & pl.col("Viral Confidence Level (annotate)").is_in(VIRAL_OK)).alias("annotate_avg"),
    ((pl.col("Final AVG prob") >= thr["AVG"]["medium"])
     & (pl.col("Viral prob") >= thr["Viral"]["medium"])).alias("denovo_avg"))
full = full.filter(pl.col("annotate_avg") | pl.col("denovo_avg")).select(
    ["Protein", "Genome", "dataset", "annotate_avg", "denovo_avg",
     pl.col("Classification (annotate)").alias("annotate_classification"),
     pl.col("Function (annotate)").alias("annotate_function")]).with_columns(
    (pl.col("denovo_avg") & ~pl.col("annotate_avg")
     & (pl.col("annotate_function").is_null() | (pl.col("annotate_function") == ""))
     ).alias("homology_invisible"))
n_feat = applied_features.height
missing = full.height - n_feat
applied = full.join(applied_features.drop(["Genome", "dataset", "annotate_avg", "denovo_avg",
                          "homology_invisible", "annotate_classification", "annotate_function"]),
                on="Protein", how="left")
assert applied.height == full.height, "the union join changed the row count"
n_missing_with_function = (full.join(applied_features.select("Protein"), on="Protein", how="anti")
                           .filter(pl.col("annotate_function").is_not_null() & (pl.col("annotate_function") != ""))
                           .height)
print(f"full applied AVG set: {full.height:,}")
print(f"carrying a de-novo embedding, so scorable by tier 2: {n_feat:,}")
print(f"without a de-novo embedding: {missing:,} ({100*missing/full.height:.3f}%), "
      f"of which {n_missing_with_function:,} carry their own annotate function")

full applied AVG set: 541,199
carrying a de-novo embedding, so scorable by tier 2: 541,007
without a de-novo embedding: 192 (0.035%), of which 192 carry their own annotate function


Score the union with the calibrated models. Only proteins that carry an embedding are scored. The rest keep a null probability rather than a probability computed from zero-filled features.

In [51]:
has_emb = applied["d1"].is_not_null().to_numpy()
Xap = np.nan_to_num(applied.select(FEATS).to_numpy().astype(np.float64),
                    nan=0.0, posinf=0.0, neginf=0.0)
for lvl in LEVELS:
    v = np.full(applied.height, np.nan)
    if has_emb.any():
        v[has_emb] = isotonic[lvl].predict(models[lvl].predict_proba(Xap[has_emb])[:, 1])
    applied = applied.with_columns(pl.Series("p_" + lvl, v, nan_to_null=True))
print(f"scored {int(has_emb.sum()):,} of {applied.height:,} applied proteins")
print(applied.filter(pl.col("d1").is_not_null()).select(["p_specific", "p_L1", "p_category"]).describe())
assert applied.filter(pl.col("d1").is_null())["p_specific"].is_null().all(), \
    "unembedded proteins were assigned a probability"

scored 541,007 of 541,199 applied proteins
shape: (9, 4)
┌────────────┬────────────┬──────────┬────────────┐
│ statistic  ┆ p_specific ┆ p_L1     ┆ p_category │
│ ---        ┆ ---        ┆ ---      ┆ ---        │
│ str        ┆ f64        ┆ f64      ┆ f64        │
╞════════════╪════════════╪══════════╪════════════╡
│ count      ┆ 541007.0   ┆ 541007.0 ┆ 541007.0   │
│ null_count ┆ 0.0        ┆ 0.0      ┆ 0.0        │
│ mean       ┆ 0.645663   ┆ 0.810054 ┆ 0.894501   │
│ std        ┆ 0.350256   ┆ 0.226642 ┆ 0.127793   │
│ min        ┆ 0.0        ┆ 0.170213 ┆ 0.0        │
│ 25%        ┆ 0.315789   ┆ 0.664894 ┆ 0.827448   │
│ 50%        ┆ 0.77095    ┆ 0.913636 ┆ 0.95302    │
│ 75%        ┆ 0.981164   ┆ 0.994687 ┆ 0.994056   │
│ max        ┆ 1.0        ┆ 1.0      ┆ 1.0        │
└────────────┴────────────┴──────────┴────────────┘


### 12.1 Tier-1 reach on the applied set

The applied clustering was run on a single shared universe of the labeled training proteins and the applied AVGs together, so a cluster can contain both and a vote can cross from one to the other. Reach here is the fraction of applied AVGs that share a cluster with at least one labeled training protein.

In [52]:
_uf = [SEQCLUST / CLU_APPLIED.format(p=p) for p in PCTS]
_uf = [f for f in _uf if f.exists()]
assert _uf, "no applied clustering output is present"
uni = load_clu(_uf[0]).select("member").unique()
print(f"universe read from {_uf[0].name}: {uni.height:,} distinct members")
applied = applied.with_columns(
    pl.col("Protein").is_in(uni["member"].implode()).alias("cluster_reachable"))
UNIVERSE_N = int(uni.height)
n_unclustered = int((~applied["cluster_reachable"]).sum())
n_collide = int(applied.join(reference.select("nn_ref_id"), left_on="Protein",
                         right_on="nn_ref_id", how="inner").height)
print(f"shared clustered universe: {UNIVERSE_N:,} sequences")
print(f"applied proteins present in it: {applied.height - n_unclustered:,} of {applied.height:,}")
print(f"absent from it: {n_unclustered:,}")
print(f"id collisions between the training and applied id spaces: {n_collide}")
assert n_collide == 0, "training and applied protein ids overlap, cluster membership is ambiguous"
assert UNIVERSE_N == reference.height + applied.height - n_unclustered, (
    f"universe {UNIVERSE_N:,} does not reconcile with "
    f"{reference.height:,} training + {applied.height - n_unclustered:,} clustered applied")
print(applied.filter(~pl.col("cluster_reachable")).select(["Protein", "dataset", "homology_invisible"]))

universe read from clu_all_id30.tsv: 1,500,646 distinct members


shared clustered universe: 1,500,646 sequences
applied proteins present in it: 541,197 of 541,199
absent from it: 2
id collisions between the training and applied id spaces: 0
shape: (2, 3)
┌──────────────────────────────────────────────────────────────────┬─────────┬────────────────────┐
│ Protein                                                          ┆ dataset ┆ homology_invisible │
│ ---                                                              ┆ ---     ┆ ---                │
│ str                                                              ┆ str     ┆ bool               │
╞══════════════════════════════════════════════════════════════════╪═════════╪════════════════════╡
│ IMGVR_UViG_3300000289_000048|3300000289|EM283_1053392_9          ┆ gut     ┆ false              │
│ IMGVR_UViG_3300000294_000142|3300000294|EM326_1031029|1-32714_59 ┆ gut     ┆ false              │
└──────────────────────────────────────────────────────────────────┴─────────┴────────────────────┘


Proteins absent from the clustered universe are also absent from the biome-filtered protein FASTAs used as clustering input. Tier 1 cannot reach them, but they stay in the table and tier 2 still applies.

Attach the cluster vote at every identity threshold and every granularity.

In [53]:
for p in PCTS:
    f = SEQCLUST / CLU_APPLIED.format(p=p)
    if not f.exists():
        print(f"  {f.name}: absent, thresholds using it will have no vote")
        continue
    cl = load_clu(f)
    for lvl in LEVELS:
        t = tier1(cl, applied, lvl)
        cols = [pl.col("vote_label").alias(f"c{p}_{lvl}")]
        if lvl == "specific":
            cols += [pl.col("cluster_id").alias(f"c{p}_cluster"),
                     pl.col("vote_frac").alias(f"c{p}_frac"),
                     pl.col("n_labeled").alias(f"c{p}_nlab")]
        applied = applied.join(t.select(["Protein"] + cols), on="Protein", how="left")
    del cl
rows = []
for p in PCTS:
    if f"c{p}_specific" not in applied.columns:
        continue
    for lvl in LEVELS:
        for scope, g in [("all", applied), ("sequence_similarity_invisible",
                                        applied.filter(pl.col("homology_invisible")))]:
            rows.append({"min_seq_id": p / 100, "level": lvl, "scope": scope,
                         "n": g.height,
                         "n_voted": int(g[f"c{p}_{lvl}"].is_not_null().sum()),
                         "reach": float(g[f"c{p}_{lvl}"].is_not_null().mean())})
areach = pl.DataFrame(rows)
print(areach.pivot(on=["scope", "level"], index="min_seq_id", values="reach"))

shape: (4, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ min_seq_id ┆ {"all","spec ┆ {"sequence_s ┆ {"all","L1" ┆ {"sequence_ ┆ {"all","cat ┆ {"sequence_ │
│ ---        ┆ ific"}       ┆ imilarity_in ┆ }           ┆ similarity_ ┆ egory"}     ┆ similarity_ │
│ f64        ┆ ---          ┆ visible","sp ┆ ---         ┆ invisible", ┆ ---         ┆ invisible", │
│            ┆ f64          ┆ ecific"}     ┆ f64         ┆ "L1"}       ┆ f64         ┆ "category"} │
│            ┆              ┆ ---          ┆             ┆ ---         ┆             ┆ ---         │
│            ┆              ┆ f64          ┆             ┆ f64         ┆             ┆ f64         │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 0.3        ┆ 0.717503     ┆ 0.315369     ┆ 0.723228    ┆ 0.319915    ┆ 0.724074    ┆ 0.320887    │
│ 0.5        ┆ 0.498294     ┆ 0.12419      ┆ 0.501446    ┆ 0.125107    ┆ 0.50

### 12.2 The tiered assignment applied

The rule from section 6, unchanged, at every precision target. Each protein also records what each tier alone would have said, so the contribution of each tier is auditable per row.

In [54]:
for t in TARGETS:
    lab, lvl, src = assign(applied, t)
    tag = f"{int(round(t*100))}"
    applied = applied.with_columns(lab.alias(f"label_{tag}"), lvl.alias(f"level_{tag}"),
                           src.alias(f"source_{tag}"))
for nm, meth in [("tier1", "cluster_only"), ("tier2", "tier2_deployed")]:
    lab, lvl, _ = _fold(variant_terms(PRIMARY_TARGET, meth))
    applied = applied.with_columns(lab.alias(f"{nm}_label"), lvl.alias(f"{nm}_level"))
applied = applied.with_columns(pl.col(f"label_{PT}").alias("final_label"),
                       pl.col(f"level_{PT}").alias("final_level"),
                       pl.col(f"source_{PT}").alias("label_source"),
                       pl.lit(PRIMARY_TARGET).alias("precision_target"))
assert applied.filter(pl.col("final_level").is_null()).height == 0, "null final_level remains"
assert applied.filter(pl.col("label_source").is_null()).height == 0, "null label_source remains"
print(f"applied proteins assigned at target {PRIMARY_TARGET:.2f}: "
      f"{applied.filter(pl.col('final_level') != 'unassigned').height:,} of {applied.height:,}")
print(applied.group_by(["final_level", "label_source"]).agg(pl.len().alias("n"))
         .with_columns((100 * pl.col("n") / applied.height).round(2).alias("pct"))
         .sort(["final_level", "label_source"]))

applied proteins assigned at target 0.90: 463,898 of 541,199
shape: (7, 4)
┌─────────────┬──────────────┬────────┬───────┐
│ final_level ┆ label_source ┆ n      ┆ pct   │
│ ---         ┆ ---          ┆ ---    ┆ ---   │
│ str         ┆ str          ┆ u64    ┆ f64   │
╞═════════════╪══════════════╪════════╪═══════╡
│ L1          ┆ cluster      ┆ 66493  ┆ 12.29 │
│ L1          ┆ embedding    ┆ 13723  ┆ 2.54  │
│ category    ┆ cluster      ┆ 268    ┆ 0.05  │
│ category    ┆ embedding    ┆ 9110   ┆ 1.68  │
│ specific    ┆ cluster      ┆ 269676 ┆ 49.83 │
│ specific    ┆ embedding    ┆ 104628 ┆ 19.33 │
│ unassigned  ┆ none         ┆ 77301  ┆ 14.28 │
└─────────────┴──────────────┴────────┴───────┘


The same breakdown for the sequence-similarity invisible AVGs, written for functional_propagation_figures.Rmd.

In [55]:
rows = []
for scope, g in [("all", applied), ("sequence_similarity_invisible", applied.filter(pl.col("homology_invisible")))]:
    for t in TARGETS:
        tag = f"{int(round(t*100))}"
        for lvl in LEVELS + ["unassigned"]:
            for src in ["cluster", "embedding", "none"]:
                s = g.filter((pl.col(f"level_{tag}") == lvl) & (pl.col(f"source_{tag}") == src))
                if not s.height:
                    continue
                rows.append({"scope": scope, "target": t, "level": lvl, "label_source": src,
                             "n": s.height, "n_scope": g.height, "frac": s.height / g.height})
abrk = pl.DataFrame(rows)
print(abrk.filter((pl.col("target") == PRIMARY_TARGET) & (pl.col("scope") == "sequence_similarity_invisible")))
abrk.write_parquet(OUT / "fig_tier_assignment_breakdown.parquet")

shape: (7, 7)
┌───────────────────────────────┬────────┬────────────┬──────────────┬───────┬─────────┬──────────┐
│ scope                         ┆ target ┆ level      ┆ label_source ┆ n     ┆ n_scope ┆ frac     │
│ ---                           ┆ ---    ┆ ---        ┆ ---          ┆ ---   ┆ ---     ┆ ---      │
│ str                           ┆ f64    ┆ str        ┆ str          ┆ i64   ┆ i64     ┆ f64      │
╞═══════════════════════════════╪════════╪════════════╪══════════════╪═══════╪═════════╪══════════╡
│ sequence_similarity_invisible ┆ 0.9    ┆ specific   ┆ cluster      ┆ 13550 ┆ 109107  ┆ 0.12419  │
│ sequence_similarity_invisible ┆ 0.9    ┆ specific   ┆ embedding    ┆ 31842 ┆ 109107  ┆ 0.291842 │
│ sequence_similarity_invisible ┆ 0.9    ┆ L1         ┆ cluster      ┆ 12935 ┆ 109107  ┆ 0.118553 │
│ sequence_similarity_invisible ┆ 0.9    ┆ L1         ┆ embedding    ┆ 7716  ┆ 109107  ┆ 0.07072  │
│ sequence_similarity_invisible ┆ 0.9    ┆ category   ┆ cluster      ┆ 59    ┆ 109107 

Assignment depth by dataset and by sequence-similarity invisible status, and sensitivity to the target.

In [56]:
print("by dataset:")
print(applied.group_by(["dataset", "final_level"]).agg(pl.len().alias("n")).sort(["dataset", "n"], descending=[False, True]))
print("\nby sequence-similarity invisible status:")
print(applied.group_by(["homology_invisible", "final_level"]).agg(pl.len().alias("n"))
        .sort(["homology_invisible", "n"], descending=[False, True]))
print("\nsensitivity to the precision target:")
sens = pl.concat([applied.group_by(f"level_{int(t*100)}").agg(pl.len().alias("n"))
                    .rename({f"level_{int(t*100)}": "level"})
                    .with_columns(pl.lit(t).alias("target")) for t in TARGETS])
print(sens.pivot(values="n", index="level", on="target").fill_null(0))

by dataset:
shape: (8, 3)
┌─────────┬─────────────┬────────┐
│ dataset ┆ final_level ┆ n      │
│ ---     ┆ ---         ┆ ---    │
│ str     ┆ str         ┆ u64    │
╞═════════╪═════════════╪════════╡
│ gut     ┆ specific    ┆ 206503 │
│ gut     ┆ L1          ┆ 22619  │
│ gut     ┆ unassigned  ┆ 20497  │
│ gut     ┆ category    ┆ 2413   │
│ soil    ┆ specific    ┆ 167801 │
│ soil    ┆ L1          ┆ 57597  │
│ soil    ┆ unassigned  ┆ 56804  │
│ soil    ┆ category    ┆ 6965   │
└─────────┴─────────────┴────────┘

by sequence-similarity invisible status:
shape: (8, 3)
┌────────────────────┬─────────────┬────────┐
│ homology_invisible ┆ final_level ┆ n      │
│ ---                ┆ ---         ┆ ---    │
│ bool               ┆ str         ┆ u64    │
╞════════════════════╪═════════════╪════════╡
│ false              ┆ specific    ┆ 328912 │
│ false              ┆ L1          ┆ 59565  │
│ false              ┆ unassigned  ┆ 39344  │
│ false              ┆ category    ┆ 4271   │
│ true        

Mean information content of the assigned labels, against category-only assignment.

In [57]:
tot = reference.height
ic_specific = {k: -np.log2(v / tot) for k, v in
            zip(freq["ref_specific"].to_list(), freq["ref_n"].to_list())}
l1n = reference.group_by("ref_L1").len()
ic_L1 = {k: -np.log2(v / tot) for k, v in zip(l1n["ref_L1"].to_list(), l1n["len"].to_list())}
catn = reference.group_by("ref_category").len()
ic_cat = {k: -np.log2(v / tot) for k, v in zip(catn["ref_category"].to_list(), catn["len"].to_list())}
IC = {**ic_cat, **ic_L1, **ic_specific}
bits = applied.select([
    pl.col("final_label").replace_strict(IC, default=None).alias("bits_adaptive"),
    pl.col("raw_category").replace_strict(ic_cat, default=None).alias("bits_category_only")])
mean_ad = float(bits["bits_adaptive"].fill_null(0.0).mean())
mean_ct = float(bits["bits_category_only"].fill_null(0.0).mean())
print(f"mean information content per applied protein:")
print(f"  adaptive granularity : {mean_ad:.3f} bits")
print(f"  category only        : {mean_ct:.3f} bits")
print(f"  gain                 : {mean_ad - mean_ct:+.3f} bits")

mean information content per applied protein:
  adaptive granularity : 7.305 bits
  category only        : 1.996 bits
  gain                 : +5.309 bits


Hierarchy consistency by label source. Embedding-sourced labels are checked against the neighbor's `raw_L1` and `raw_category`. Cluster-sourced labels are checked against the reference vocabulary, because they have no relation to `raw_L1`.

In [58]:
emb = applied.filter(pl.col("label_source") == "embedding")
e_spec = emb.filter(pl.col("final_level") == "specific").with_columns(
    pl.col("final_label").replace_strict(EX2L1, default=None).alias("implied_L1"))
bad_emb_L1 = e_spec.filter(pl.col("implied_L1") != pl.col("raw_L1")).height
bad_emb_cat = emb.filter((pl.col("final_level") == "category")
                         & (pl.col("final_label") != pl.col("raw_category"))).height

VOCAB = {"specific": reference["ref_specific"].drop_nulls().unique().to_list(),
         "L1": reference["ref_L1"].drop_nulls().unique().to_list(),
         "category": reference["ref_category"].drop_nulls().unique().to_list()}
clu = applied.filter(pl.col("label_source") == "cluster")
bad_vocab = {lvl: clu.filter((pl.col("final_level") == lvl)
                             & ~pl.col("final_label").is_in(VOCAB[lvl])).height
             for lvl in LEVELS}
c_spec = clu.filter(pl.col("final_level") == "specific").with_columns(
    pl.col("final_label").replace_strict(EX2L1, default=None).alias("implied_L1"))
no_parent = c_spec.filter(pl.col("implied_L1").is_null()).height
differs = c_spec.filter(pl.col("raw_L1").is_not_null()
                        & (pl.col("implied_L1") != pl.col("raw_L1"))).height

print(f"embedding-sourced specific labels whose implied L1 disagrees with raw_L1: {bad_emb_L1}")
print(f"embedding-sourced category labels disagreeing with raw_category: {bad_emb_cat}")
print("cluster-sourced labels outside the reference vocabulary: "
      + ", ".join(f"{l}={bad_vocab[l]}" for l in LEVELS))
print(f"cluster-sourced specific labels with no L1 parent: {no_parent}")
print(f"cluster-sourced specific labels whose implied L1 differs from the embedding's raw_L1: {differs:,}"
      f" of {c_spec.filter(pl.col('raw_L1').is_not_null()).height:,} with an embedding")
assert bad_emb_L1 == 0 and bad_emb_cat == 0, "embedding tier hierarchy consistency violated"
assert sum(bad_vocab.values()) == 0, "cluster tier emitted a label outside the reference vocabulary"
assert no_parent == 0, "cluster tier emitted a specific label with no L1 parent"

embedding-sourced specific labels whose implied L1 disagrees with raw_L1: 0
embedding-sourced category labels disagreeing with raw_category: 0
cluster-sourced labels outside the reference vocabulary: specific=0, L1=0, category=0
cluster-sourced specific labels with no L1 parent: 0
cluster-sourced specific labels whose implied L1 differs from the embedding's raw_L1: 34,598 of 269,575 with an embedding


Cluster-sourced labels whose implied L1 differs from the embedding neighbor's L1 are counted rather than asserted, because the two label sources are independent.

## 13. Orthogonal evidence: DefenseFinder

DefenseFinder agreement is the only estimate measured on applied proteins rather than held-out reference proteins. The scan runs here because it depends only on the sequence-similarity invisible set defined in this notebook. Structural evidence is computed in novel_avgs.ipynb, so `consensus_structure` is null in this table and `consensus_tier` reaches at most `single_evidence`. DefenseFinder is scored at L1 and category only, because its vocabulary names system roles rather than molecular functions.

Scan every sequence-similarity invisible AVG against all DefenseFinder and AntiDefenseFinder profiles.

In [60]:
DF_DB_DIR = Path("/storage2/scratch/kosmopoulos/databases/defensefinder/defense-finder-models")
DF_PROFILE_DIR = DF_DB_DIR / "profiles"
HITS, RECOV = DFDIR / "hi_defensefinder_hits.parquet", DFDIR / "hi_df_recovery.parquet"
if HITS.exists() and RECOV.exists():
    print("DefenseFinder scan cached")
else:
    import pyhmmer, glob
    t0 = time.time()
    def hmm_ga(p):
        ga = None
        with open(p) as f:
            for raw in f:
                if raw.startswith("GA"):
                    pr = raw.replace(";", " ").split()
                    ga = float(pr[1]) if len(pr) > 1 else None
                elif raw.startswith("HMM "):
                    break
        return ga
    prof_files = sorted(glob.glob(str(DF_PROFILE_DIR / "*.hmm")))
    ga_map = {Path(p).stem: hmm_ga(Path(p)) for p in prof_files}
    rows = []
    for line in (DF_DB_DIR / "Liste_hmm_system.md").read_text().splitlines():
        if line.strip().startswith("|"):
            c = [x.strip() for x in line.strip().strip("|").split("|")]
            if len(c) >= 5:
                rows.append(c[:5])
    meta_df = pl.DataFrame({rows[0][i]: [r[i] for r in rows[2:]] for i in range(5)})
    meta_ga = {g: (float(v) if v not in ("", None) else None)
               for g, v in zip(meta_df["gene_name"], meta_df["GA_cut"])}
    meta_sys = {g: s for g, s in zip(meta_df["gene_name"], meta_df["System"])}
    ga = {g: (ga_map.get(g) if ga_map.get(g) is not None else meta_ga.get(g)) for g in ga_map}
    system = {g: (meta_sys.get(g) or g.split("__")[0]) for g in ga_map}
    alph = pyhmmer.easel.Alphabet.amino()
    with pyhmmer.easel.SequenceFile(str(FAA), digital=True, alphabet=alph) as sf:
        pseqs = sf.read_block()
    hmms = []
    for p in prof_files:
        with pyhmmer.plan7.HMMFile(p) as hf:
            hmms.append(hf.read())
    dec = lambda x: x.decode() if isinstance(x, (bytes, bytearray)) else x
    hr = []
    for h in pyhmmer.hmmsearch(hmms, pseqs, E=1e-3, cpus=64):
        gn = dec(h.query.name)
        for hit in h:
            if hit.included or hit.evalue <= 1e-3:
                hr.append((dec(hit.name), gn, float(hit.score), float(hit.evalue)))
    hits = pl.DataFrame(hr, schema=["Protein", "profile", "bitscore", "evalue"], orient="row")
    hits = hits.with_columns(pl.col("profile").replace_strict(ga, default=None).alias("ga"),
                             pl.col("profile").replace_strict(system, default=None).alias("System"))
    hits = hits.with_columns((pl.col("ga").is_not_null() & (pl.col("bitscore") >= pl.col("ga"))).alias("ga_pass"))
    hits.write_parquet(HITS)
    gap = set(hits.filter(pl.col("ga_pass"))["Protein"].to_list())
    rlx = set(hits["Protein"].to_list())
    hi_ids.with_columns(
        pl.when(pl.col("Protein").is_in(list(gap))).then(pl.lit("recovered_GA"))
         .when(pl.col("Protein").is_in(list(rlx))).then(pl.lit("recovered_relaxed_only"))
         .otherwise(pl.lit("no_hit_P_novel")).alias("df_recovery")).write_parquet(RECOV)
    print(f"DefenseFinder scan done ({time.time()-t0:.0f}s)")
df_hits = pl.read_parquet(HITS)
df_rec = pl.read_parquet(RECOV)
print(f"hits {df_hits.height:,} on {df_hits['Protein'].n_unique():,} proteins; "
      f"GA-passing {df_hits.filter(pl.col('ga_pass'))['Protein'].n_unique():,}")
print(df_rec.group_by("df_recovery").agg(pl.len().alias("n")).sort("n", descending=True))

DefenseFinder scan cached


hits 95,987 on 23,297 proteins; GA-passing 14,778
shape: (3, 2)
┌────────────────────────┬───────┐
│ df_recovery            ┆ n     │
│ ---                    ┆ ---   │
│ str                    ┆ u64   │
╞════════════════════════╪═══════╡
│ no_hit_P_novel         ┆ 85810 │
│ recovered_GA           ┆ 14778 │
│ recovered_relaxed_only ┆ 8519  │
└────────────────────────┴───────┘


Propagated category and L1 for the DefenseFinder-positive proteins.

In [61]:
defensefinder_positive = (df_rec.filter(pl.col("df_recovery") != "no_hit_P_novel").select("Protein")
        .join(applied.select(["Protein", "raw_category", "raw_L1", "final_label", "final_level",
                          "label_source", "p_category", "p_L1", "p_specific"]), on="Protein", how="left"))
print(f"DefenseFinder-positive sequence-similarity invisible AVGs: {defensefinder_positive.height:,}")
L12CAT = (reference.group_by("ref_L1").agg(pl.col("ref_category").mode().sort().first())
            .to_dict(as_series=False))
L12CAT = dict(zip(L12CAT["ref_L1"], L12CAT["ref_category"]))
defensefinder_positive = defensefinder_positive.with_columns(
    pl.when(pl.col("final_level") == "category").then(pl.col("final_label"))
      .when(pl.col("final_level") == "L1")
        .then(pl.col("final_label").replace_strict(L12CAT, default=None))
      .when(pl.col("final_level") == "specific")
        .then(pl.col("final_label").replace_strict(EX2CAT, default=None))
      .otherwise(None).alias("emitted_category"))
dfp_lab = defensefinder_positive.filter(pl.col("final_level") != "unassigned")
DF_LABELLED = dfp_lab.height
print(f"of these, carrying a propagated label: {DF_LABELLED:,}")
print(f"emitted category unresolved among labeled: "
      f"{int(dfp_lab.filter(pl.col('emitted_category').is_null()).height)}")
print(dfp_lab.group_by("label_source").agg(pl.len().alias("n")).sort("n", descending=True))
cat_break = (dfp_lab.group_by("emitted_category").agg(pl.len().alias("n"))
               .with_columns((100 * pl.col("n") / DF_LABELLED).round(1).alias("pct"))
               .sort("n", descending=True))
print(cat_break)
print("\ntop propagated L1 among DefenseFinder-positive:")
print(defensefinder_positive.group_by("raw_L1").agg(pl.len().alias("n")).sort("n", descending=True).head(10))
print("\nassignment depth among DefenseFinder-positive:")
print(defensefinder_positive.group_by("final_level").agg(pl.len().alias("n")).sort("n", descending=True))

DefenseFinder-positive sequence-similarity invisible AVGs: 23,297
of these, carrying a propagated label: 15,164
emitted category unresolved among labeled: 0
shape: (2, 2)
┌──────────────┬──────┐
│ label_source ┆ n    │
│ ---          ┆ ---  │
│ str          ┆ u64  │
╞══════════════╪══════╡
│ cluster      ┆ 9380 │
│ embedding    ┆ 5784 │
└──────────────┴──────┘
shape: (3, 3)
┌──────────────────┬───────┬──────┐
│ emitted_category ┆ n     ┆ pct  │
│ ---              ┆ ---   ┆ ---  │
│ str              ┆ u64   ┆ f64  │
╞══════════════════╪═══════╪══════╡
│ regulatory       ┆ 10569 ┆ 69.7 │
│ physiological    ┆ 2563  ┆ 16.9 │
│ metabolic        ┆ 2032  ┆ 13.4 │
└──────────────────┴───────┴──────┘

top propagated L1 among DefenseFinder-positive:
shape: (10, 2)
┌─────────────────────────────────────────────┬───────┐
│ raw_L1                                      ┆ n     │
│ ---                                         ┆ ---   │
│ str                                         ┆ u64   │
╞══════════

Is regulatory over-predicted globally, or only on defense proteins?

In [62]:
rep_true = report.group_by("true_category").agg(pl.len().alias("true_n"))
rep_pred = report.group_by("raw_category").agg(pl.len().alias("pred_n")).rename({"raw_category": "true_category"})
bal = (rep_true.join(rep_pred, on="true_category", how="full", coalesce=True).fill_null(0)
        .with_columns((pl.col("pred_n") / pl.col("true_n")).round(3).alias("pred_over_true")))
print("held-out reporting slice, predicted vs true category counts:")
print(bal)
reg_ratio = float(bal.filter(pl.col("true_category") == "regulatory")["pred_over_true"][0])
reg_rows = cat_break.filter(pl.col("emitted_category") == "regulatory")
df_reg = float(reg_rows["pct"][0]) if reg_rows.height else float("nan")
print(f"\nregulatory predicted/true ratio on held-out data: {reg_ratio:.3f}")
print(f"regulatory share of DefenseFinder-positive applied proteins: {df_reg:.1f}%")

held-out reporting slice, predicted vs true category counts:
shape: (3, 4)
┌───────────────┬────────┬────────┬────────────────┐
│ true_category ┆ true_n ┆ pred_n ┆ pred_over_true │
│ ---           ┆ ---    ┆ ---    ┆ ---            │
│ str           ┆ u64    ┆ u64    ┆ f64            │
╞═══════════════╪════════╪════════╪════════════════╡
│ regulatory    ┆ 10657  ┆ 10630  ┆ 0.997          │
│ metabolic     ┆ 27818  ┆ 27793  ┆ 0.999          │
│ physiological ┆ 3482   ┆ 3534   ┆ 1.015          │
└───────────────┴────────┴────────┴────────────────┘

regulatory predicted/true ratio on held-out data: 0.997
regulatory share of DefenseFinder-positive applied proteins: 69.7%


## 14. Sensitivity of the tiered rule to the precision target

Assignment rate and realized precision across precision targets, by identity to the training set and by embedding distance.

Assignment rate and realized precision of the tiered rule across precision targets, per granularity, on the held-out reporting slice. Realized precision is measured against the emitted label at the granularity it was emitted, so a tier-1 label is scored against the truth at that granularity rather than against the embedding neighbor's label.

In [63]:
SWEEP = TARGETS_R
rows = []
for t in SWEEP:
    lab, lvl, src = assign(report, t)
    s = score_assignment(report.with_columns(lab.alias("_l"), lvl.alias("_v"), src.alias("_s")),
                         "_l", "_v")
    for level in LEVELS:
        sel = s.filter(pl.col("_v") == level)
        if not sel.height:
            continue
        rows.append({"target": t, "level": level, "n": sel.height,
                     "rate_heldout": sel.height / report.height,
                     "realized_precision": float(sel["_ok"].mean())})
    unas = s.filter(pl.col("_v") == "unassigned").height
    rows.append({"target": t, "level": "unassigned", "n": unas,
                 "rate_heldout": unas / report.height, "realized_precision": None})
sweep = pl.DataFrame(rows)
print(sweep.filter(pl.col("level") != "unassigned").pivot(
    values="realized_precision", index="target", on="level"))
print()
print(sweep.pivot(values="rate_heldout", index="target", on="level").fill_null(0))

shape: (9, 4)
┌────────┬──────────┬──────────┬──────────┐
│ target ┆ specific ┆ L1       ┆ category │
│ ---    ┆ ---      ┆ ---      ┆ ---      │
│ f64    ┆ f64      ┆ f64      ┆ f64      │
╞════════╪══════════╪══════════╪══════════╡
│ 0.5    ┆ 0.846456 ┆ 0.435945 ┆ 0.692691 │
│ 0.6    ┆ 0.854102 ┆ 0.561525 ┆ 0.730448 │
│ 0.7    ┆ 0.858041 ┆ 0.681245 ┆ 0.737224 │
│ 0.75   ┆ 0.859551 ┆ 0.757127 ┆ 0.747157 │
│ 0.8    ┆ 0.861086 ┆ 0.802581 ┆ 0.782711 │
│ 0.85   ┆ 0.861731 ┆ 0.845455 ┆ 0.812919 │
│ 0.9    ┆ 0.945314 ┆ 0.92452  ┆ 0.849876 │
│ 0.95   ┆ 0.950107 ┆ 0.893599 ┆ 0.98918  │
│ 0.99   ┆ 0.987269 ┆ 0.983333 ┆ 0.993796 │
└────────┴──────────┴──────────┴──────────┘

shape: (9, 5)
┌────────┬──────────┬──────────┬──────────┬────────────┐
│ target ┆ specific ┆ L1       ┆ category ┆ unassigned │
│ ---    ┆ ---      ┆ ---      ┆ ---      ┆ ---        │
│ f64    ┆ f64      ┆ f64      ┆ f64      ┆ f64        │
╞════════╪══════════╪══════════╪══════════╪════════════╡
│ 0.5    ┆ 0.92142  ┆ 0.06

Applied-set assignment depth across the same targets, for the sequence-similarity invisible subset.

In [64]:
rows = []
hi_ap = applied.filter(pl.col("homology_invisible"))
for t in SWEEP:
    lab, lvl, _ = assign(hi_ap, t)
    s = hi_ap.with_columns(lvl.alias("_v"))
    for level in LEVELS + ["unassigned"]:
        n = s.filter(pl.col("_v") == level).height
        rows.append({"target": t, "level": level, "n": n, "pct": 100 * n / s.height})
hi_sweep = pl.DataFrame(rows)
print(hi_sweep.pivot(values="pct", index="target", on="level").fill_null(0))
hi_sweep.write_parquet(OUT / "fig_target_sweep_applied.parquet")
sweep.write_parquet(OUT / "fig_target_sweep_heldout.parquet")

shape: (9, 5)
┌────────┬───────────┬───────────┬───────────┬────────────┐
│ target ┆ specific  ┆ L1        ┆ category  ┆ unassigned │
│ ---    ┆ ---       ┆ ---       ┆ ---       ┆ ---        │
│ f64    ┆ f64       ┆ f64       ┆ f64       ┆ f64        │
╞════════╪═══════════╪═══════════╪═══════════╪════════════╡
│ 0.5    ┆ 79.891299 ┆ 18.179402 ┆ 1.912801  ┆ 0.016498   │
│ 0.6    ┆ 72.707526 ┆ 19.726507 ┆ 7.543971  ┆ 0.021997   │
│ 0.7    ┆ 65.496256 ┆ 20.631124 ┆ 13.850624 ┆ 0.021997   │
│ 0.75   ┆ 61.306791 ┆ 19.332398 ┆ 19.338814 ┆ 0.021997   │
│ 0.8    ┆ 56.59032  ┆ 19.172005 ┆ 6.980304  ┆ 17.257371  │
│ 0.85   ┆ 53.454865 ┆ 16.725783 ┆ 5.071169  ┆ 24.748183  │
│ 0.9    ┆ 41.603197 ┆ 18.927292 ┆ 4.680726  ┆ 34.788785  │
│ 0.95   ┆ 32.531368 ┆ 16.109874 ┆ 10.910391 ┆ 40.448367  │
│ 0.99   ┆ 21.773122 ┆ 8.211206  ┆ 17.975015 ┆ 52.040657  │
└────────┴───────────┴───────────┴───────────┴────────────┘


Realized precision jointly by identity to the training set and by embedding distance.

In [65]:
ident_bands = [(0.0, 0.3), (0.3, 0.5), (0.5, 0.7), (0.7, 0.9), (0.9, 1.01)]
d1q = report["d1"].quantile
dcuts = [0.0] + [float(d1q(q)) for q in (0.2, 0.4, 0.6, 0.8)] + [float(report["d1"].max()) + 1]
rows = []
for lo, hi in ident_bands:
    for k in range(len(dcuts) - 1):
        s = report.filter((pl.col("max_ident_to_train") >= lo) & (pl.col("max_ident_to_train") < hi)
                      & (pl.col("d1") >= dcuts[k]) & (pl.col("d1") < dcuts[k + 1]))
        if s.height < 50:
            continue
        rows.append({"identity_band": f"{lo:.0%}-{hi:.0%}",
                     "d1_bin": f"Q{k+1}", "d1_lo": round(dcuts[k], 1), "d1_hi": round(dcuts[k + 1], 1),
                     "n": s.height,
                     "specific": round(float(s["y_specific"].mean()), 4),
                     "L1": round(float(s["y_L1"].mean()), 4),
                     "category": round(float(s["y_category"].mean()), 4)})
joint = pl.DataFrame(rows)
print(joint.pivot(values="specific", index="identity_band", on="d1_bin"))
joint.write_parquet(OUT / "fig_identity_by_distance.parquet")

shape: (5, 6)
┌───────────────┬────────┬────────┬────────┬────────┬────────┐
│ identity_band ┆ Q3     ┆ Q4     ┆ Q5     ┆ Q1     ┆ Q2     │
│ ---           ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ str           ┆ f64    ┆ f64    ┆ f64    ┆ f64    ┆ f64    │
╞═══════════════╪════════╪════════╪════════╪════════╪════════╡
│ 0%-30%        ┆ 0.3103 ┆ 0.336  ┆ 0.2031 ┆ null   ┆ null   │
│ 30%-50%       ┆ 0.3069 ┆ 0.3139 ┆ 0.236  ┆ 0.423  ┆ 0.3508 │
│ 50%-70%       ┆ 0.3203 ┆ 0.2994 ┆ 0.2476 ┆ 0.4143 ┆ 0.3595 │
│ 70%-90%       ┆ 0.4066 ┆ 0.37   ┆ 0.2836 ┆ 0.5321 ┆ 0.4478 │
│ 90%-101%      ┆ 0.5586 ┆ 0.4863 ┆ 0.3774 ┆ 0.724  ┆ 0.5989 │
└───────────────┴────────┴────────┴────────┴────────┴────────┘


Are the emitted labels internally faithful, separate from how many are emitted?

In [66]:
rows = []
for t in [0.70, 0.80, 0.90, 0.95]:
    a = report.filter(pl.col("p_specific") >= t)
    if a.height < 100:
        continue
    tc = a.group_by("true_specific").len().rename({"len": "true_n"})
    pc = a.group_by("raw_specific").len().rename({"raw_specific": "true_specific", "len": "pred_n"})
    f = (tc.join(pc, on="true_specific", how="full", coalesce=True).fill_null(0)
           .filter(pl.col("true_n") >= 20)
           .with_columns(pl.col("true_specific").replace_strict(EX2CAT, default=None).alias("category"))
           .sort("true_specific"))
    if f.height < 3:
        continue
    sp_all = spear(f["true_n"].to_numpy(), f["pred_n"].to_numpy())
    m = f.filter((pl.col("true_n") > 0) & (pl.col("pred_n") > 0))
    slope = float(np.polyfit(np.log(m["true_n"].to_numpy().astype(float)),
                             np.log(m["pred_n"].to_numpy().astype(float)), 1)[0]) if m.height > 2 else float("nan")
    r = {"target": t, "n_assigned": a.height, "coverage": a.height / report.height,
         "realized_precision": float(a["y_specific"].mean()), "spearman_conditional": round(sp_all, 4),
         "log_log_slope": round(slope, 4), "n_labels": f.height}
    for cat in CAT3:
        s = f.filter(pl.col("category") == cat)
        if s.height < 3:
            continue
        t12 = set(s.sort(["true_n", "true_specific"], descending=[True, False]).head(12)["true_specific"].to_list())
        p12 = set(s.sort(["pred_n", "true_specific"], descending=[True, False]).head(12)["true_specific"].to_list())
        r[f"overlap12_{cat}"] = len(t12 & p12)
    rows.append(r)
cond = pl.DataFrame(rows)
print(cond)

shape: (4, 10)
┌────────┬────────────┬──────────┬────────────┬───┬──────────┬────────────┬────────────┬───────────┐
│ target ┆ n_assigned ┆ coverage ┆ realized_p ┆ … ┆ n_labels ┆ overlap12_ ┆ overlap12_ ┆ overlap12 │
│ ---    ┆ ---        ┆ ---      ┆ recision   ┆   ┆ ---      ┆ metabolic  ┆ physiologi ┆ _regulato │
│ f64    ┆ i64        ┆ f64      ┆ ---        ┆   ┆ i64      ┆ ---        ┆ cal        ┆ ry        │
│        ┆            ┆          ┆ f64        ┆   ┆          ┆ i64        ┆ ---        ┆ ---       │
│        ┆            ┆          ┆            ┆   ┆          ┆            ┆ i64        ┆ i64       │
╞════════╪════════════╪══════════╪════════════╪═══╪══════════╪════════════╪════════════╪═══════════╡
│ 0.7    ┆ 9519       ┆ 0.226875 ┆ 0.906818   ┆ … ┆ 88       ┆ 10         ┆ 11         ┆ 12        │
│ 0.8    ┆ 7674       ┆ 0.182902 ┆ 0.943445   ┆ … ┆ 73       ┆ 10         ┆ 11         ┆ 12        │
│ 0.9    ┆ 5556       ┆ 0.132421 ┆ 0.974982   ┆ … ┆ 55       ┆ 12         ┆ 

Write the method summary table.

In [67]:
summary_rows = []
for r in ladder.iter_rows(named=True):
    summary_rows.append({"level": r["level"], "quantity": "raw 1-NN micro precision", "value": r["micro"],
                         "note": "deployed rule, distinct held-out proteins"})
    summary_rows.append({"level": r["level"], "quantity": "raw 1-NN macro precision", "value": r["macro"],
                         "note": f"unweighted over {r['n_labels']:,} labels"})
for r in risk_coverage_summary.iter_rows(named=True):
    if r["ranking"] != "calibrated":
        continue
    summary_rows.append({"level": r["level"], "quantity": "coverage at 0.90 precision", "value": r["cov@0.90"],
                         "note": "calibrated ranking"})
    summary_rows.append({"level": r["level"], "quantity": "AURC", "value": r["aurc"], "note": "calibrated ranking"})
for r in risk_coverage_summary.filter(pl.col("ranking") == "d1_only").iter_rows(named=True):
    summary_rows.append({"level": r["level"], "quantity": "coverage at 0.90 precision", "value": r["cov@0.90"],
                         "note": "distance-only baseline"})
for r in ece.iter_rows(named=True):
    summary_rows.append({"level": r["level"], "quantity": "expected calibration error", "value": r["ece"],
                         "note": "reporting slice, 20 bins"})
for r in terciles.filter(pl.col("freq_tercile") == "rare").iter_rows(named=True):
    summary_rows.append({"level": r["level"], "quantity": "precision, rare label tercile", "value": r["precision"],
                         "note": f"n={r['n']:,}"})
supp = pl.DataFrame(summary_rows)
supp.write_parquet(OUT / "fig_method_summary.parquet")
print(supp)

shape: (21, 4)
┌──────────┬───────────────────────────────┬──────────┬───────────────────────────────────────────┐
│ level    ┆ quantity                      ┆ value    ┆ note                                      │
│ ---      ┆ ---                           ┆ ---      ┆ ---                                       │
│ str      ┆ str                           ┆ f64      ┆ str                                       │
╞══════════╪═══════════════════════════════╪══════════╪═══════════════════════════════════════════╡
│ category ┆ raw 1-NN micro precision      ┆ 0.8952   ┆ deployed rule, distinct held-out proteins │
│ category ┆ raw 1-NN macro precision      ┆ 0.8491   ┆ unweighted over 3 labels                  │
│ L1       ┆ raw 1-NN micro precision      ┆ 0.5866   ┆ deployed rule, distinct held-out proteins │
│ L1       ┆ raw 1-NN macro precision      ┆ 0.5595   ┆ unweighted over 28 labels                 │
│ specific ┆ raw 1-NN micro precision      ┆ 0.3813   ┆ deployed rule, distinct held-

Export the specific-function to L2 mapping for the biogeochemical category analysis in soil_gut_avgs.ipynb. L2 is not an assignment level (section 2). Every valid name-to-L2 pair is kept, because the curated tables let one function belong to several L2 categories and reference-annotated proteins are counted in all of them. `n_L2_parents` records the multiplicity.

In [68]:
l2_pairs = (cz_all.filter(pl.col("category_L2").is_not_null())
            .select(["name", "category_L1", "category_L2"]).unique())
l2_n = (l2_pairs.group_by("name").agg(pl.col("category_L2").n_unique().alias("n_L2_parents")))
sp2l2 = (l2_pairs.join(l2_n, on="name", how="left").rename({"name": "specific"})
         .sort(["specific", "category_L1", "category_L2"]))
sp2l2.write_parquet(OUT / "fig_specific_to_L2.parquet")
print(f"specific-to-L2 pairs: {sp2l2.height:,} over {sp2l2['specific'].n_unique():,} names")
print(f"  names with one L2      : {l2_n.filter(pl.col('n_L2_parents') == 1).height:,}")
print(f"  names with more than one: {l2_n.filter(pl.col('n_L2_parents') > 1).height:,}")
print(f"  distinct L2 categories : {sp2l2['category_L2'].n_unique():,}")

specific-to-L2 pairs: 21,654 over 15,733 names
  names with one L2      : 12,612
  names with more than one: 3,121
  distinct L2 categories : 235


Export the reference label-frequency terciles, which downstream notebooks use to set count floors.

In [69]:
freq.write_parquet(OUT / "fig_label_frequency.parquet")
print(f"wrote fig_label_frequency.parquet ({freq.height:,} labels)")
print(freq.group_by("freq_tercile").agg(pl.len().alias("labels"),
      pl.col("ref_n").min().alias("min_ref_n"), pl.col("ref_n").max().alias("max_ref_n")))

wrote fig_label_frequency.parquet (9,906 labels)
shape: (3, 4)
┌──────────────┬────────┬───────────┬───────────┐
│ freq_tercile ┆ labels ┆ min_ref_n ┆ max_ref_n │
│ ---          ┆ ---    ┆ ---       ┆ ---       │
│ str          ┆ u64    ┆ u64       ┆ u64       │
╞══════════════╪════════╪═══════════╪═══════════╡
│ rare         ┆ 3407   ┆ 1         ┆ 5         │
│ mid          ┆ 3224   ┆ 6         ┆ 40        │
│ common       ┆ 3275   ┆ 41        ┆ 9764      │
└──────────────┴────────┴───────────┴───────────┘


## 15. Canonical assignment table, evaluation summary, and manifest

One row per protein called an AVG by either module. `raw_L2` and `p_L2` are not included, because many specific names have more than one L2 parent (section 2). The function of a protein is `coalesce(annotate_function, final_label)`, read together with `final_level`.

Attach DefenseFinder consensus and assemble the contract schema.

In [70]:
dfjoin = df_rec.select(["Protein", "df_recovery"])
asg = (applied.join(dfjoin, on="Protein", how="left")
          .with_columns(
              pl.when(pl.col("annotate_avg") & pl.col("denovo_avg")).then(pl.lit("both"))
                .when(pl.col("annotate_avg")).then(pl.lit("annotate"))
                .otherwise(pl.lit("de_novo")).alias("module"),
              pl.lit(None, dtype=pl.Boolean).alias("consensus_structure")))
asg = asg.with_columns(
    pl.when(pl.col("df_recovery").is_null()).then(None)
      .when(pl.col("df_recovery") == "no_hit_P_novel").then(False)
      .otherwise(True).alias("consensus_defensefinder"))
asg = asg.with_columns(
    pl.when(pl.col("consensus_structure").is_not_null() & pl.col("consensus_defensefinder"))
      .then(pl.lit("corroborated"))
      .when(pl.col("consensus_defensefinder") | pl.col("consensus_structure"))
      .then(pl.lit("single_evidence"))
      .otherwise(pl.lit("none")).alias("consensus_tier"))
COLS = ["protein_id", "dataset", "module", "homology_invisible", "nn_ref_id", "d1", "d2", "margin",
        "ratio", "vote_purity_k5", "vote_purity_k20", "vote_purity_k50", "density_norm_d1",
        "hub_count", "raw_category", "raw_L1", "raw_specific", "p_category", "p_L1", "p_specific",
        "final_label", "final_level", "label_source", "precision_target",
        "tier1_label", "tier1_level", "tier2_label", "tier2_level", "cluster_reachable"]
COLS += [f"{k}_{int(round(t*100))}" for t in TARGETS for k in ("label", "level", "source")]
COLS += ["consensus_structure", "consensus_defensefinder", "consensus_tier",
         "annotate_function", "max_ident_to_train"]
assignments = asg.rename({"Protein": "protein_id"}).select(COLS)
print(f"assignments: {assignments.height:,} x {assignments.width}")
print(assignments.head(3))

assignments: 541,199 x 43
shape: (3, 43)
┌────────────┬─────────┬──────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ protein_id ┆ dataset ┆ module   ┆ homology_i ┆ … ┆ consensus ┆ consensus ┆ annotate_ ┆ max_ident │
│ ---        ┆ ---     ┆ ---      ┆ nvisible   ┆   ┆ _defensef ┆ _tier     ┆ function  ┆ _to_train │
│ str        ┆ str     ┆ str      ┆ ---        ┆   ┆ inder     ┆ ---       ┆ ---       ┆ ---       │
│            ┆         ┆          ┆ bool       ┆   ┆ ---       ┆ str       ┆ str       ┆ f32       │
│            ┆         ┆          ┆            ┆   ┆ bool      ┆           ┆           ┆           │
╞════════════╪═════════╪══════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ IMGVR_UViG ┆ soil    ┆ annotate ┆ false      ┆ … ┆ null      ┆ none      ┆ transcrip ┆ null      │
│ _208809001 ┆         ┆          ┆            ┆   ┆           ┆           ┆ tional    ┆           │
│ 4_000009|2 ┆         ┆          ┆            ┆  

Verify the contract before writing: every protein in the union is present exactly once, and no assignment column is null.

In [71]:
assert assignments.height == applied.height == full.height, "assignment table lost or gained rows"
assert assignments["protein_id"].n_unique() == assignments.height, "duplicate protein_id"
for c in ["final_level", "label_source", "tier1_level", "tier2_level", "cluster_reachable"]:
    n = assignments[c].null_count()
    assert n == 0, f"{c} has {n:,} nulls"
assert set(assignments["label_source"].unique().to_list()) <= {"cluster", "embedding", "none"}, \
    "label_source outside the closed vocabulary"
assert assignments.filter((pl.col("final_level") == "unassigned")
                          & (pl.col("label_source") != "none")).height == 0, \
    "unassigned rows carry a tier"
assert assignments.filter((pl.col("final_level") != "unassigned")
                          & pl.col("final_label").is_null()).height == 0, \
    "assigned rows with a null label"
print(f"contract checks passed on {assignments.height:,} rows, {assignments.width} columns")

contract checks passed on 541,199 rows, 43 columns


Write the assignment table.

In [72]:
ASG_PATH = OUT / "protein_assignments.parquet"
assignments.rename({"homology_invisible": "sequence_similarity_invisible"}).write_parquet(ASG_PATH)
print(f"wrote {ASG_PATH.name} ({assignments.height:,} rows, {ASG_PATH.stat().st_size/1e6:.1f} MB)")
print(assignments.group_by("final_level").agg(pl.len().alias("n")).sort("n", descending=True))

wrote protein_assignments.parquet (541,199 rows, 39.6 MB)
shape: (4, 2)
┌─────────────┬────────┐
│ final_level ┆ n      │
│ ---         ┆ ---    │
│ str         ┆ u64    │
╞═════════════╪════════╡
│ specific    ┆ 374304 │
│ L1          ┆ 80216  │
│ unassigned  ┆ 77301  │
│ category    ┆ 9378   │
└─────────────┴────────┘


Collect every metric computed above into the evaluation summary.

In [73]:
evaluation = {
    "schema_version": SCHEMA_VERSION,
    "split_integrity": {
        "criterion": "fewer than 5% of held-out proteins above 50% identity to any training protein",
        "passed": SPLIT_PASS,
        "frac_above_50pct": float((max_identity > 0.5).mean()),
        "frac_above_90pct": float((max_identity > 0.9).mean()),
        "median_max_identity": float(np.median(max_identity)),
        "applied_median_max_identity": float(np.median(applied_max_identity)),
        "note": "split is genome-cluster-atomic, not family-disjoint; all precision reported stratified by identity"},
    "ladder": {r["level"]: {"n": r["n"], "micro": r["micro"], "macro": r["macro"], "n_labels": r["n_labels"]}
               for r in ladder.iter_rows(named=True)},
    "majority_baseline": float(majority_baseline),
    "ladder_by_frequency": terciles.to_dicts(),
    "ladder_by_identity": strat.to_dicts(),
    "score_auroc": auc.to_dicts(),
    "score_gate_passed": SCORE_GATE,
    "calibration_ece": {r["level"]: r["ece"] for r in ece.iter_rows(named=True)},
    "ece_gate_passed": ECE_GATE,
    "risk_coverage": risk_coverage_summary.to_dicts(),
    "assignment_depth": {r["final_level"]: r["n"] for r in
                         assignments.group_by("final_level").agg(pl.len().alias("n")).iter_rows(named=True)},
    "information_bits": {"adaptive": mean_ad, "category_only": mean_ct, "gain": mean_ad - mean_ct},
    "frequency_recovery": {"spearman": sp_ok, "min_overlap_at_12": ov_min,
                           "slopes": slopes.to_dicts(), "gate_passed": FREQ_GATE},
    "defensefinder": {"positive_proteins": int(defensefinder_positive.height),
                      "category_breakdown": cat_break.to_dicts(),
                      "regulatory_pred_over_true_heldout": reg_ratio},
    "sequence_clustering": {
        "mmseqs_version": MMSEQS_VERSION,
        "command": MMSEQS_CMD,
        "min_seq_ids": [p / 100 for p in PCTS],
        "cluster_stats": cstat.to_dicts(),
        "tier1_min_seq_id_chosen": {lvl: (PSTAR[PRIMARY_TARGET][lvl] / 100
                                          if PSTAR[PRIMARY_TARGET][lvl] else None) for lvl in LEVELS},
        "tier2_threshold_refit": THR[PRIMARY_TARGET],
        "tier2_threshold_unmasked": THR_FULL[PRIMARY_TARGET],
        "ablation_primary_target": ablation.filter(pl.col("target") == PRIMARY_TARGET).to_dicts(),
        "tier_overlap": ovl.to_dicts(),
        "recall_at_matched_precision": rec.to_dicts(),
        "tiered_reach_by_identity": tstrat.to_dicts(),
        "tier1_reach_by_identity": reach_id.to_dicts(),
        "applied_reach": areach.to_dicts(),
        "applied_unclustered": int(n_unclustered),
        "applied_label_source": {r["label_source"]: r["n"] for r in
                                 applied.group_by("label_source").agg(pl.len().alias("n"))
                                    .iter_rows(named=True)}},
    "controls": {"real": real, "permuted_mean": float(perms.mean()), "permuted_sd": float(perms.std()),
                 "majority": basec, "cross_domain": cross_domain, "specificity": spec,
                 "esm2_baseline": {k: float(v) for k, v in esm.items()} if esm else None},
}
(OUT / "evaluation_summary.json").write_text(json.dumps(evaluation, indent=2, default=str))
print("wrote evaluation_summary.json")

wrote evaluation_summary.json


Write the manifest that downstream notebooks assert against.

In [74]:
INPUTS = [TRAIN_INDEX, NOVEL / "train_ref_index_order.parquet", NOVEL / "test_avglike_meta.parquet",
          FILES / "AMGs_categorized.tsv", FILES / "APGs_categorized.tsv", FILES / "AReGs_categorized.tsv",
          FILES / "pst_thresholds.json"] + [agg(ds) for ds in DATASETS] + \
         [den(ds) / "predictions.tsv" for ds in DATASETS]
try:
    commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
except Exception:
    commit = "unknown"
manifest = {
    "schema_version": SCHEMA_VERSION,
    "git_commit": commit,
    "generated": datetime.datetime.now().isoformat(timespec="seconds"),
    "db_stamp": DB_STAMP,
    "n_proteins": int(assignments.height),
    "levels": LEVELS,
    "l2_dropped": True,
    "l2_reason": f"{HIER['multi_L2_function_pct']:.1f}% of training function names map to more than one L2 parent",
    "l1_canonical_parent_rule": "L1 assigned by the most HMM ids for that name, ties broken alphabetically",
    "hierarchy_stats": HIER,
    "precision_targets": TARGETS,
    "target_semantics": "realized precision on held-out data, not the calibrated probability cutoff",
    "probability_thresholds": {str(t): THR[t] for t in TARGETS},
    "primary_target": PRIMARY_TARGET,
    "sequence_clustering": {
        "mmseqs_version": MMSEQS_VERSION,
        "command": MMSEQS_CMD,
        "min_seq_ids": [p / 100 for p in PCTS],
        "shared_universe_sequences": UNIVERSE_N,
        "training_sequences": int(reference.height),
        "applied_sequences_total": int(applied.height),
        "applied_sequences_clustered": int(applied.height) - int(n_unclustered),
        "applied_sequences_unclustered": int(n_unclustered),
        "unclustered_reason": "absent from the biome-filtered protein FASTAs the clustering input "
                              "was drawn from; cause is upstream of this notebook and not diagnosed here",
        "id_collisions_train_vs_applied": int(n_collide),
        "tier1_min_seq_id": {lvl: (PSTAR[PRIMARY_TARGET][lvl] / 100
                                   if PSTAR[PRIMARY_TARGET][lvl] else None) for lvl in LEVELS},
        "tier2_threshold_refit_on_tier1_unreachable": THR[PRIMARY_TARGET],
        "tier2_threshold_unmasked": THR_FULL[PRIMARY_TARGET],
        "threshold_fit_order": "tier 1 identity chosen first on the calibration slice, then the "
                               "tier 2 probability inverted on the calibration proteins tier 1 "
                               "does not reach at that granularity"},
    "label_source_vocabulary": ["cluster", "embedding", "none"],
    "tier_definitions": {"cluster": "MMseqs2 cluster majority over labeled training members, ties abstain",
                         "embedding": "CheckAMG-PST 1-NN label gated on the calibrated correctness probability",
                         "none": "no tier reached the protein at any granularity"},
    "consensus_structure_populated": False,
    "consensus_structure_reason": "structural search depends on novel_avgs clustering and stays in stage 3",
    "inputs_sha256": {str(p): (sha256(p) if Path(p).exists() else None) for p in INPUTS},
    "headline": {
        "split_gate_passed": SPLIT_PASS,
        "category_micro": evaluation["ladder"]["category"]["micro"],
        "specific_micro": evaluation["ladder"]["specific"]["micro"],
        "specific_macro": evaluation["ladder"]["specific"]["macro"],
        "majority_baseline": float(majority_baseline),
        "frequency_gate_passed": FREQ_GATE,
        "ece_gate_passed": ECE_GATE},
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest["headline"], indent=2))

{
  "split_gate_passed": false,
  "category_micro": 0.8952,
  "specific_micro": 0.3813,
  "specific_macro": 0.2255,
  "majority_baseline": 0.6688300035889461,
  "frequency_gate_passed": false,
  "ece_gate_passed": true
}


## 16. Sequence clustering compared with embedding transfer

The reporting-slice proteins, the thresholds, and the tier 1 identities are read back from the tables written above. Each transfer rule is scored by identity stratum and label depth: embedding 1-NN, the cluster majority at each identity threshold, the calibrated embedding rule, and the tiered rule. Micro precision carries a Wilson 95% interval, and macro precision is the unweighted mean over the labels a rule assigns.

In [ ]:
CLUSTER_STRATA = [("<30%", 0.0, 0.3), ("30-50%", 0.3, 0.5), ("<50%", 0.0, 0.5), ("50-90%", 0.5, 0.9), (">=90%", 0.9, 1.01),
                  ("all", 0.0, 1.01)]
TRANSFER_RULES = ["embedding_1nn", "cluster_id30", "cluster_id50", "cluster_id70", "cluster_id90", "embedding_calibrated", "tiered"]

def macro_precision(predicted, correct):
    _, codes = np.unique(predicted.astype(str), return_inverse=True)
    n_labels = int(codes.max()) + 1
    per_label = np.bincount(codes, weights=correct, minlength=n_labels) / np.bincount(codes, minlength=n_labels)
    return float(per_label.mean()), n_labels

saved_manifest = json.loads((OUT / "manifest.json").read_text())
primary_thresholds = saved_manifest["probability_thresholds"][str(saved_manifest["primary_target"])]
tier1_identity = saved_manifest["sequence_clustering"]["tier1_min_seq_id"]
saved_report = pl.read_parquet(OUT / "evaluation_per_protein.parquet").filter(pl.col("split") == "report")
saved_report = saved_report.with_columns(pl.Series("raw_specific_n", normfn(saved_report["raw_specific"].fill_null("").to_numpy().astype(str))))
assert ((saved_report["raw_specific_n"] == saved_report["true_specific_n"]) == saved_report["y_specific"]).all()
comparison_rows = []
for stratum, low, high in CLUSTER_STRATA:
    stratum_rows = saved_report.filter((pl.col("max_ident_to_train") >= low) & (pl.col("max_ident_to_train") < high))
    for level in LEVELS:
        truth = np.asarray(stratum_rows[TRUE[level]].fill_null("__missing_truth__").to_numpy(), dtype=object)
        embedding = np.asarray(stratum_rows[EMB[level]].to_numpy(), dtype=object)
        probability = stratum_rows["p_" + level].to_numpy().astype(float)
        embedding_assigned = np.array([e is not None and e != "" for e in embedding])
        confident = embedding_assigned & ~np.isnan(probability) & (probability >= primary_thresholds[level])
        for rule in TRANSFER_RULES:
            if rule == "embedding_1nn":
                predicted, assigned = embedding, embedding_assigned
            elif rule == "embedding_calibrated":
                predicted, assigned = embedding, confident
            elif rule == "tiered":
                cluster = np.asarray(stratum_rows[f"c{round(tier1_identity[level] * 100)}_{level}"].to_numpy(), dtype=object)
                has_cluster = np.array([c is not None for c in cluster])
                predicted = np.where(has_cluster, cluster, np.where(confident, embedding, None))
                assigned = has_cluster | confident
            else:
                cluster = np.asarray(stratum_rows[f"c{rule.replace('cluster_id', '')}_{level}"].to_numpy(), dtype=object)
                predicted, assigned = cluster, np.array([c is not None for c in cluster])
            n_assigned = int(assigned.sum())
            row = {"method": rule, "level": level, "identity_stratum": stratum, "n_report": stratum_rows.height,
                   "n_assigned": n_assigned, "coverage": n_assigned / stratum_rows.height,
                   "micro_precision": None, "micro_ci95_lo": None, "micro_ci95_hi": None, "macro_precision": None,
                   "n_distinct_labels": 0}
            if n_assigned:
                correct = (predicted[assigned] == truth[assigned]).astype(float)
                low_ci, high_ci = wilson(correct.sum(), n_assigned)
                macro, n_labels = macro_precision(predicted[assigned], correct)
                row.update({"micro_precision": float(correct.mean()), "micro_ci95_lo": low_ci, "micro_ci95_hi": high_ci,
                            "macro_precision": macro, "n_distinct_labels": n_labels})
            comparison_rows.append(row)
clustering_comparison = pl.DataFrame(comparison_rows, infer_schema_length=None)
clustering_comparison.write_csv(OUT / "clustering_vs_embedding_comparison.csv")
print(f"clustering_vs_embedding_comparison.csv: {clustering_comparison.height} rows")

clustering_vs_embedding_comparison.csv: 126 rows


The embedding ranked by calibrated probability and cut at the coverage each clustering reaches, so the two are compared at matched coverage.

In [ ]:
matched_rows = []
for stratum, low, high in CLUSTER_STRATA:
    stratum_rows = saved_report.filter((pl.col("max_ident_to_train") >= low) & (pl.col("max_ident_to_train") < high))
    for level in LEVELS:
        truth = np.asarray(stratum_rows[TRUE[level]].fill_null("\x00null\x00").to_numpy(), dtype=object)
        embedding = np.asarray(stratum_rows[EMB[level]].to_numpy(), dtype=object)
        probability = stratum_rows["p_" + level].fill_null(-1.0).to_numpy()
        embedding_correct = (embedding == truth).astype(float)
        order = np.argsort(-probability, kind="stable")
        for p in PCTS:
            cluster = np.asarray(stratum_rows[f"c{p}_{level}"].to_numpy(), dtype=object)
            voted = np.array([c is not None for c in cluster])
            k = int(voted.sum())
            if k == 0:
                continue
            cluster_correct = (cluster[voted] == truth[voted]).astype(float)
            cluster_lo, cluster_hi = wilson(float(cluster_correct.sum()), k)
            cluster_macro, cluster_labels = macro_precision(cluster[voted], cluster_correct)
            top = order[:k]
            top_correct = embedding_correct[top]
            embedding_lo, embedding_hi = wilson(float(top_correct.sum()), k)
            embedding_macro, embedding_labels = macro_precision(embedding[top], top_correct)
            matched_rows.append({
                "identity_stratum": stratum, "level": level, "min_seq_id": p / 100, "n_report": stratum_rows.height,
                "n_matched_coverage": k, "coverage": k / stratum_rows.height,
                "cluster_micro": float(cluster_correct.mean()), "cluster_micro_lo": cluster_lo, "cluster_micro_hi": cluster_hi,
                "cluster_macro": cluster_macro, "cluster_n_labels": cluster_labels,
                "embedding_micro": float(top_correct.mean()), "embedding_micro_lo": embedding_lo,
                "embedding_micro_hi": embedding_hi, "embedding_macro": embedding_macro, "embedding_n_labels": embedding_labels,
                "embedding_p_at_cut": float(probability[top][-1]),
                "macro_delta_cluster_minus_embedding": cluster_macro - embedding_macro,
                "micro_delta_cluster_minus_embedding": float(cluster_correct.mean()) - float(top_correct.mean())})
coverage_matched = pl.DataFrame(matched_rows)
coverage_matched.write_csv(OUT / "clustering_vs_embedding_coverage_matched.csv")
print(f"clustering_vs_embedding_coverage_matched.csv: {coverage_matched.height} rows")

clustering_vs_embedding_coverage_matched.csv: 51 rows


Embedding 1-NN with every neighbor in the query's 30 percent identity cluster masked, so no label comes from a close sequence relative. The first unmasked neighbor among the 50 cached ones donates the label.

In [ ]:
REFERENCE_LABEL = {"specific": "ref_specific", "L1": "ref_L1", "category": "ref_category"}
query_ids = pl.read_parquet(EVAL_KEYS)["Protein"].to_list()
labeled = pl.read_parquet(LABELED_REFERENCE)
eval_neighbors = np.load(CACHE / "eval_I.npy")
assert eval_neighbors.shape[0] == len(query_ids) and int(eval_neighbors.max()) < labeled.height
clusters30 = load_clu(SEQCLUST / CLU_EVAL.format(p=30))
member_to_cluster = dict(zip(clusters30["member"].to_list(), clusters30["cluster"].to_list()))
query_cluster = np.array([member_to_cluster.get(q, "\x00none\x00") for q in query_ids], dtype=object)
reference_cluster = np.array([member_to_cluster.get(r, "\x00none\x00") for r in labeled["nn_ref_id"].to_list()], dtype=object)
same_cluster = (reference_cluster[eval_neighbors] == query_cluster[:, None]) & (query_cluster[:, None] != "\x00none\x00")
n_surviving = (~same_cluster).sum(axis=1)
first_surviving = np.where(n_surviving > 0, np.argmax(~same_cluster, axis=1), -1)
position = {q: i for i, q in enumerate(query_ids)}
report_index = np.array([position[p] for p in saved_report["Protein"].to_list()])
stress_rows = []
for level in LEVELS:
    truth = np.asarray(saved_report[TRUE[level]].fill_null("\x00null\x00").to_numpy(), dtype=object)
    labels = np.asarray(labeled[REFERENCE_LABEL[level]].to_numpy(), dtype=object)
    first = first_surviving[report_index]
    kept = first >= 0
    masked = np.full(len(report_index), None, dtype=object)
    donated = labels[eval_neighbors[report_index[kept], first[kept]]]
    masked[kept] = normfn(donated) if level == "specific" else donated
    standard = labels[eval_neighbors[report_index, 0]]
    standard = normfn(standard) if level == "specific" else standard
    for rule, predicted, scored in [("embedding_leave_cluster_out", masked, kept),
                                    ("embedding_standard_1nn", standard, np.ones(len(report_index), bool))]:
        correct = (predicted[scored] == truth[scored]).astype(float)
        n = int(scored.sum())
        low_ci, high_ci = wilson(float(correct.sum()), n)
        macro, n_labels = macro_precision(predicted[scored], correct)
        stress_rows.append({"method": rule, "level": level, "n_scored": n, "coverage_of_report": n / len(report_index),
                            "micro_precision": float(correct.mean()), "micro_ci95_lo": low_ci, "micro_ci95_hi": high_ci,
                            "macro_precision": macro, "n_distinct_labels": n_labels})
stress = pl.DataFrame(stress_rows)
stress.write_csv(OUT / "leave_cluster_out_stress_test.csv")
(OUT / "leave_cluster_out_provenance.json").write_text(json.dumps(
    {"n_report": int(len(report_index)), "n_all_neighbors_masked": int((n_surviving[report_index] == 0).sum()),
     "clustering_used": "clu_id30.tsv", "cached_neighbors_per_query": int(eval_neighbors.shape[1])}, indent=1))
print(f"leave_cluster_out_stress_test.csv: {stress.height} rows")

leave_cluster_out_stress_test.csv: 6 rows
